In [137]:
import pandas as pd
from pathlib import Path
import numpy as np
import json
from sklearn.preprocessing import OneHotEncoder,PowerTransformer
ROOT = Path.cwd().parent
motor='DATA_MODEL_Dev/Motor_data_models.csv'
data_split='DATA_RAW_MODELS/Data_split.json'

# DATA SPLIT

In [138]:
with open(ROOT/data_split, "r", encoding="utf-8") as archivo:
    data_subjects = json.load(archivo)

subjects_train=data_subjects['Train_80']
subjects_test=data_subjects['Test_20']

## Normal Data

In [139]:
X_train=pd.read_csv(ROOT/motor)
X_train=X_train[X_train['subject_visit'].isin(subjects_train)]
X_train.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

X_test=pd.read_csv(ROOT/motor)
X_test=X_test[X_test['subject_visit'].isin(subjects_test)]
X_test.drop(columns=['subject_visit','UPDRS_III_ProgressionType'], inplace=True)

y_train=pd.read_csv(ROOT/motor)
y_train=y_train[y_train['subject_visit'].isin(subjects_train)]
y_train=y_train['UPDRS_III_ProgressionType']
y_test=pd.read_csv(ROOT/motor)
y_test=y_test[y_test['subject_visit'].isin(subjects_test)]
y_test=y_test['UPDRS_III_ProgressionType']

## Data Agrupations Stability-Improvement VS Worsening

In [140]:
y_train_2_IS_W= y_train.copy()
y_train_2_IS_W=y_train_2_IS_W.replace({0:0,-1:1,1:1})

## Data Agrupations Stability VS Improvement-Worsening

In [141]:
y_train_2_S_IW= y_train.copy()
y_train_2_S_IW=y_train_2_S_IW.replace({0:0,-1:0,1:1})

## Feature Engineering

In [142]:
X_train_fe=X_train.copy()
y_train_fe=X_test.copy()


X_train_fe['motor_burden']=X_train_fe['MDS-UPDRS Part III Total Score']+X_train_fe['MDS-UPDRS Part IV Total Score']
X_train_fe['QoL_burden']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']+X_train_fe['MDS-UPDRS Part II Total Score']
X_train_fe['motor_ratio']=X_train_fe['motor_burden']/(X_train_fe['QoL_burden']+1)


X_train_fe['motor_cog_ratio_qol']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Part II Total Score']+1)

X_train_fe['UPDRS1_weight']=X_train_fe['MDS-UPDRS Part I (Patient Questionnaire) Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS2_weight']=X_train_fe['MDS-UPDRS Part II Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)
X_train_fe['UPDRS3_weight']=X_train_fe['MDS-UPDRS Part III Total Score']/(X_train_fe['MDS-UPDRS Total Score']+1)

X_train_fe['Class3_H&Y']=X_train_fe['UPDRS_III_Class']/(X_train_fe['3.21 HOEHN AND YAHR STAGE']+0.01)
X_train_fe['Functional_impairment']= X_train_fe['SCHWAB & ENGLAND ADL'].apply(lambda x: 100 - x)


encoder = OneHotEncoder(sparse_output=False)

resultados = encoder.fit_transform(X_train_fe[['3.21 HOEHN AND YAHR STAGE']])
columnas_encoder = encoder.get_feature_names_out(['3.21 HOEHN AND YAHR STAGE'])

df_resultados = pd.DataFrame(
    resultados,
    columns=columnas_encoder,
    index=X_train_fe.index
)

X_train_fe = pd.concat([X_train_fe, df_resultados], axis=1)
X_train_fe.drop(columns=['3.21 HOEHN AND YAHR STAGE'], inplace=True)

X_train_fe.head()


,SCHWAB & ENGLAND ADL,MDS-UPDRS Part I (Patient Questionnaire) Total Score,MDS-UPDRS Part II Total Score,Does participant have DBS,MDS-UPDRS Part III Total Score,MDS-UPDRS Part IV Total Score,DBS_Transition_Visit,DBS_Post_Transition,MDS-UPDRS Total Score,UPDRS_I_Class,...,UPDRS1_weight,UPDRS2_weight,UPDRS3_weight,Class3_H&Y,Functional_impairment,3.21 HOEHN AND YAHR STAGE_0,3.21 HOEHN AND YAHR STAGE_1,3.21 HOEHN AND YAHR STAGE_2,3.21 HOEHN AND YAHR STAGE_3,3.21 HOEHN AND YAHR STAGE_4
0,90,6,5.0,0,35,0.0,0,0.0,46.0,0,...,0.127660,0.106383,0.744681,0.497512,10,0.0,0.0,1.0,0.0,0.0
1,85,9,10.0,0,53,0.0,0,0.0,72.0,0,...,0.123288,0.136986,0.726027,0.332226,15,0.0,0.0,0.0,1.0,0.0
2,80,11,16.0,0,18,7.0,0,0.0,52.0,1,...,0.207547,0.301887,0.339623,0.000000,20,0.0,0.0,1.0,0.0,0.0
3,90,8,8.0,0,34,0.0,0,0.0,50.0,0,...,0.156863,0.156863,0.666667,0.497512,10,0.0,0.0,1.0,0.0,0.0
4,85,4,9.0,0,16,0.0,0,0.0,29.0,0,...,0.133333,0.300000,0.533333,0.000000,15,0.0,0.0,1.0,0.0,0.0


# Feature Selection No Model Univariate

In [143]:
from sklearn.feature_selection import GenericUnivariateSelect
from sklearn.feature_selection import chi2, f_classif, mutual_info_classif
from sklearn.feature_selection import VarianceThreshold

In order to perform univariate feature selection we must summarize the methodology and all the methods that we will study:
1. Variance treshold: Fast/Easy it may not capture the all picture. 
2. Using the GenericUnivarianceSelect method we can apply SelectKBest/SelectPercentile and the statistical function associated.

## VarianceThreshold

In [144]:
threshold_range = np.arange(0, 2, 0.1).tolist()

for tresh in threshold_range:
    selector = VarianceThreshold(threshold=tresh)
    X_train_reduced = selector.fit_transform(X_train)
    n_features = X_train_reduced.shape[1]
    print(f'Threshold: {tresh:.2}, Number of features retained: {n_features}')
    # Columnas eliminadas
    eliminadas = X_train.columns[~selector.get_support()]
    print("Deleted:", eliminadas.tolist())

    # Columnas conservadas
    conservadas = X_train.columns[selector.get_support()]
    print("Conserved:", conservadas.tolist())
    print("---------------------------------------------------")



Threshold: 0.0, Number of features retained: 14
Deleted: []
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'Does participant have DBS', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Transition_Visit', 'DBS_Post_Transition', 'MDS-UPDRS Total Score', 'UPDRS_I_Class', 'UPDRS_II_Class', 'UPDRS_III_Class', 'UPDRS_IV_Class']
---------------------------------------------------
Threshold: 0.1, Number of features retained: 10
Deleted: ['Does participant have DBS', 'DBS_Transition_Visit', 'UPDRS_I_Class', 'UPDRS_III_Class']
Conserved: ['SCHWAB & ENGLAND ADL', 'MDS-UPDRS Part I (Patient Questionnaire) Total Score', 'MDS-UPDRS Part II Total Score', 'MDS-UPDRS Part III Total Score', '3.21 HOEHN AND YAHR STAGE', 'MDS-UPDRS Part IV Total Score', 'DBS_Post_Transition', 'MDS-UPDRS Total Score', 'UPDRS_II_Class', 'UPDRS_IV_Class']
------------------------------------------

This method is not usefull due to some features that have low variance are related to not frequent events

## Generic Univariate Select

In [145]:
mode_list = ['percentile', 'k_best']
score_funcs = [f_classif, mutual_info_classif, chi2]
k_list = range(5, 10)
percentile_list = range(10, 100, 10)

results = []

for mode in mode_list:
    for score in score_funcs:

        if mode == 'k_best':
            for k in k_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=k
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": k,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

        elif mode == 'percentile':
            for p in percentile_list:
                
                transformer = GenericUnivariateSelect(
                    score_func=score,
                    mode=mode,
                    param=p
                )

                X_new = transformer.fit_transform(X_train, y_train)

                selected_features = X_train.columns[transformer.get_support()]
                delete_features = X_train.columns[~transformer.get_support()]

                results.append({
                    "mode": mode,
                    "score_func": score.__name__,
                    "param": p,
                    "n_features": X_new.shape[1],
                    "features Added": list(selected_features),
                    "features Deleted": list(delete_features)
                })

df_results = pd.DataFrame(results)
df_results.to_csv(ROOT/'DATA_MODEL_Dev/Motor_feature_selection_results.csv', index=False)
df_results.head()


,mode,score_func,param,n_features,features Added,features Deleted
0,percentile,f_classif,10,2,"[MDS-UPDRS Part III Total Score, MDS-UPDRS Tot...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
1,percentile,f_classif,20,3,"[MDS-UPDRS Part III Total Score, MDS-UPDRS Tot...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
2,percentile,f_classif,30,4,"[MDS-UPDRS Part III Total Score, 3.21 HOEHN AN...","[SCHWAB & ENGLAND ADL, MDS-UPDRS Part I (Patie..."
3,percentile,f_classif,40,6,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...
4,percentile,f_classif,50,7,"[SCHWAB & ENGLAND ADL, MDS-UPDRS Part II Total...",[MDS-UPDRS Part I (Patient Questionnaire) Tota...


Given that the motor data only present 9 features all will be selected

# Motor Model Development
<p align="center">
    <img src="../../../figures/model_process.png" alt="Model process" width="600">
</p>





---

Linear Models  
(assume a linear decision boundary)

- Logistic Regression  
- Linear Support Vector Machine (Linear SVM)  
- SGD Classifier (logistic loss / hinge loss)

---

Non-Linear Models  
(can model complex decision boundaries)

Tree-Based Models  
- Decision Trees  
- Random Forest  
- Extra Trees (Extremely Randomized Trees)  
- Gradient Boosting  
- XGBoost  
- LightGBM  
- CatBoost  

Distance-Based Models  
- K-Nearest Neighbors (KNN)

Margin-Based Models  
- Support Vector Machines (SVM) with kernels:  
  - RBF kernel  
  - Polynomial kernel  
  - Sigmoid kernel  

Probabilistic Models  
- Multinomial Naive Bayes  
- Bernoulli Naive Bayes  
- Bayesian Classifiers  

Neural Network Models  
- Multilayer Perceptron (MLP)  
---



In [146]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn import svm
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB, BernoulliNB, GaussianNB
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.feature_selection import SequentialFeatureSelector
from imblearn.pipeline import Pipeline 
from imblearn.over_sampling import RandomOverSampler, SMOTE
from sklearn.base import clone

from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import make_scorer, recall_score, confusion_matrix,f1_score,precision_score,balanced_accuracy_score


def specificity_weighted(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    n_classes = cm.shape[0]

    total_samples = np.sum(cm)
    specificities = []
    supports = []

    for k in range(n_classes):
        TP = cm[k, k]
        FN = np.sum(cm[k, :]) - TP
        FP = np.sum(cm[:, k]) - TP
        TN = total_samples - (TP + FN + FP)

        spec_k = TN / (TN + FP) if (TN + FP) > 0 else 0
        specificities.append(spec_k)
        supports.append(np.sum(cm[k, :]))

    return np.average(specificities, weights=supports)


def g_mean(y_true, y_pred):
    sens = recall_score(y_true, y_pred, average="macro")
    spec = specificity_weighted(y_true, y_pred)
    return np.sqrt(sens * spec)




def cv_results_to_row(results, modelo_name, parameters, sep=" ± "):

    row = {
        "Model": modelo_name,
        "Parameters": parameters
    }

    for k in results:
        if k.startswith("train_") or k.startswith("test_"):
            prefix, metric = k.split("_", 1)

            mean = results[k].mean()
            

            col_name = f"{prefix}_{metric}"
            row[col_name] = f"{mean:.4f}"

    return pd.DataFrame([row])


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "acc": "accuracy",
    "bal_acc": make_scorer(balanced_accuracy_score),
    "f1_macro": make_scorer(f1_score, average="macro"),
    "f1_weighted": make_scorer(f1_score, average="weighted"),
    "prec_macro": make_scorer(precision_score, average="macro", zero_division=0),
    "rec_macro": make_scorer(recall_score, average="macro", zero_division=0),
    "specificity_weighted": make_scorer(specificity_weighted),
    "g_mean": make_scorer(g_mean),
    
}


## Dummy Classifier

In [147]:
# 1) Dummy model (baseline)
Dummy_model = DummyClassifier(
    strategy="most_frequent",
    random_state=42
)
# 2) Pipeline
pipe_dummy = Pipeline([
    ("clf", Dummy_model)
])

# 3) Cross-validation
results_dummy = cross_validate(
    pipe_dummy,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

# 4) Convertir resultados a fila
df_dummy = cv_results_to_row(
    results_dummy,
    modelo_name="DummyClassifier",
    parameters="Most frequent (Baseline), No Scaler (Pipeline)",
    sep=" ± "
)

# 5) Mostrar resultados
df_dummy


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,DummyClassifier,"Most frequent (Baseline), No Scaler (Pipeline)",0.3956,0.3956,0.3333,0.3333,0.1890,0.1890,0.2243,0.2243,0.1319,0.1319,0.3333,0.3333,0.6044,0.6044,0.4489,0.4488


---

## Linear Models

### Logistic Regression

In [148]:
LogReg_model = LogisticRegression(class_weight="balanced", max_iter=1000, penalty="l2", solver="lbfgs")

#### Normal Data 

##### No Scaler

In [149]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_logreg_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_results


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-le

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,Normal / No Scaler / 3 Class,0.4341,0.4821,0.4246,0.4782,0.4226,0.4748,0.4351,0.4835,0.4276,0.4751,0.4246,0.4782,0.7125,0.7376,0.5494,0.5939
1,LogisticRegression,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5647,0.5962,0.5698,0.6049,0.5600,0.5932,0.5687,0.6004,0.5671,0.6005,0.5698,0.6049,0.5749,0.6136,0.5723,0.6092
2,LogisticRegression,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.5838,0.6119,0.5650,0.6093,0.5357,0.5698,0.6110,0.6378,0.5498,0.5826,0.5650,0.6093,0.5462,0.6067,0.5555,0.6080
3,LogisticRegression,Normal / No Scaler / 2 Class IS_W / SMOTE,0.6196,0.6360,0.5822,0.5992,0.5803,0.5995,0.6070,0.6247,0.5911,0.6121,0.5822,0.5992,0.5448,0.5623,0.5631,0.5805
4,LogisticRegression,Normal / No Scaler / 2 Class S_IW / SMOTE,0.7418,0.7503,0.5170,0.5329,0.4827,0.5091,0.6677,0.6830,0.5640,0.6165,0.5170,0.5329,0.2921,0.3155,0.3884,0.4098
5,LogisticRegression,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5837,0.6154,0.5668,0.6065,0.5363,0.5702,0.6110,0.6406,0.5509,0.5810,0.5668,0.6065,0.5499,0.5976,0.5582,0.6020
6,LogisticRegression,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5701,0.5938,0.5760,0.6033,0.5658,0.5911,0.5743,0.5979,0.5730,0.5991,0.5760,0.6033,0.5819,0.6129,0.5789,0.6081


##### StandardScaler

In [150]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_logreg_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,Normal / StandardScaler / 3 Class,0.4314,0.4780,0.4198,0.4742,0.4181,0.4707,0.4321,0.4793,0.4237,0.4710,0.4198,0.4742,0.7109,0.7354,0.5456,0.5905
1,LogisticRegression,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5660,0.5931,0.5715,0.6025,0.5614,0.5903,0.5700,0.5972,0.5688,0.5983,0.5715,0.6025,0.5769,0.6119,0.5742,0.6072
2,LogisticRegression,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.5783,0.6109,0.5632,0.6063,0.5323,0.5679,0.6063,0.6368,0.5481,0.5804,0.5632,0.6063,0.5481,0.6017,0.5556,0.6040
3,LogisticRegression,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5647,0.5941,0.5722,0.6026,0.5607,0.5910,0.5685,0.5983,0.5695,0.5983,0.5722,0.6026,0.5797,0.6111,0.5759,0.6068
4,LogisticRegression,Normal / StandardScaler / 2 Class S_IW,0.5892,0.6089,0.5686,0.6003,0.5400,0.5640,0.6158,0.6346,0.5529,0.5761,0.5686,0.6003,0.5480,0.5917,0.5582,0.5960
5,LogisticRegression,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5851,0.6113,0.5677,0.6000,0.5375,0.5652,0.6121,0.6367,0.5519,0.5761,0.5677,0.6000,0.5503,0.5888,0.5589,0.5944
6,LogisticRegression,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5564,0.5903,0.5634,0.6007,0.5523,0.5879,0.5603,0.5944,0.5611,0.5966,0.5634,0.6007,0.5705,0.6110,0.5669,0.6058


##### MinMax Scaler

In [151]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_logreg_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,Normal / MinMaxScaler / 3 Class,0.4533,0.4828,0.4369,0.4729,0.4355,0.4719,0.4520,0.4831,0.4393,0.4715,0.4369,0.4729,0.7163,0.7312,0.5588,0.5880
1,LogisticRegression,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5591,0.5900,0.5700,0.6022,0.5565,0.5880,0.5627,0.5938,0.5677,0.5984,0.5700,0.6022,0.5809,0.6144,0.5754,0.6082
2,LogisticRegression,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.5741,0.6181,0.5493,0.6083,0.5232,0.5724,0.6012,0.6431,0.5383,0.5825,0.5493,0.6083,0.5244,0.5985,0.5366,0.6034
3,LogisticRegression,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5605,0.5941,0.5705,0.6044,0.5578,0.5916,0.5644,0.5981,0.5680,0.6002,0.5705,0.6044,0.5805,0.6147,0.5755,0.6095
4,LogisticRegression,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5783,0.6085,0.5594,0.6005,0.5303,0.5640,0.6055,0.6344,0.5460,0.5762,0.5594,0.6005,0.5406,0.5925,0.5499,0.5965
5,LogisticRegression,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5810,0.6171,0.5631,0.6072,0.5334,0.5714,0.6082,0.6421,0.5485,0.5816,0.5631,0.6072,0.5453,0.5972,0.5541,0.6022
6,LogisticRegression,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5619,0.5903,0.5734,0.6035,0.5597,0.5886,0.5658,0.5940,0.5709,0.5998,0.5734,0.6035,0.5850,0.6167,0.5792,0.6101


#### FE_DATA

##### NO SCALER

In [152]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_logreg_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_results_fe


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-le

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,FE / No Scaler / 3 Class,0.4190,0.4894,0.4076,0.4857,0.4050,0.4817,0.4201,0.4907,0.4113,0.4828,0.4076,0.4857,0.7089,0.7428,0.5367,0.6006
1,LogisticRegression,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5715,0.6102,0.5730,0.6138,0.5650,0.6054,0.5754,0.6145,0.5699,0.6090,0.5730,0.6138,0.5746,0.6174,0.5738,0.6156
2,LogisticRegression,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.5810,0.6209,0.5613,0.6106,0.5321,0.5749,0.6080,0.6455,0.5471,0.5843,0.5613,0.6106,0.5416,0.6003,0.5513,0.6054
3,LogisticRegression,FE / No Scaler / 2 Class IS_W / SMOTE,0.6155,0.6374,0.5799,0.6027,0.5794,0.6032,0.6050,0.6275,0.5895,0.6141,0.5799,0.6027,0.5443,0.5681,0.5618,0.5851
4,LogisticRegression,FE / No Scaler / 2 Class S_IW / SMOTE,0.7404,0.7490,0.5347,0.5432,0.5194,0.5291,0.6840,0.6917,0.5951,0.6177,0.5347,0.5432,0.3290,0.3374,0.4193,0.4279
5,LogisticRegression,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5769,0.6222,0.5567,0.6106,0.5281,0.5755,0.6047,0.6467,0.5431,0.5845,0.5567,0.6106,0.5365,0.5989,0.5464,0.6047
6,LogisticRegression,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5811,0.6051,0.5851,0.6106,0.5754,0.6009,0.5848,0.6093,0.5818,0.6060,0.5851,0.6106,0.5892,0.6162,0.5871,0.6134


##### StandardScaler

In [153]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_logreg_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,FE / StandardScaler / 3 Class,0.4218,0.4955,0.4108,0.4913,0.4078,0.4871,0.4221,0.4964,0.4125,0.4889,0.4108,0.4913,0.7083,0.7461,0.5389,0.6055
1,LogisticRegression,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5660,0.6123,0.5673,0.6148,0.5594,0.6069,0.5699,0.6165,0.5647,0.6099,0.5673,0.6148,0.5685,0.6173,0.5678,0.6160
2,LogisticRegression,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.5797,0.6209,0.5511,0.6101,0.5256,0.5746,0.6060,0.6455,0.5391,0.5840,0.5511,0.6101,0.5225,0.5994,0.5364,0.6047
3,LogisticRegression,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5715,0.6137,0.5736,0.6162,0.5652,0.6083,0.5753,0.6178,0.5706,0.6114,0.5736,0.6162,0.5757,0.6188,0.5746,0.6175
4,LogisticRegression,FE / StandardScaler / 2 Class S_IW,0.5741,0.6099,0.5530,0.6033,0.5244,0.5660,0.6020,0.6357,0.5398,0.5784,0.5530,0.6033,0.5319,0.5967,0.5422,0.6000
5,LogisticRegression,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5811,0.6236,0.5613,0.6082,0.5318,0.5751,0.6078,0.6476,0.5473,0.5830,0.5613,0.6082,0.5416,0.5928,0.5512,0.6005
6,LogisticRegression,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5660,0.6099,0.5690,0.6146,0.5603,0.6054,0.5700,0.6141,0.5664,0.6097,0.5690,0.6146,0.5720,0.6193,0.5705,0.6170


##### MinMax

In [154]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(LogReg_model))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="LogisticRegression",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_logreg_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_logreg_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,FE / MinMaxScaler / 3 Class,0.4313,0.4952,0.4219,0.4935,0.4196,0.4877,0.4333,0.4959,0.4272,0.4902,0.4219,0.4935,0.7173,0.7481,0.5495,0.6076
1,LogisticRegression,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5729,0.6126,0.5711,0.6121,0.5645,0.6057,0.5763,0.6164,0.5685,0.6079,0.5711,0.6121,0.5694,0.6115,0.5702,0.6118
2,LogisticRegression,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.5837,0.6212,0.5762,0.6169,0.5408,0.5779,0.6117,0.6462,0.5575,0.5887,0.5762,0.6169,0.5686,0.6126,0.5723,0.6147
3,LogisticRegression,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5770,0.6065,0.5746,0.6053,0.5681,0.5993,0.5802,0.6103,0.5717,0.6013,0.5746,0.6053,0.5721,0.6042,0.5733,0.6047
4,LogisticRegression,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5851,0.6085,0.5696,0.6043,0.5381,0.5658,0.6125,0.6346,0.5527,0.5789,0.5696,0.6043,0.5541,0.6000,0.5617,0.6021
5,LogisticRegression,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5783,0.6185,0.5651,0.6081,0.5327,0.5725,0.6063,0.6433,0.5492,0.5824,0.5651,0.6081,0.5519,0.5977,0.5584,0.6028
6,LogisticRegression,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5729,0.6047,0.5729,0.6067,0.5649,0.5990,0.5760,0.6087,0.5703,0.6026,0.5729,0.6067,0.5729,0.6087,0.5728,0.6077


In [155]:
df_logreg_final=pd.concat([df_logreg_results, df_logreg_StandardScaler, df_logreg_MinMaxScaler,
                            df_logreg_results_fe, df_logreg_StandardScaler_fe, df_logreg_MinMaxScaler_fe], ignore_index=True)

df_logreg_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,Normal / No Scaler / 3 Class,0.4341,0.4821,0.4246,0.4782,0.4226,0.4748,0.4351,0.4835,0.4276,0.4751,0.4246,0.4782,0.7125,0.7376,0.5494,0.5939
1,LogisticRegression,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5647,0.5962,0.5698,0.6049,0.5600,0.5932,0.5687,0.6004,0.5671,0.6005,0.5698,0.6049,0.5749,0.6136,0.5723,0.6092
2,LogisticRegression,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.5838,0.6119,0.5650,0.6093,0.5357,0.5698,0.6110,0.6378,0.5498,0.5826,0.5650,0.6093,0.5462,0.6067,0.5555,0.6080
3,LogisticRegression,Normal / No Scaler / 2 Class IS_W / SMOTE,0.6196,0.6360,0.5822,0.5992,0.5803,0.5995,0.6070,0.6247,0.5911,0.6121,0.5822,0.5992,0.5448,0.5623,0.5631,0.5805
4,LogisticRegression,Normal / No Scaler / 2 Class S_IW / SMOTE,0.7418,0.7503,0.5170,0.5329,0.4827,0.5091,0.6677,0.6830,0.5640,0.6165,0.5170,0.5329,0.2921,0.3155,0.3884,0.4098
5,LogisticRegression,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5837,0.6154,0.5668,0.6065,0.5363,0.5702,0.6110,0.6406,0.5509,0.5810,0.5668,0.6065,0.5499,0.5976,0.5582,0.6020
6,LogisticRegression,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5701,0.5938,0.5760,0.6033,0.5658,0.5911,0.5743,0.5979,0.5730,0.5991,0.5760,0.6033,0.5819,0.6129,0.5789,0.6081
7,LogisticRegression,Normal / StandardScaler / 3 Class,0.4314,0.4780,0.4198,0.4742,0.4181,0.4707,0.4321,0.4793,0.4237,0.4710,0.4198,0.4742,0.7109,0.7354,0.5456,0.5905
8,LogisticRegression,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5660,0.5931,0.5715,0.6025,0.5614,0.5903,0.5700,0.5972,0.5688,0.5983,0.5715,0.6025,0.5769,0.6119,0.5742,0.6072
9,LogisticRegression,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.5783,0.6109,0.5632,0.6063,0.5323,0.5679,0.6063,0.6368,0.5481,0.5804,0.5632,0.6063,0.5481,0.6017,0.5556,0.6040


----

### Linear Support Vector Machine (Linear SVM)

#### Normal Data 

##### NO SCALER

In [156]:
# Modelo base
linear_svc = svm.SVC(kernel="linear")
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_linear_svc_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,Normal / No Scaler / 3 Class,0.4602,0.4924,0.4091,0.4401,0.3672,0.4006,0.4080,0.4407,0.3686,0.4920,0.4091,0.4401,0.6738,0.6906,0.5249,0.5513
1,linear_svc,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6333,0.6346,0.5804,0.5815,0.5727,0.5743,0.6062,0.6078,0.6072,0.6103,0.5804,0.5815,0.5274,0.5284,0.5532,0.5543
2,linear_svc,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7534,0.5000,0.5014,0.4295,0.4324,0.6466,0.6482,0.3764,0.4766,0.5000,0.5014,0.2473,0.2493,0.3516,0.3536
3,linear_svc,Normal / No Scaler / 2 Class IS_W / SMOTE,0.6333,0.6332,0.5815,0.5814,0.5745,0.5752,0.6074,0.6080,0.6061,0.6082,0.5815,0.5814,0.5298,0.5296,0.5550,0.5549
4,linear_svc,Normal / No Scaler / 2 Class S_IW / SMOTE,0.7500,0.7534,0.4982,0.5023,0.4286,0.4350,0.6452,0.6494,0.3760,0.5268,0.4982,0.5023,0.2464,0.2512,0.3503,0.3552
5,linear_svc,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6071,0.6223,0.5730,0.5989,0.5501,0.5697,0.6307,0.6456,0.5578,0.5764,0.5730,0.5989,0.5390,0.5756,0.5557,0.5871
6,linear_svc,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5852,0.5948,0.5932,0.6056,0.5816,0.5920,0.5890,0.5983,0.5899,0.6018,0.5932,0.6056,0.6011,0.6163,0.5971,0.6109


##### StandardScaler

In [157]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_linear_svc_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,Normal / StandardScaler / 3 Class,0.4574,0.4921,0.4062,0.4397,0.3628,0.3993,0.4041,0.4399,0.3610,0.4731,0.4062,0.4397,0.6719,0.6910,0.5223,0.5512
1,linear_svc,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6333,0.6346,0.5798,0.5806,0.5717,0.5725,0.6055,0.6065,0.6079,0.6108,0.5798,0.5806,0.5263,0.5266,0.5523,0.5530
2,linear_svc,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7431,0.7531,0.4936,0.5021,0.4263,0.4349,0.6418,0.6492,0.3751,0.4934,0.4936,0.5021,0.2441,0.2511,0.3471,0.3551
3,linear_svc,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5743,0.5982,0.5819,0.6064,0.5710,0.5951,0.5786,0.6024,0.5785,0.6020,0.5819,0.6064,0.5895,0.6147,0.5856,0.6105
4,linear_svc,Normal / StandardScaler / 2 Class S_IW,0.5838,0.6064,0.5706,0.6024,0.5381,0.5638,0.6112,0.6326,0.5538,0.5775,0.5706,0.6024,0.5574,0.5984,0.5639,0.6004
5,linear_svc,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6057,0.6264,0.5703,0.6021,0.5478,0.5731,0.6293,0.6492,0.5555,0.5791,0.5703,0.6021,0.5348,0.5779,0.5522,0.5898
6,linear_svc,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5743,0.5962,0.5823,0.6064,0.5707,0.5934,0.5780,0.5998,0.5794,0.6024,0.5823,0.6064,0.5904,0.6167,0.5863,0.6115


##### MinMax

In [158]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_linear_svc_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,Normal / MinMaxScaler / 3 Class,0.4478,0.4863,0.3956,0.4321,0.3508,0.3913,0.3914,0.4317,0.3638,0.4632,0.3956,0.4321,0.6633,0.6843,0.5121,0.5438
1,linear_svc,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5934,0.6154,0.4992,0.5232,0.4031,0.4295,0.4728,0.4968,0.4873,0.5439,0.4992,0.5232,0.4050,0.4311,0.4495,0.4747
2,linear_svc,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7458,0.7531,0.4954,0.5012,0.4272,0.4322,0.6431,0.6480,0.3755,0.4433,0.4954,0.5012,0.2450,0.2492,0.3484,0.3534
3,linear_svc,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5783,0.5975,0.5846,0.6060,0.5738,0.5944,0.5821,0.6016,0.5816,0.6017,0.5846,0.6060,0.5909,0.6145,0.5877,0.6102
4,linear_svc,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5865,0.6075,0.5649,0.6003,0.5367,0.5629,0.6128,0.6331,0.5506,0.5761,0.5649,0.6003,0.5434,0.5931,0.5540,0.5966
5,linear_svc,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6016,0.6312,0.5824,0.6035,0.5532,0.5757,0.6268,0.6530,0.5649,0.5807,0.5824,0.6035,0.5632,0.5757,0.5727,0.5893
6,linear_svc,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5454,0.5814,0.5632,0.6014,0.5434,0.5799,0.5460,0.5824,0.5616,0.6008,0.5632,0.6014,0.5809,0.6214,0.5719,0.6113


#### FE_DATA

##### NO SCALER

In [159]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_linear_svc_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,FE / No Scaler / 3 Class,0.4602,0.4962,0.4100,0.4449,0.3693,0.4073,0.4097,0.4465,0.3657,0.4946,0.4100,0.4449,0.6770,0.6945,0.5267,0.5558
1,linear_svc,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6319,0.6332,0.5798,0.5810,0.5724,0.5744,0.6056,0.6075,0.6043,0.6082,0.5798,0.5810,0.5277,0.5287,0.5531,0.5543
2,linear_svc,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7500,0.7534,0.4982,0.5023,0.4286,0.4350,0.6452,0.6494,0.3760,0.5268,0.4982,0.5023,0.2464,0.2512,0.3503,0.3552
3,linear_svc,FE / No Scaler / 2 Class IS_W / SMOTE,0.6319,0.6332,0.5798,0.5810,0.5724,0.5744,0.6056,0.6075,0.6043,0.6082,0.5798,0.5810,0.5277,0.5287,0.5531,0.5543
4,linear_svc,FE / No Scaler / 2 Class S_IW / SMOTE,0.7486,0.7524,0.4973,0.5016,0.4281,0.4346,0.6445,0.6489,0.3759,0.4667,0.4973,0.5016,0.2459,0.2509,0.3497,0.3547
5,linear_svc,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6072,0.6384,0.5657,0.6110,0.5449,0.5833,0.6289,0.6597,0.5535,0.5872,0.5657,0.6110,0.5241,0.5837,0.5443,0.5972
6,linear_svc,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5770,0.6126,0.5677,0.6062,0.5584,0.5977,0.5737,0.6111,0.5708,0.6078,0.5677,0.6062,0.5583,0.5997,0.5628,0.6028


##### StandardScaler

In [160]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_linear_svc_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,FE / StandardScaler / 3 Class,0.4657,0.5014,0.4142,0.4494,0.3675,0.4095,0.4097,0.4495,0.3542,0.5171,0.4142,0.4494,0.6781,0.6969,0.5298,0.5596
1,linear_svc,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6319,0.6332,0.5798,0.5810,0.5724,0.5744,0.6056,0.6075,0.6043,0.6082,0.5798,0.5810,0.5277,0.5287,0.5531,0.5543
2,linear_svc,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7445,0.7534,0.4945,0.5028,0.4267,0.4363,0.6424,0.6500,0.3753,0.5036,0.4945,0.5028,0.2445,0.2521,0.3477,0.3560
3,linear_svc,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5715,0.6130,0.5716,0.6134,0.5638,0.6060,0.5747,0.6163,0.5698,0.6096,0.5716,0.6134,0.5718,0.6139,0.5717,0.6136
4,linear_svc,FE / StandardScaler / 2 Class S_IW,0.5838,0.6181,0.5612,0.6134,0.5335,0.5747,0.6107,0.6433,0.5468,0.5861,0.5612,0.6134,0.5387,0.6087,0.5498,0.6111
5,linear_svc,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5962,0.6380,0.5546,0.6173,0.5331,0.5858,0.6181,0.6595,0.5436,0.5917,0.5546,0.6173,0.5130,0.5966,0.5330,0.6068
6,linear_svc,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5757,0.6126,0.5665,0.6042,0.5586,0.5969,0.5737,0.6111,0.5695,0.6059,0.5665,0.6042,0.5574,0.5958,0.5618,0.5999


##### MinMax

In [161]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(linear_svc))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="linear_svc",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_linear_svc_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_linear_svc_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,FE / MinMaxScaler / 3 Class,0.4671,0.5034,0.4160,0.4503,0.3696,0.4066,0.4115,0.4484,0.3797,0.5299,0.4160,0.4503,0.6798,0.6987,0.5316,0.5609
1,linear_svc,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6305,0.6329,0.5787,0.5807,0.5712,0.5742,0.6045,0.6072,0.6023,0.6077,0.5787,0.5807,0.5268,0.5285,0.5521,0.5540
2,linear_svc,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7514,0.7527,0.4991,0.5000,0.4290,0.4295,0.6459,0.6466,0.3762,0.3764,0.4991,0.5000,0.2468,0.2473,0.3510,0.3516
3,linear_svc,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.6072,0.6267,0.5755,0.5973,0.5690,0.5962,0.5942,0.6187,0.5807,0.6064,0.5755,0.5973,0.5439,0.5680,0.5593,0.5824
4,linear_svc,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5769,0.6154,0.5567,0.6135,0.5277,0.5733,0.6044,0.6409,0.5431,0.5859,0.5567,0.6135,0.5365,0.6116,0.5464,0.6125
5,linear_svc,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6044,0.6270,0.5657,0.6072,0.5436,0.5761,0.6271,0.6501,0.5519,0.5830,0.5657,0.6072,0.5269,0.5874,0.5457,0.5972
6,linear_svc,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.6003,0.6212,0.5776,0.5991,0.5675,0.5943,0.5895,0.6139,0.5784,0.6090,0.5776,0.5991,0.5548,0.5770,0.5659,0.5878


In [162]:
df_linear_svc_final=pd.concat([df_linear_svc_results, df_linear_svc_StandardScaler, df_linear_svc_MinMaxScaler,
                            df_linear_svc_results_fe, df_linear_svc_StandardScaler_fe, df_linear_svc_MinMaxScaler_fe], ignore_index=True)

df_linear_svc_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,linear_svc,Normal / No Scaler / 3 Class,0.4602,0.4924,0.4091,0.4401,0.3672,0.4006,0.4080,0.4407,0.3686,0.4920,0.4091,0.4401,0.6738,0.6906,0.5249,0.5513
1,linear_svc,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6333,0.6346,0.5804,0.5815,0.5727,0.5743,0.6062,0.6078,0.6072,0.6103,0.5804,0.5815,0.5274,0.5284,0.5532,0.5543
2,linear_svc,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7534,0.5000,0.5014,0.4295,0.4324,0.6466,0.6482,0.3764,0.4766,0.5000,0.5014,0.2473,0.2493,0.3516,0.3536
3,linear_svc,Normal / No Scaler / 2 Class IS_W / SMOTE,0.6333,0.6332,0.5815,0.5814,0.5745,0.5752,0.6074,0.6080,0.6061,0.6082,0.5815,0.5814,0.5298,0.5296,0.5550,0.5549
4,linear_svc,Normal / No Scaler / 2 Class S_IW / SMOTE,0.7500,0.7534,0.4982,0.5023,0.4286,0.4350,0.6452,0.6494,0.3760,0.5268,0.4982,0.5023,0.2464,0.2512,0.3503,0.3552
5,linear_svc,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6071,0.6223,0.5730,0.5989,0.5501,0.5697,0.6307,0.6456,0.5578,0.5764,0.5730,0.5989,0.5390,0.5756,0.5557,0.5871
6,linear_svc,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5852,0.5948,0.5932,0.6056,0.5816,0.5920,0.5890,0.5983,0.5899,0.6018,0.5932,0.6056,0.6011,0.6163,0.5971,0.6109
7,linear_svc,Normal / StandardScaler / 3 Class,0.4574,0.4921,0.4062,0.4397,0.3628,0.3993,0.4041,0.4399,0.3610,0.4731,0.4062,0.4397,0.6719,0.6910,0.5223,0.5512
8,linear_svc,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6333,0.6346,0.5798,0.5806,0.5717,0.5725,0.6055,0.6065,0.6079,0.6108,0.5798,0.5806,0.5263,0.5266,0.5523,0.5530
9,linear_svc,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7431,0.7531,0.4936,0.5021,0.4263,0.4349,0.6418,0.6492,0.3751,0.4934,0.4936,0.5021,0.2441,0.2511,0.3471,0.3551


----

## SGD Classifier 

#### Normal Data

##### NO SCALER

In [163]:
# Modelo base
SGD_classifier = SGDClassifier(loss="hinge")

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SGD_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,Normal / No Scaler / 3 Class,0.4218,0.4433,0.3867,0.4094,0.3039,0.3224,0.3335,0.3518,0.3139,0.3739,0.3867,0.4094,0.6733,0.6842,0.5099,0.5292
1,SGD_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5630,0.5667,0.5011,0.5049,0.3601,0.3703,0.4110,0.4194,0.3811,0.3760,0.5011,0.5049,0.4392,0.4432,0.4673,0.4712
2,SGD_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6527,0.6531,0.5084,0.5018,0.4017,0.3890,0.5458,0.5426,0.3845,0.4548,0.5084,0.5018,0.3640,0.3506,0.4177,0.4064
3,SGD_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5357,0.5481,0.5298,0.5421,0.4197,0.4356,0.4365,0.4515,0.3953,0.4814,0.5298,0.5421,0.5239,0.5361,0.5250,0.5373
4,SGD_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5467,0.5278,0.5628,0.5366,0.4589,0.4366,0.5152,0.5008,0.5236,0.5155,0.5628,0.5366,0.5789,0.5455,0.5637,0.5337
5,SGD_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.3035,0.2981,0.5298,0.5161,0.2716,0.2688,0.2013,0.2021,0.4193,0.5720,0.5298,0.5161,0.7562,0.7340,0.6326,0.6151
6,SGD_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5305,0.5456,0.5216,0.5390,0.4139,0.4277,0.4303,0.4435,0.4916,0.5900,0.5216,0.5390,0.5127,0.5323,0.5153,0.5337


##### StandardScaler

In [164]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SGD_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,Normal / StandardScaler / 3 Class,0.3859,0.4317,0.3850,0.4263,0.3722,0.4174,0.3796,0.4277,0.3896,0.4347,0.3850,0.4263,0.6905,0.7142,0.5150,0.5516
1,SGD_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5467,0.5855,0.5239,0.5625,0.5145,0.5556,0.5378,0.5774,0.5181,0.5654,0.5239,0.5625,0.5010,0.5395,0.5121,0.5507
2,SGD_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6593,0.6923,0.5107,0.5433,0.4982,0.5385,0.6397,0.6716,0.5131,0.5625,0.5107,0.5433,0.3622,0.3943,0.4282,0.4621
3,SGD_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5481,0.5508,0.5500,0.5574,0.5218,0.5325,0.5322,0.5400,0.5539,0.5691,0.5500,0.5574,0.5519,0.5639,0.5505,0.5603
4,SGD_classifier,Normal / StandardScaler / 2 Class S_IW,0.4958,0.4815,0.5606,0.5306,0.4722,0.4553,0.5080,0.4971,0.5492,0.5262,0.5606,0.5306,0.6253,0.5797,0.5908,0.5534
5,SGD_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5808,0.5577,0.5685,0.5412,0.5291,0.5072,0.6012,0.5824,0.5545,0.5321,0.5685,0.5412,0.5561,0.5246,0.5612,0.5321
6,SGD_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5522,0.5515,0.5566,0.5520,0.5464,0.5434,0.5557,0.5545,0.5548,0.5500,0.5566,0.5520,0.5610,0.5525,0.5588,0.5522


##### MinMax

In [165]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SGD_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,Normal / MinMaxScaler / 3 Class,0.4547,0.4653,0.4081,0.4191,0.3384,0.3589,0.3741,0.3945,0.3897,0.5167,0.4081,0.4191,0.6755,0.6825,0.5248,0.5348
1,SGD_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6004,0.6123,0.5589,0.5655,0.5315,0.5396,0.5649,0.5747,0.5553,0.5850,0.5589,0.5655,0.5174,0.5187,0.5373,0.5412
2,SGD_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7280,0.7527,0.5060,0.5401,0.4673,0.5162,0.6551,0.6869,0.5081,0.6376,0.5060,0.5401,0.2839,0.3275,0.3783,0.4198
3,SGD_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5606,0.5714,0.5469,0.5629,0.4992,0.5158,0.5200,0.5334,0.6099,0.5767,0.5469,0.5629,0.5333,0.5544,0.5392,0.5578
4,SGD_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6812,0.6758,0.5738,0.5748,0.5559,0.5624,0.6711,0.6713,0.5699,0.5915,0.5738,0.5748,0.4663,0.4738,0.5149,0.5203
5,SGD_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5834,0.6107,0.5253,0.5596,0.4662,0.4903,0.5565,0.5736,0.5511,0.5822,0.5253,0.5596,0.4673,0.5085,0.4890,0.5265
6,SGD_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5262,0.5241,0.5699,0.5672,0.4865,0.4885,0.4722,0.4753,0.6078,0.6165,0.5699,0.5672,0.6136,0.6104,0.5908,0.5879


#### FE_DATA

##### NO SCALER

In [166]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SGD_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,FE / No Scaler / 3 Class,0.3901,0.3894,0.3743,0.3751,0.2683,0.2634,0.2796,0.2749,0.3771,0.4131,0.3743,0.3751,0.6719,0.6742,0.5010,0.5027
1,SGD_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5672,0.5862,0.5233,0.5420,0.4363,0.4565,0.4734,0.4941,0.4779,0.4905,0.5233,0.5420,0.4793,0.4977,0.4994,0.5179
2,SGD_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.6426,0.6559,0.5349,0.5491,0.4735,0.4909,0.6014,0.6187,0.4858,0.5011,0.5349,0.5491,0.4272,0.4422,0.4701,0.4853
3,SGD_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5771,0.5659,0.5654,0.5554,0.4888,0.4772,0.5035,0.4908,0.5589,0.5726,0.5654,0.5554,0.5536,0.5450,0.5581,0.5488
4,SGD_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6197,0.6198,0.5329,0.5264,0.4365,0.4280,0.5452,0.5417,0.4456,0.5561,0.5329,0.5264,0.4462,0.4330,0.4763,0.4658
5,SGD_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5742,0.6002,0.5270,0.5414,0.4291,0.4392,0.5133,0.5333,0.5463,0.5552,0.5270,0.5414,0.4797,0.4825,0.4925,0.5000
6,SGD_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.4724,0.4880,0.5330,0.5522,0.4048,0.4233,0.3766,0.3929,0.4340,0.4606,0.5330,0.5522,0.5937,0.6163,0.5619,0.5828


##### StandardScaler

In [167]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SGD_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,FE / StandardScaler / 3 Class,0.4012,0.4457,0.3830,0.4318,0.3740,0.4232,0.3912,0.4381,0.3853,0.4364,0.3830,0.4318,0.6817,0.7116,0.5102,0.5542
1,SGD_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5207,0.5697,0.5055,0.5555,0.5011,0.5537,0.5204,0.5706,0.5041,0.5566,0.5055,0.5555,0.4903,0.5413,0.4978,0.5483
2,SGD_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.6591,0.6886,0.5217,0.5465,0.5171,0.5364,0.6474,0.6688,0.5304,0.5482,0.5217,0.5465,0.3843,0.4044,0.4468,0.4686
3,SGD_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5907,0.5625,0.5877,0.5635,0.5766,0.5502,0.5891,0.5604,0.5865,0.5649,0.5877,0.5635,0.5847,0.5646,0.5861,0.5639
4,SGD_classifier,FE / StandardScaler / 2 Class S_IW,0.5027,0.5192,0.5037,0.5361,0.4617,0.4848,0.5301,0.5447,0.5023,0.5282,0.5037,0.5361,0.5046,0.5530,0.5031,0.5437
5,SGD_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5781,0.5656,0.5295,0.5385,0.5113,0.5132,0.6017,0.5930,0.5222,0.5296,0.5295,0.5385,0.4809,0.5113,0.5041,0.5246
6,SGD_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5261,0.5587,0.5169,0.5532,0.5128,0.5496,0.5282,0.5625,0.5159,0.5521,0.5169,0.5532,0.5076,0.5478,0.5122,0.5505


##### MinMax

In [168]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SGD_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SGD_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SGD_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SGD_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,FE / MinMaxScaler / 3 Class,0.4081,0.4540,0.3928,0.4409,0.3465,0.3937,0.3645,0.4106,0.3935,0.5339,0.3928,0.4409,0.6929,0.7181,0.5211,0.5627
1,SGD_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5976,0.6140,0.5613,0.5776,0.5514,0.5640,0.5788,0.5913,0.5790,0.6045,0.5613,0.5776,0.5251,0.5412,0.5427,0.5588
2,SGD_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.6385,0.6738,0.5247,0.5558,0.5000,0.5244,0.6239,0.6520,0.5139,0.5358,0.5247,0.5558,0.4108,0.4378,0.4610,0.4893
3,SGD_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5757,0.5803,0.5523,0.5560,0.4823,0.4792,0.5062,0.5057,0.6509,0.6033,0.5523,0.5560,0.5290,0.5316,0.5391,0.5422
4,SGD_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5318,0.5751,0.5400,0.5956,0.4695,0.5201,0.5264,0.5660,0.5524,0.5847,0.5400,0.5956,0.5481,0.6160,0.5407,0.6029
5,SGD_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.4767,0.5058,0.5498,0.5878,0.4636,0.4947,0.4991,0.5279,0.5425,0.5701,0.5498,0.5878,0.6229,0.6698,0.5848,0.6273
6,SGD_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5275,0.5553,0.5421,0.5753,0.4926,0.5254,0.4984,0.5271,0.5533,0.5951,0.5421,0.5753,0.5568,0.5952,0.5487,0.5845


In [169]:
df_SGD_classifier_final=pd.concat([df_SGD_classifier_results, df_SGD_classifier_StandardScaler, df_SGD_classifier_MinMaxScaler,
                            df_SGD_classifier_results_fe, df_SGD_classifier_StandardScaler_fe, df_SGD_classifier_MinMaxScaler_fe], ignore_index=True)
df_SGD_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SGD_classifier,Normal / No Scaler / 3 Class,0.4218,0.4433,0.3867,0.4094,0.3039,0.3224,0.3335,0.3518,0.3139,0.3739,0.3867,0.4094,0.6733,0.6842,0.5099,0.5292
1,SGD_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5630,0.5667,0.5011,0.5049,0.3601,0.3703,0.4110,0.4194,0.3811,0.3760,0.5011,0.5049,0.4392,0.4432,0.4673,0.4712
2,SGD_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6527,0.6531,0.5084,0.5018,0.4017,0.3890,0.5458,0.5426,0.3845,0.4548,0.5084,0.5018,0.3640,0.3506,0.4177,0.4064
3,SGD_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5357,0.5481,0.5298,0.5421,0.4197,0.4356,0.4365,0.4515,0.3953,0.4814,0.5298,0.5421,0.5239,0.5361,0.5250,0.5373
4,SGD_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5467,0.5278,0.5628,0.5366,0.4589,0.4366,0.5152,0.5008,0.5236,0.5155,0.5628,0.5366,0.5789,0.5455,0.5637,0.5337
5,SGD_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.3035,0.2981,0.5298,0.5161,0.2716,0.2688,0.2013,0.2021,0.4193,0.5720,0.5298,0.5161,0.7562,0.7340,0.6326,0.6151
6,SGD_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5305,0.5456,0.5216,0.5390,0.4139,0.4277,0.4303,0.4435,0.4916,0.5900,0.5216,0.5390,0.5127,0.5323,0.5153,0.5337
7,SGD_classifier,Normal / StandardScaler / 3 Class,0.3859,0.4317,0.3850,0.4263,0.3722,0.4174,0.3796,0.4277,0.3896,0.4347,0.3850,0.4263,0.6905,0.7142,0.5150,0.5516
8,SGD_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5467,0.5855,0.5239,0.5625,0.5145,0.5556,0.5378,0.5774,0.5181,0.5654,0.5239,0.5625,0.5010,0.5395,0.5121,0.5507
9,SGD_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6593,0.6923,0.5107,0.5433,0.4982,0.5385,0.6397,0.6716,0.5131,0.5625,0.5107,0.5433,0.3622,0.3943,0.4282,0.4621


----

## Non-Linear Models  

### Decision Tree

#### Normal Data

##### NO SCALER

In [170]:
# Modelo base
Tree_classifier = DecisionTreeClassifier(max_depth=5,
                                         min_samples_leaf=10,
                                         min_samples_split=20, random_state=42)

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_Tree_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,Normal / No Scaler / 3 Class,0.4245,0.5505,0.3959,0.5256,0.3880,0.5197,0.4091,0.5352,0.4218,0.5704,0.3959,0.5256,0.6749,0.7460,0.5165,0.6262
1,Tree_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6112,0.6847,0.5710,0.6462,0.5689,0.6480,0.5970,0.6716,0.5830,0.6725,0.5710,0.6462,0.5307,0.6077,0.5504,0.6267
2,Tree_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7349,0.7833,0.5404,0.5991,0.5297,0.6068,0.6865,0.7399,0.6006,0.7391,0.5404,0.5991,0.3459,0.4149,0.4320,0.4983
3,Tree_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5769,0.6559,0.5702,0.6522,0.5659,0.6465,0.5788,0.6580,0.5698,0.6483,0.5702,0.6522,0.5635,0.6485,0.5668,0.6503
4,Tree_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6182,0.7108,0.5114,0.6246,0.5057,0.6127,0.6225,0.7083,0.5107,0.6281,0.5114,0.6246,0.4046,0.5384,0.4541,0.5786
5,Tree_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6210,0.7026,0.5581,0.6592,0.5393,0.6317,0.6310,0.7074,0.5588,0.6527,0.5581,0.6592,0.4952,0.6159,0.5243,0.6357
6,Tree_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5880,0.6308,0.5960,0.6411,0.5771,0.6229,0.5838,0.6286,0.6016,0.6419,0.5960,0.6411,0.6040,0.6514,0.5998,0.6461


##### StandardScaler

In [171]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_Tree_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,Normal / StandardScaler / 3 Class,0.4245,0.5505,0.3959,0.5256,0.3880,0.5197,0.4091,0.5352,0.4218,0.5704,0.3959,0.5256,0.6749,0.7460,0.5165,0.6262
1,Tree_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6112,0.6847,0.5710,0.6462,0.5689,0.6480,0.5970,0.6716,0.5830,0.6725,0.5710,0.6462,0.5307,0.6077,0.5504,0.6267
2,Tree_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7349,0.7833,0.5404,0.5991,0.5297,0.6068,0.6865,0.7399,0.6006,0.7391,0.5404,0.5991,0.3459,0.4149,0.4320,0.4983
3,Tree_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5289,0.6463,0.5376,0.6574,0.5259,0.6429,0.5326,0.6490,0.5358,0.6524,0.5376,0.6574,0.5463,0.6684,0.5419,0.6628
4,Tree_classifier,Normal / StandardScaler / 2 Class S_IW,0.6235,0.6968,0.5597,0.6666,0.5474,0.6401,0.6410,0.7118,0.5495,0.6372,0.5597,0.6666,0.4958,0.6365,0.5266,0.6513
5,Tree_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6210,0.7026,0.5581,0.6592,0.5393,0.6317,0.6310,0.7074,0.5588,0.6527,0.5581,0.6592,0.4952,0.6159,0.5243,0.6357
6,Tree_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5880,0.6308,0.5960,0.6411,0.5771,0.6229,0.5838,0.6286,0.6016,0.6419,0.5960,0.6411,0.6040,0.6514,0.5998,0.6461


##### MinMax

In [172]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_Tree_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,Normal / MinMaxScaler / 3 Class,0.4245,0.5505,0.3959,0.5256,0.3880,0.5197,0.4091,0.5352,0.4218,0.5704,0.3959,0.5256,0.6749,0.7460,0.5165,0.6262
1,Tree_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6112,0.6847,0.5710,0.6462,0.5689,0.6480,0.5970,0.6716,0.5830,0.6725,0.5710,0.6462,0.5307,0.6077,0.5504,0.6267
2,Tree_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7349,0.7833,0.5404,0.5991,0.5297,0.6068,0.6865,0.7399,0.6006,0.7391,0.5404,0.5991,0.3459,0.4149,0.4320,0.4983
3,Tree_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5426,0.6374,0.5518,0.6524,0.5379,0.6331,0.5446,0.6375,0.5510,0.6512,0.5518,0.6524,0.5609,0.6674,0.5563,0.6598
4,Tree_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6030,0.6649,0.5833,0.6617,0.5475,0.6163,0.6227,0.6813,0.5704,0.6369,0.5833,0.6617,0.5636,0.6586,0.5724,0.6593
5,Tree_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6210,0.7026,0.5581,0.6592,0.5393,0.6317,0.6310,0.7074,0.5588,0.6527,0.5581,0.6592,0.4952,0.6159,0.5243,0.6357
6,Tree_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5880,0.6308,0.5960,0.6411,0.5771,0.6229,0.5838,0.6286,0.6016,0.6419,0.5960,0.6411,0.6040,0.6514,0.5998,0.6461


#### FE_DATA

##### NO SCALER

In [173]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_Tree_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,FE / No Scaler / 3 Class,0.4040,0.5707,0.3821,0.5512,0.3694,0.5491,0.3853,0.5608,0.3820,0.5844,0.3821,0.5512,0.6673,0.7620,0.5042,0.6481
1,Tree_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5866,0.6909,0.5447,0.6520,0.5367,0.6515,0.5675,0.6753,0.5523,0.6878,0.5447,0.6520,0.5029,0.6130,0.5233,0.6321
2,Tree_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7362,0.7960,0.5413,0.6337,0.5302,0.6519,0.6872,0.7650,0.5904,0.7568,0.5413,0.6337,0.3463,0.4713,0.4325,0.5463
3,Tree_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5605,0.6738,0.5463,0.6677,0.5359,0.6586,0.5538,0.6702,0.5466,0.6746,0.5463,0.6677,0.5321,0.6615,0.5389,0.6645
4,Tree_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6552,0.7521,0.5211,0.6371,0.5217,0.6429,0.6486,0.7423,0.5315,0.6611,0.5211,0.6371,0.3869,0.5222,0.4488,0.5765
5,Tree_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6334,0.7359,0.5533,0.6902,0.5387,0.6680,0.6385,0.7402,0.5567,0.6856,0.5533,0.6902,0.4732,0.6446,0.5103,0.6659
6,Tree_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5413,0.6734,0.5372,0.6720,0.5205,0.6573,0.5344,0.6663,0.5386,0.6846,0.5372,0.6720,0.5332,0.6705,0.5349,0.6710


##### StandardScaler

In [174]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_Tree_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,FE / StandardScaler / 3 Class,0.4054,0.5707,0.3833,0.5512,0.3704,0.5491,0.3865,0.5608,0.3841,0.5844,0.3833,0.5512,0.6677,0.7620,0.5052,0.6481
1,Tree_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5880,0.6909,0.5459,0.6520,0.5377,0.6515,0.5685,0.6753,0.5536,0.6878,0.5459,0.6520,0.5038,0.6130,0.5243,0.6321
2,Tree_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7362,0.7960,0.5413,0.6337,0.5302,0.6519,0.6872,0.7650,0.5904,0.7568,0.5413,0.6337,0.3463,0.4713,0.4325,0.5463
3,Tree_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5193,0.6662,0.5272,0.6738,0.5129,0.6563,0.5199,0.6623,0.5277,0.6793,0.5272,0.6738,0.5351,0.6814,0.5311,0.6774
4,Tree_classifier,FE / StandardScaler / 2 Class S_IW,0.6415,0.7311,0.5717,0.6810,0.5599,0.6642,0.6539,0.7391,0.5685,0.6622,0.5717,0.6810,0.5019,0.6309,0.5352,0.6552
5,Tree_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6321,0.7359,0.5524,0.6902,0.5376,0.6680,0.6372,0.7402,0.5560,0.6856,0.5524,0.6902,0.4728,0.6446,0.5096,0.6659
6,Tree_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5413,0.6734,0.5372,0.6720,0.5205,0.6573,0.5344,0.6663,0.5386,0.6846,0.5372,0.6720,0.5332,0.6705,0.5349,0.6710


##### MinMax

In [175]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(Tree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="Tree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_Tree_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_Tree_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,FE / MinMaxScaler / 3 Class,0.4053,0.5707,0.3834,0.5512,0.3704,0.5491,0.3864,0.5608,0.3830,0.5844,0.3834,0.5512,0.6682,0.7620,0.5054,0.6481
1,Tree_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5866,0.6909,0.5447,0.6520,0.5367,0.6515,0.5675,0.6753,0.5523,0.6878,0.5447,0.6520,0.5029,0.6130,0.5233,0.6321
2,Tree_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7362,0.7960,0.5413,0.6337,0.5302,0.6519,0.6872,0.7650,0.5904,0.7568,0.5413,0.6337,0.3463,0.4713,0.4325,0.5463
3,Tree_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5330,0.6916,0.5215,0.6810,0.5160,0.6770,0.5325,0.6903,0.5230,0.6823,0.5215,0.6810,0.5100,0.6703,0.5156,0.6755
4,Tree_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5783,0.6947,0.5053,0.6737,0.4932,0.6404,0.5940,0.7084,0.5076,0.6465,0.5053,0.6737,0.4324,0.6526,0.4665,0.6625
5,Tree_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6334,0.7359,0.5533,0.6902,0.5387,0.6680,0.6385,0.7402,0.5567,0.6856,0.5533,0.6902,0.4732,0.6446,0.5103,0.6659
6,Tree_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5413,0.6734,0.5372,0.6720,0.5205,0.6573,0.5344,0.6663,0.5386,0.6846,0.5372,0.6720,0.5332,0.6705,0.5349,0.6710


In [176]:
df_Tree_classifier_final=pd.concat([df_Tree_classifier_results, df_Tree_classifier_StandardScaler, df_Tree_classifier_MinMaxScaler,
                            df_Tree_classifier_results_fe, df_Tree_classifier_StandardScaler_fe, df_Tree_classifier_MinMaxScaler_fe], ignore_index=True)

df_Tree_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,Tree_classifier,Normal / No Scaler / 3 Class,0.4245,0.5505,0.3959,0.5256,0.3880,0.5197,0.4091,0.5352,0.4218,0.5704,0.3959,0.5256,0.6749,0.7460,0.5165,0.6262
1,Tree_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6112,0.6847,0.5710,0.6462,0.5689,0.6480,0.5970,0.6716,0.5830,0.6725,0.5710,0.6462,0.5307,0.6077,0.5504,0.6267
2,Tree_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7349,0.7833,0.5404,0.5991,0.5297,0.6068,0.6865,0.7399,0.6006,0.7391,0.5404,0.5991,0.3459,0.4149,0.4320,0.4983
3,Tree_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5769,0.6559,0.5702,0.6522,0.5659,0.6465,0.5788,0.6580,0.5698,0.6483,0.5702,0.6522,0.5635,0.6485,0.5668,0.6503
4,Tree_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6182,0.7108,0.5114,0.6246,0.5057,0.6127,0.6225,0.7083,0.5107,0.6281,0.5114,0.6246,0.4046,0.5384,0.4541,0.5786
5,Tree_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6210,0.7026,0.5581,0.6592,0.5393,0.6317,0.6310,0.7074,0.5588,0.6527,0.5581,0.6592,0.4952,0.6159,0.5243,0.6357
6,Tree_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5880,0.6308,0.5960,0.6411,0.5771,0.6229,0.5838,0.6286,0.6016,0.6419,0.5960,0.6411,0.6040,0.6514,0.5998,0.6461
7,Tree_classifier,Normal / StandardScaler / 3 Class,0.4245,0.5505,0.3959,0.5256,0.3880,0.5197,0.4091,0.5352,0.4218,0.5704,0.3959,0.5256,0.6749,0.7460,0.5165,0.6262
8,Tree_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6112,0.6847,0.5710,0.6462,0.5689,0.6480,0.5970,0.6716,0.5830,0.6725,0.5710,0.6462,0.5307,0.6077,0.5504,0.6267
9,Tree_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7349,0.7833,0.5404,0.5991,0.5297,0.6068,0.6865,0.7399,0.6006,0.7391,0.5404,0.5991,0.3459,0.4149,0.4320,0.4983


---

### Random Forest

#### Normal Data

##### NO SCALER

In [177]:
# Modelo base
RF_classifier = RandomForestClassifier(
    n_estimators=1000,
    max_depth=10,           
    min_samples_leaf=5,     
    bootstrap=True,
    random_state=42,
)

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_RF_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,Normal / No Scaler / 3 Class,0.4340,0.6913,0.3961,0.6521,0.3775,0.6548,0.4047,0.6744,0.4281,0.7399,0.3961,0.6521,0.6680,0.8149,0.5142,0.7289
1,RF_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6223,0.7579,0.5622,0.7082,0.5454,0.7151,0.5842,0.7381,0.5881,0.7919,0.5622,0.7082,0.5022,0.6586,0.5313,0.6829
2,RF_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7486,0.7830,0.4991,0.5611,0.4334,0.5450,0.6470,0.7113,0.4762,0.8881,0.4991,0.5611,0.2496,0.3393,0.3530,0.4362
3,RF_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5728,0.7909,0.5555,0.7816,0.5547,0.7814,0.5734,0.7909,0.5546,0.7816,0.5555,0.7816,0.5381,0.7723,0.5467,0.7769
4,RF_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6565,0.8438,0.5387,0.7876,0.5377,0.7888,0.6556,0.8432,0.5412,0.7920,0.5387,0.7876,0.4208,0.7314,0.4759,0.7589
5,RF_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6703,0.8685,0.5628,0.8413,0.5612,0.8291,0.6715,0.8706,0.5616,0.8200,0.5628,0.8413,0.4552,0.8141,0.5059,0.8275
6,RF_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5811,0.7926,0.5731,0.7906,0.5697,0.7860,0.5841,0.7938,0.5705,0.7839,0.5731,0.7906,0.5652,0.7887,0.5691,0.7897


##### StandardScaler

In [178]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_RF_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,Normal / StandardScaler / 3 Class,0.4368,0.6916,0.3984,0.6525,0.3797,0.6554,0.4070,0.6749,0.4319,0.7402,0.3984,0.6525,0.6689,0.8151,0.5160,0.7293
1,RF_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6223,0.7576,0.5628,0.7078,0.5463,0.7146,0.5848,0.7377,0.5877,0.7917,0.5628,0.7078,0.5033,0.6581,0.5322,0.6825
2,RF_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7486,0.7823,0.4991,0.5597,0.4334,0.5427,0.6470,0.7100,0.4762,0.8879,0.4991,0.5597,0.2496,0.3372,0.3530,0.4343
3,RF_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5646,0.8046,0.5473,0.7976,0.5460,0.7964,0.5649,0.8049,0.5464,0.7961,0.5473,0.7976,0.5301,0.7905,0.5386,0.7941
4,RF_classifier,Normal / StandardScaler / 2 Class S_IW,0.6868,0.8637,0.5700,0.8148,0.5691,0.8161,0.6826,0.8634,0.5718,0.8189,0.5700,0.8148,0.4533,0.7659,0.5079,0.7899
5,RF_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6703,0.8681,0.5628,0.8411,0.5612,0.8288,0.6715,0.8702,0.5616,0.8196,0.5628,0.8411,0.4552,0.8140,0.5059,0.8274
6,RF_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5797,0.7926,0.5714,0.7905,0.5681,0.7860,0.5827,0.7938,0.5688,0.7839,0.5714,0.7905,0.5631,0.7884,0.5672,0.7894


##### MinMax

In [179]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_RF_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,Normal / MinMaxScaler / 3 Class,0.4340,0.6906,0.3961,0.6514,0.3775,0.6542,0.4047,0.6738,0.4281,0.7387,0.3961,0.6514,0.6680,0.8145,0.5142,0.7284
1,RF_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6223,0.7582,0.5622,0.7085,0.5454,0.7154,0.5842,0.7384,0.5881,0.7926,0.5622,0.7085,0.5022,0.6588,0.5313,0.6832
2,RF_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7486,0.7830,0.4991,0.5611,0.4334,0.5450,0.6470,0.7113,0.4762,0.8881,0.4991,0.5611,0.2496,0.3393,0.3530,0.4362
3,RF_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5714,0.8012,0.5561,0.7935,0.5538,0.7927,0.5718,0.8014,0.5541,0.7928,0.5561,0.7935,0.5408,0.7859,0.5484,0.7897
4,RF_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6799,0.8496,0.5561,0.7956,0.5552,0.7971,0.6737,0.8492,0.5589,0.8004,0.5561,0.7956,0.4323,0.7417,0.4899,0.7681
5,RF_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6675,0.8678,0.5591,0.8408,0.5572,0.8284,0.6686,0.8699,0.5574,0.8192,0.5591,0.8408,0.4506,0.8139,0.5016,0.8272
6,RF_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5811,0.7936,0.5731,0.7915,0.5697,0.7870,0.5841,0.7948,0.5705,0.7849,0.5731,0.7915,0.5652,0.7894,0.5691,0.7904


#### FE_DATA

##### NO SCALER

In [180]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_RF_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,FE / No Scaler / 3 Class,0.4258,0.7679,0.3947,0.7388,0.3836,0.7477,0.4053,0.7606,0.4127,0.7923,0.3947,0.7388,0.6683,0.8644,0.5135,0.7991
1,RF_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5948,0.8352,0.5413,0.8020,0.5302,0.8155,0.5666,0.8280,0.5534,0.8600,0.5413,0.8020,0.4877,0.7689,0.5138,0.7853
2,RF_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7431,0.8245,0.5048,0.6465,0.4551,0.6735,0.6554,0.7857,0.5106,0.8999,0.5048,0.6465,0.2665,0.4686,0.3667,0.5503
3,RF_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5797,0.8695,0.5581,0.8621,0.5568,0.8631,0.5776,0.8693,0.5568,0.8643,0.5581,0.8621,0.5365,0.8546,0.5471,0.8583
4,RF_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6827,0.8867,0.5748,0.8431,0.5723,0.8464,0.6815,0.8861,0.5759,0.8505,0.5748,0.8431,0.4668,0.7995,0.5176,0.8210
5,RF_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6977,0.9282,0.5810,0.9076,0.5801,0.9043,0.6919,0.9285,0.5847,0.9014,0.5810,0.9076,0.4643,0.8869,0.5189,0.8972
6,RF_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5769,0.8740,0.5606,0.8680,0.5586,0.8682,0.5769,0.8739,0.5588,0.8684,0.5606,0.8680,0.5444,0.8620,0.5524,0.8650


##### StandardScaler

In [181]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_RF_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,FE / StandardScaler / 3 Class,0.4258,0.7672,0.3947,0.7384,0.3837,0.7473,0.4053,0.7600,0.4126,0.7915,0.3947,0.7384,0.6682,0.8641,0.5134,0.7988
1,RF_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5934,0.8355,0.5395,0.8023,0.5281,0.8158,0.5647,0.8284,0.5514,0.8606,0.5395,0.8023,0.4857,0.7691,0.5119,0.7855
2,RF_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7431,0.8242,0.5048,0.6458,0.4551,0.6727,0.6554,0.7852,0.5106,0.8996,0.5048,0.6458,0.2665,0.4675,0.3667,0.5494
3,RF_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5659,0.8716,0.5485,0.8657,0.5461,0.8657,0.5654,0.8715,0.5460,0.8658,0.5485,0.8657,0.5311,0.8598,0.5397,0.8628
4,RF_classifier,FE / StandardScaler / 2 Class S_IW,0.6895,0.9093,0.5756,0.8708,0.5759,0.8763,0.6865,0.9086,0.5805,0.8825,0.5756,0.8708,0.4616,0.8322,0.5152,0.8512
5,RF_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6977,0.9275,0.5810,0.9071,0.5800,0.9035,0.6919,0.9278,0.5843,0.9002,0.5810,0.9071,0.4643,0.8867,0.5189,0.8968
6,RF_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5755,0.8740,0.5595,0.8681,0.5574,0.8682,0.5756,0.8740,0.5576,0.8683,0.5595,0.8681,0.5435,0.8623,0.5514,0.8652


##### MinMax

In [182]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(RF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="RF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_RF_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_RF_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,FE / MinMaxScaler / 3 Class,0.4258,0.7675,0.3947,0.7385,0.3836,0.7474,0.4053,0.7602,0.4127,0.7921,0.3947,0.7385,0.6683,0.8642,0.5135,0.7989
1,RF_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5948,0.8352,0.5407,0.8020,0.5292,0.8155,0.5659,0.8281,0.5533,0.8600,0.5407,0.8020,0.4866,0.7689,0.5129,0.7853
2,RF_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7445,0.8245,0.5076,0.6465,0.4602,0.6735,0.6582,0.7857,0.5262,0.8999,0.5076,0.6465,0.2706,0.4686,0.3706,0.5503
3,RF_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5618,0.8733,0.5439,0.8673,0.5412,0.8674,0.5608,0.8732,0.5413,0.8678,0.5439,0.8673,0.5261,0.8613,0.5349,0.8643
4,RF_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.7019,0.9069,0.5875,0.8650,0.5869,0.8723,0.6965,0.9059,0.5932,0.8808,0.5875,0.8650,0.4731,0.8230,0.5268,0.8437
5,RF_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6977,0.9279,0.5810,0.9073,0.5801,0.9039,0.6919,0.9282,0.5847,0.9009,0.5810,0.9073,0.4643,0.8868,0.5189,0.8970
6,RF_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5755,0.8743,0.5595,0.8684,0.5574,0.8685,0.5757,0.8743,0.5576,0.8687,0.5595,0.8684,0.5435,0.8625,0.5514,0.8655


In [183]:
df_RF_classifier_final=pd.concat([df_RF_classifier_results, df_RF_classifier_StandardScaler, df_RF_classifier_MinMaxScaler,
                            df_RF_classifier_results_fe, df_RF_classifier_StandardScaler_fe, df_RF_classifier_MinMaxScaler_fe], ignore_index=True)

df_RF_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,RF_classifier,Normal / No Scaler / 3 Class,0.4340,0.6913,0.3961,0.6521,0.3775,0.6548,0.4047,0.6744,0.4281,0.7399,0.3961,0.6521,0.6680,0.8149,0.5142,0.7289
1,RF_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6223,0.7579,0.5622,0.7082,0.5454,0.7151,0.5842,0.7381,0.5881,0.7919,0.5622,0.7082,0.5022,0.6586,0.5313,0.6829
2,RF_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7486,0.7830,0.4991,0.5611,0.4334,0.5450,0.6470,0.7113,0.4762,0.8881,0.4991,0.5611,0.2496,0.3393,0.3530,0.4362
3,RF_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5728,0.7909,0.5555,0.7816,0.5547,0.7814,0.5734,0.7909,0.5546,0.7816,0.5555,0.7816,0.5381,0.7723,0.5467,0.7769
4,RF_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6565,0.8438,0.5387,0.7876,0.5377,0.7888,0.6556,0.8432,0.5412,0.7920,0.5387,0.7876,0.4208,0.7314,0.4759,0.7589
5,RF_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6703,0.8685,0.5628,0.8413,0.5612,0.8291,0.6715,0.8706,0.5616,0.8200,0.5628,0.8413,0.4552,0.8141,0.5059,0.8275
6,RF_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5811,0.7926,0.5731,0.7906,0.5697,0.7860,0.5841,0.7938,0.5705,0.7839,0.5731,0.7906,0.5652,0.7887,0.5691,0.7897
7,RF_classifier,Normal / StandardScaler / 3 Class,0.4368,0.6916,0.3984,0.6525,0.3797,0.6554,0.4070,0.6749,0.4319,0.7402,0.3984,0.6525,0.6689,0.8151,0.5160,0.7293
8,RF_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6223,0.7576,0.5628,0.7078,0.5463,0.7146,0.5848,0.7377,0.5877,0.7917,0.5628,0.7078,0.5033,0.6581,0.5322,0.6825
9,RF_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7486,0.7823,0.4991,0.5597,0.4334,0.5427,0.6470,0.7100,0.4762,0.8879,0.4991,0.5597,0.2496,0.3372,0.3530,0.4343


---

### Extra Trees 

#### Normal Data

##### NO SCALER

In [184]:

# Modelo base
extratree_classifier = ExtraTreesClassifier(max_depth=10)

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_extratree_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,Normal / No Scaler / 3 Class,0.4286,0.7977,0.3867,0.7677,0.3621,0.7834,0.3929,0.7922,0.3902,0.8529,0.3867,0.7677,0.6625,0.8729,0.5060,0.8186
1,extratree_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6333,0.8166,0.5702,0.7750,0.5509,0.7889,0.5908,0.8048,0.6060,0.8579,0.5702,0.7750,0.5070,0.7334,0.5376,0.7539
2,extratree_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7404,0.8424,0.4992,0.6813,0.4444,0.7180,0.6493,0.8126,0.4748,0.9135,0.4992,0.6813,0.2581,0.5201,0.3589,0.5952
3,extratree_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5659,0.8489,0.5557,0.8546,0.5532,0.8453,0.5689,0.8502,0.5536,0.8425,0.5557,0.8546,0.5455,0.8603,0.5506,0.8575
4,extratree_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6401,0.8372,0.5633,0.8368,0.5541,0.8019,0.6526,0.8441,0.5546,0.7856,0.5633,0.8368,0.4864,0.8365,0.5232,0.8366
5,extratree_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6951,0.9131,0.5625,0.8798,0.5629,0.8832,0.6840,0.9130,0.5708,0.8924,0.5625,0.8798,0.4298,0.8465,0.4913,0.8629
6,extratree_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5866,0.8492,0.5764,0.8565,0.5735,0.8461,0.5888,0.8507,0.5742,0.8432,0.5764,0.8565,0.5662,0.8638,0.5712,0.8602


##### StandardScaler

In [185]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_extratree_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,Normal / StandardScaler / 3 Class,0.4272,0.7960,0.3865,0.7659,0.3636,0.7820,0.3927,0.7906,0.4015,0.8528,0.3865,0.7659,0.6624,0.8716,0.5059,0.8171
1,extratree_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6333,0.8197,0.5713,0.7789,0.5538,0.7930,0.5929,0.8085,0.6096,0.8597,0.5713,0.7789,0.5093,0.7381,0.5393,0.7582
2,extratree_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7445,0.8427,0.5094,0.6819,0.4644,0.7192,0.6601,0.8133,0.5332,0.9136,0.5094,0.6819,0.2744,0.5212,0.3738,0.5961
3,extratree_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5646,0.8290,0.5578,0.8377,0.5540,0.8258,0.5681,0.8306,0.5553,0.8251,0.5578,0.8377,0.5509,0.8464,0.5543,0.8420
4,extratree_classifier,Normal / StandardScaler / 2 Class S_IW,0.6552,0.8709,0.5528,0.8475,0.5466,0.8342,0.6574,0.8733,0.5487,0.8316,0.5528,0.8475,0.4504,0.8242,0.4983,0.8356
5,extratree_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6841,0.9100,0.5477,0.8773,0.5473,0.8791,0.6726,0.9099,0.5544,0.8871,0.5477,0.8773,0.4113,0.8445,0.4742,0.8607
6,extratree_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5797,0.8458,0.5737,0.8548,0.5692,0.8430,0.5828,0.8473,0.5709,0.8412,0.5737,0.8548,0.5676,0.8637,0.5706,0.8592


##### MinMax

In [186]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_extratree_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,Normal / MinMaxScaler / 3 Class,0.4313,0.7960,0.3893,0.7655,0.3654,0.7811,0.3962,0.7903,0.3959,0.8516,0.3893,0.7655,0.6652,0.8719,0.5088,0.8170
1,extratree_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6388,0.8207,0.5770,0.7793,0.5607,0.7936,0.5993,0.8092,0.6134,0.8631,0.5770,0.7793,0.5153,0.7379,0.5453,0.7583
2,extratree_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7417,0.8407,0.5039,0.6778,0.4539,0.7141,0.6542,0.8103,0.5305,0.9127,0.5039,0.6778,0.2660,0.5149,0.3660,0.5907
3,extratree_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5700,0.8437,0.5616,0.8520,0.5578,0.8406,0.5727,0.8452,0.5592,0.8391,0.5616,0.8520,0.5531,0.8602,0.5573,0.8561
4,extratree_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6539,0.8805,0.5426,0.8586,0.5376,0.8465,0.6536,0.8828,0.5394,0.8438,0.5426,0.8586,0.4313,0.8367,0.4831,0.8475
5,extratree_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6882,0.9107,0.5542,0.8815,0.5542,0.8811,0.6772,0.9110,0.5617,0.8854,0.5542,0.8815,0.4201,0.8522,0.4820,0.8667
6,extratree_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5728,0.8441,0.5680,0.8533,0.5634,0.8413,0.5765,0.8457,0.5654,0.8394,0.5680,0.8533,0.5633,0.8626,0.5656,0.8579


#### FE_DATA

##### NO SCALER

In [187]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_extratree_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,FE / No Scaler / 3 Class,0.4216,0.8599,0.3889,0.8377,0.3763,0.8504,0.3997,0.8569,0.3982,0.8878,0.3889,0.8377,0.6679,0.9145,0.5094,0.8753
1,extratree_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6291,0.8479,0.5734,0.8125,0.5611,0.8281,0.5969,0.8403,0.5988,0.8832,0.5734,0.8125,0.5177,0.7772,0.5448,0.7947
2,extratree_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7445,0.8695,0.5150,0.7361,0.4760,0.7804,0.6656,0.8511,0.5515,0.9262,0.5150,0.7361,0.2856,0.6027,0.3832,0.6660
3,extratree_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5906,0.8822,0.5714,0.8783,0.5699,0.8772,0.5892,0.8824,0.5708,0.8765,0.5714,0.8783,0.5522,0.8743,0.5617,0.8763
4,extratree_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6470,0.8743,0.5585,0.8811,0.5524,0.8464,0.6556,0.8793,0.5547,0.8297,0.5585,0.8811,0.4700,0.8878,0.5121,0.8844
5,extratree_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6909,0.9385,0.5504,0.9251,0.5497,0.9190,0.6767,0.9391,0.5592,0.9175,0.5504,0.9251,0.4098,0.9117,0.4745,0.9183
6,extratree_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5989,0.9014,0.5818,0.9005,0.5798,0.8976,0.5981,0.9017,0.5800,0.8955,0.5818,0.9005,0.5648,0.8995,0.5732,0.9000


##### StandardScaler

In [188]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_extratree_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,FE / StandardScaler / 3 Class,0.4134,0.8644,0.3819,0.8433,0.3704,0.8558,0.3929,0.8617,0.3931,0.8913,0.3819,0.8433,0.6642,0.9172,0.5034,0.8794
1,extratree_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6428,0.8479,0.5870,0.8127,0.5769,0.8281,0.6116,0.8402,0.6231,0.8829,0.5870,0.8127,0.5313,0.7775,0.5584,0.7949
2,extratree_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7362,0.8678,0.5040,0.7326,0.4599,0.7769,0.6551,0.8489,0.4867,0.9253,0.5040,0.7326,0.2717,0.5975,0.3698,0.6616
3,extratree_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5865,0.8922,0.5699,0.8911,0.5682,0.8880,0.5864,0.8925,0.5681,0.8860,0.5699,0.8911,0.5532,0.8901,0.5614,0.8906
4,extratree_classifier,FE / StandardScaler / 2 Class S_IW,0.6497,0.8925,0.5622,0.8987,0.5540,0.8674,0.6571,0.8964,0.5561,0.8522,0.5622,0.8987,0.4747,0.9050,0.5161,0.9018
5,extratree_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6964,0.9399,0.5485,0.9260,0.5452,0.9207,0.6771,0.9404,0.5567,0.9183,0.5485,0.9260,0.4005,0.9122,0.4679,0.9190
6,extratree_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.6058,0.9042,0.5839,0.9011,0.5828,0.9000,0.6029,0.9043,0.5837,0.8992,0.5839,0.9011,0.5621,0.8980,0.5729,0.8996


##### MinMax

In [189]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(extratree_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="extratree_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_extratree_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_extratree_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,FE / MinMaxScaler / 3 Class,0.4190,0.8733,0.3854,0.8517,0.3713,0.8641,0.3961,0.8705,0.3931,0.8986,0.3854,0.8517,0.6662,0.9225,0.5065,0.8864
1,extratree_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6319,0.8479,0.5762,0.8127,0.5645,0.8282,0.6000,0.8403,0.6040,0.8828,0.5762,0.8127,0.5206,0.7775,0.5476,0.7949
2,extratree_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7445,0.8657,0.5095,0.7285,0.4635,0.7725,0.6597,0.8461,0.5040,0.9243,0.5095,0.7285,0.2744,0.5912,0.3736,0.6562
3,extratree_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5906,0.8918,0.5733,0.8907,0.5712,0.8877,0.5898,0.8922,0.5715,0.8858,0.5733,0.8907,0.5559,0.8896,0.5645,0.8902
4,extratree_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.6360,0.8980,0.5456,0.9047,0.5385,0.8735,0.6447,0.9015,0.5403,0.8565,0.5456,0.9047,0.4552,0.9114,0.4979,0.9080
5,extratree_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.7047,0.9457,0.5614,0.9262,0.5617,0.9273,0.6876,0.9458,0.5792,0.9311,0.5614,0.9262,0.4181,0.9066,0.4838,0.9163
6,extratree_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5907,0.8949,0.5697,0.8933,0.5687,0.8908,0.5888,0.8952,0.5688,0.8888,0.5697,0.8933,0.5487,0.8917,0.5591,0.8925


In [190]:
df_extratree_classifier_final=pd.concat([df_extratree_classifier_results, df_extratree_classifier_StandardScaler, df_extratree_classifier_MinMaxScaler,
                            df_extratree_classifier_results_fe, df_extratree_classifier_StandardScaler_fe, df_extratree_classifier_MinMaxScaler_fe], ignore_index=True)

df_extratree_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,extratree_classifier,Normal / No Scaler / 3 Class,0.4286,0.7977,0.3867,0.7677,0.3621,0.7834,0.3929,0.7922,0.3902,0.8529,0.3867,0.7677,0.6625,0.8729,0.5060,0.8186
1,extratree_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6333,0.8166,0.5702,0.7750,0.5509,0.7889,0.5908,0.8048,0.6060,0.8579,0.5702,0.7750,0.5070,0.7334,0.5376,0.7539
2,extratree_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7404,0.8424,0.4992,0.6813,0.4444,0.7180,0.6493,0.8126,0.4748,0.9135,0.4992,0.6813,0.2581,0.5201,0.3589,0.5952
3,extratree_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5659,0.8489,0.5557,0.8546,0.5532,0.8453,0.5689,0.8502,0.5536,0.8425,0.5557,0.8546,0.5455,0.8603,0.5506,0.8575
4,extratree_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6401,0.8372,0.5633,0.8368,0.5541,0.8019,0.6526,0.8441,0.5546,0.7856,0.5633,0.8368,0.4864,0.8365,0.5232,0.8366
5,extratree_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6951,0.9131,0.5625,0.8798,0.5629,0.8832,0.6840,0.9130,0.5708,0.8924,0.5625,0.8798,0.4298,0.8465,0.4913,0.8629
6,extratree_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5866,0.8492,0.5764,0.8565,0.5735,0.8461,0.5888,0.8507,0.5742,0.8432,0.5764,0.8565,0.5662,0.8638,0.5712,0.8602
7,extratree_classifier,Normal / StandardScaler / 3 Class,0.4272,0.7960,0.3865,0.7659,0.3636,0.7820,0.3927,0.7906,0.4015,0.8528,0.3865,0.7659,0.6624,0.8716,0.5059,0.8171
8,extratree_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6333,0.8197,0.5713,0.7789,0.5538,0.7930,0.5929,0.8085,0.6096,0.8597,0.5713,0.7789,0.5093,0.7381,0.5393,0.7582
9,extratree_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7445,0.8427,0.5094,0.6819,0.4644,0.7192,0.6601,0.8133,0.5332,0.9136,0.5094,0.6819,0.2744,0.5212,0.3738,0.5961


----

### Gradient Boosting

#### Normal Data

##### NO SCALER

In [191]:

# Modelo base
GB_classifier = GradientBoostingClassifier()

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GB_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_results


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,Normal / No Scaler / 3 Class,0.4341,0.7703,0.4063,0.7488,0.3985,0.7596,0.4187,0.7668,0.4171,0.7927,0.4063,0.7488,0.6808,0.8652,0.5257,0.8049
1,GB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5934,0.8012,0.5486,0.7650,0.5432,0.7761,0.5746,0.7917,0.5581,0.8208,0.5486,0.7650,0.5038,0.7289,0.5257,0.7467
2,GB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.8458,0.5485,0.6938,0.5374,0.7320,0.6968,0.8203,0.6450,0.8964,0.5485,0.6938,0.3442,0.5418,0.4344,0.6131
3,GB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5439,0.8087,0.5260,0.7975,0.5258,0.7989,0.5451,0.8081,0.5259,0.8014,0.5260,0.7975,0.5082,0.7863,0.5170,0.7919
4,GB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6621,0.8444,0.5387,0.7857,0.5389,0.7887,0.6592,0.8435,0.5411,0.7929,0.5387,0.7857,0.4152,0.7269,0.4728,0.7557
5,GB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6429,0.8482,0.5464,0.8288,0.5420,0.8080,0.6504,0.8524,0.5413,0.7947,0.5464,0.8288,0.4499,0.8093,0.4955,0.8189
6,GB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5591,0.8005,0.5559,0.8029,0.5509,0.7955,0.5633,0.8021,0.5538,0.7931,0.5559,0.8029,0.5528,0.8053,0.5544,0.8041


##### StandardScaler

In [192]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_GB_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,Normal / StandardScaler / 3 Class,0.4327,0.7723,0.4061,0.7512,0.3990,0.7619,0.4183,0.7690,0.4152,0.7943,0.4061,0.7512,0.6819,0.8666,0.5257,0.8068
1,GB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5948,0.8012,0.5503,0.7650,0.5453,0.7761,0.5764,0.7917,0.5600,0.8208,0.5503,0.7650,0.5059,0.7289,0.5276,0.7467
2,GB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7500,0.8458,0.5429,0.6938,0.5283,0.7320,0.6916,0.8203,0.6370,0.8964,0.5429,0.6938,0.3359,0.5418,0.4268,0.6131
3,GB_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5618,0.8005,0.5439,0.7918,0.5431,0.7915,0.5623,0.8005,0.5430,0.7914,0.5439,0.7918,0.5261,0.7831,0.5349,0.7874
4,GB_classifier,Normal / StandardScaler / 2 Class S_IW,0.7019,0.8561,0.5707,0.7813,0.5744,0.7958,0.6919,0.8518,0.5848,0.8189,0.5707,0.7813,0.4394,0.7065,0.5007,0.7429
5,GB_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6387,0.8482,0.5418,0.8288,0.5371,0.8080,0.6465,0.8524,0.5367,0.7947,0.5418,0.8288,0.4448,0.8093,0.4905,0.8189
6,GB_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5604,0.8005,0.5571,0.8029,0.5522,0.7955,0.5647,0.8021,0.5548,0.7931,0.5571,0.8029,0.5537,0.8053,0.5554,0.8041


##### MinMax

In [193]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GB_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,Normal / MinMaxScaler / 3 Class,0.4368,0.7703,0.4100,0.7488,0.4035,0.7599,0.4227,0.7669,0.4215,0.7940,0.4100,0.7488,0.6827,0.8649,0.5288,0.8047
1,GB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5975,0.8012,0.5526,0.7650,0.5475,0.7761,0.5788,0.7917,0.5632,0.8208,0.5526,0.7650,0.5077,0.7289,0.5296,0.7467
2,GB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7527,0.8472,0.5503,0.6961,0.5409,0.7348,0.6984,0.8221,0.6548,0.8987,0.5503,0.6961,0.3480,0.5450,0.4375,0.6159
3,GB_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5618,0.8025,0.5445,0.7881,0.5434,0.7910,0.5624,0.8012,0.5429,0.7956,0.5445,0.7881,0.5273,0.7736,0.5358,0.7808
4,GB_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.7019,0.8571,0.5670,0.7862,0.5703,0.7986,0.6901,0.8534,0.5836,0.8170,0.5670,0.7862,0.4320,0.7152,0.4947,0.7498
5,GB_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6484,0.8499,0.5538,0.8332,0.5488,0.8108,0.6557,0.8543,0.5476,0.7967,0.5538,0.8332,0.4592,0.8164,0.5039,0.8247
6,GB_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5591,0.8005,0.5559,0.8029,0.5509,0.7955,0.5633,0.8021,0.5538,0.7931,0.5559,0.8029,0.5528,0.8053,0.5544,0.8041


#### FE_DATA

##### NO SCALER

In [194]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GB_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,FE / No Scaler / 3 Class,0.4272,0.8784,0.4042,0.8644,0.4014,0.8739,0.4169,0.8775,0.4204,0.8928,0.4042,0.8644,0.6775,0.9280,0.5228,0.8956
1,GB_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6003,0.8750,0.5561,0.8513,0.5509,0.8633,0.5818,0.8716,0.5667,0.8907,0.5561,0.8513,0.5120,0.8277,0.5336,0.8394
2,GB_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7459,0.8856,0.5495,0.7688,0.5419,0.8142,0.6963,0.8724,0.6210,0.9341,0.5495,0.7688,0.3531,0.6519,0.4404,0.7079
3,GB_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5632,0.8582,0.5421,0.8518,0.5410,0.8517,0.5617,0.8582,0.5410,0.8518,0.5421,0.8518,0.5211,0.8454,0.5315,0.8486
4,GB_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.7075,0.8884,0.5893,0.8391,0.5955,0.8466,0.7027,0.8871,0.6123,0.8557,0.5893,0.8391,0.4712,0.7899,0.5269,0.8141
5,GB_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6800,0.9093,0.5636,0.8987,0.5639,0.8829,0.6774,0.9110,0.5659,0.8707,0.5636,0.8987,0.4472,0.8881,0.5019,0.8934
6,GB_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5714,0.8802,0.5531,0.8773,0.5527,0.8753,0.5718,0.8805,0.5528,0.8737,0.5531,0.8773,0.5347,0.8745,0.5438,0.8759


##### StandardScaler

In [195]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_GB_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,FE / StandardScaler / 3 Class,0.4217,0.8784,0.3997,0.8644,0.3969,0.8739,0.4117,0.8775,0.4161,0.8928,0.3997,0.8644,0.6749,0.9280,0.5189,0.8956
1,GB_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5989,0.8750,0.5550,0.8513,0.5497,0.8633,0.5804,0.8716,0.5658,0.8907,0.5550,0.8513,0.5111,0.8277,0.5326,0.8394
2,GB_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7404,0.8863,0.5440,0.7701,0.5339,0.8156,0.6905,0.8733,0.6031,0.9344,0.5440,0.7701,0.3476,0.6539,0.4346,0.7097
3,GB_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5481,0.8592,0.5290,0.8531,0.5278,0.8528,0.5477,0.8592,0.5293,0.8528,0.5290,0.8531,0.5098,0.8470,0.5193,0.8500
4,GB_classifier,FE / StandardScaler / 2 Class S_IW,0.7006,0.8949,0.5679,0.8411,0.5711,0.8534,0.6900,0.8928,0.5810,0.8694,0.5679,0.8411,0.4353,0.7873,0.4972,0.8138
5,GB_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6813,0.9093,0.5682,0.8987,0.5684,0.8829,0.6799,0.9110,0.5701,0.8707,0.5682,0.8987,0.4551,0.8881,0.5085,0.8934
6,GB_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5700,0.8802,0.5525,0.8773,0.5520,0.8753,0.5707,0.8805,0.5519,0.8737,0.5525,0.8773,0.5350,0.8745,0.5437,0.8759


##### MinMax

In [196]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GB_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GB_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,FE / MinMaxScaler / 3 Class,0.4231,0.8812,0.4006,0.8667,0.3979,0.8764,0.4130,0.8801,0.4170,0.8958,0.4006,0.8667,0.6754,0.9295,0.5197,0.8975
1,GB_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5989,0.8750,0.5550,0.8513,0.5496,0.8633,0.5804,0.8716,0.5655,0.8907,0.5550,0.8513,0.5111,0.8277,0.5326,0.8394
2,GB_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7431,0.8863,0.5458,0.7701,0.5358,0.8156,0.6925,0.8733,0.6074,0.9344,0.5458,0.7701,0.3485,0.6539,0.4359,0.7097
3,GB_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5618,0.8668,0.5446,0.8604,0.5423,0.8606,0.5614,0.8667,0.5424,0.8610,0.5446,0.8604,0.5274,0.8540,0.5359,0.8572
4,GB_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.6992,0.8925,0.5651,0.8344,0.5670,0.8489,0.6876,0.8899,0.5770,0.8679,0.5651,0.8344,0.4311,0.7763,0.4932,0.8048
5,GB_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6814,0.9083,0.5645,0.8981,0.5649,0.8817,0.6784,0.9101,0.5681,0.8691,0.5645,0.8981,0.4477,0.8878,0.5026,0.8929
6,GB_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5728,0.8802,0.5542,0.8773,0.5539,0.8753,0.5731,0.8805,0.5538,0.8737,0.5542,0.8773,0.5356,0.8745,0.5448,0.8759


In [197]:
df_GB_classifier_final=pd.concat([df_GB_classifier_results, df_GB_classifier_StandardScaler, df_GB_classifier_MinMaxScaler,
                            df_GB_classifier_results_fe, df_GB_classifier_StandardScaler_fe, df_GB_classifier_MinMaxScaler_fe], ignore_index=True)

df_GB_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GB_classifier,Normal / No Scaler / 3 Class,0.4341,0.7703,0.4063,0.7488,0.3985,0.7596,0.4187,0.7668,0.4171,0.7927,0.4063,0.7488,0.6808,0.8652,0.5257,0.8049
1,GB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5934,0.8012,0.5486,0.7650,0.5432,0.7761,0.5746,0.7917,0.5581,0.8208,0.5486,0.7650,0.5038,0.7289,0.5257,0.7467
2,GB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.8458,0.5485,0.6938,0.5374,0.7320,0.6968,0.8203,0.6450,0.8964,0.5485,0.6938,0.3442,0.5418,0.4344,0.6131
3,GB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5439,0.8087,0.5260,0.7975,0.5258,0.7989,0.5451,0.8081,0.5259,0.8014,0.5260,0.7975,0.5082,0.7863,0.5170,0.7919
4,GB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6621,0.8444,0.5387,0.7857,0.5389,0.7887,0.6592,0.8435,0.5411,0.7929,0.5387,0.7857,0.4152,0.7269,0.4728,0.7557
5,GB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6429,0.8482,0.5464,0.8288,0.5420,0.8080,0.6504,0.8524,0.5413,0.7947,0.5464,0.8288,0.4499,0.8093,0.4955,0.8189
6,GB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5591,0.8005,0.5559,0.8029,0.5509,0.7955,0.5633,0.8021,0.5538,0.7931,0.5559,0.8029,0.5528,0.8053,0.5544,0.8041
7,GB_classifier,Normal / StandardScaler / 3 Class,0.4327,0.7723,0.4061,0.7512,0.3990,0.7619,0.4183,0.7690,0.4152,0.7943,0.4061,0.7512,0.6819,0.8666,0.5257,0.8068
8,GB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5948,0.8012,0.5503,0.7650,0.5453,0.7761,0.5764,0.7917,0.5600,0.8208,0.5503,0.7650,0.5059,0.7289,0.5276,0.7467
9,GB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7500,0.8458,0.5429,0.6938,0.5283,0.7320,0.6916,0.8203,0.6370,0.8964,0.5429,0.6938,0.3359,0.5418,0.4268,0.6131


----

### XGBoost

#### Normal Data

##### NO SCALER

In [198]:
# Modelo base
XGB_classifier = XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        tree_method="hist",
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
)

y_train_xgb = y_train.replace({0:0, 1:1, -1:2})

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_XGB_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,Normal / No Scaler / 3 Class,0.4149,0.9481,0.3939,0.9436,0.3906,0.9475,0.4080,0.9481,0.3934,0.9526,0.3939,0.9436,0.6827,0.9696,0.5185,0.9565
1,XGB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5701,0.9409,0.5342,0.9328,0.5304,0.9375,0.5585,0.9406,0.5347,0.9436,0.5342,0.9328,0.4982,0.9248,0.5158,0.9288
2,XGB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7116,0.9512,0.5323,0.9037,0.5268,0.9299,0.6761,0.9494,0.5562,0.9663,0.5323,0.9037,0.3531,0.8562,0.4334,0.8796
3,XGB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5481,0.9306,0.5267,0.9254,0.5244,0.9272,0.5459,0.9305,0.5248,0.9293,0.5267,0.9254,0.5052,0.9201,0.5158,0.9227
4,XGB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6745,0.9502,0.5189,0.9189,0.5174,0.9311,0.6565,0.9494,0.5239,0.9455,0.5189,0.9189,0.3633,0.8876,0.4340,0.9031
5,XGB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6456,0.9574,0.5165,0.9456,0.5161,0.9431,0.6424,0.9575,0.5175,0.9408,0.5165,0.9456,0.3874,0.9338,0.4472,0.9397
6,XGB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5564,0.9296,0.5371,0.9272,0.5352,0.9265,0.5554,0.9297,0.5355,0.9260,0.5371,0.9272,0.5178,0.9248,0.5273,0.9260


##### StandardScaler

In [199]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_XGB_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,Normal / StandardScaler / 3 Class,0.4149,0.9481,0.3939,0.9436,0.3906,0.9475,0.4080,0.9481,0.3934,0.9526,0.3939,0.9436,0.6827,0.9696,0.5185,0.9565
1,XGB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5701,0.9409,0.5342,0.9328,0.5304,0.9375,0.5585,0.9406,0.5347,0.9436,0.5342,0.9328,0.4982,0.9248,0.5158,0.9288
2,XGB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7116,0.9512,0.5323,0.9037,0.5268,0.9299,0.6761,0.9494,0.5562,0.9663,0.5323,0.9037,0.3531,0.8562,0.4334,0.8796
3,XGB_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5563,0.9382,0.5305,0.9327,0.5287,0.9350,0.5521,0.9380,0.5300,0.9379,0.5305,0.9327,0.5046,0.9272,0.5173,0.9299
4,XGB_classifier,Normal / StandardScaler / 2 Class S_IW,0.6745,0.9516,0.5301,0.9133,0.5307,0.9319,0.6619,0.9504,0.5367,0.9553,0.5301,0.9133,0.3857,0.8750,0.4521,0.8939
5,XGB_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6456,0.9574,0.5165,0.9456,0.5161,0.9431,0.6424,0.9575,0.5175,0.9408,0.5165,0.9456,0.3874,0.9338,0.4472,0.9397
6,XGB_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5564,0.9296,0.5371,0.9272,0.5352,0.9265,0.5554,0.9297,0.5355,0.9260,0.5371,0.9272,0.5178,0.9248,0.5273,0.9260


##### MinMax

In [200]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_XGB_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,Normal / MinMaxScaler / 3 Class,0.4149,0.9481,0.3939,0.9436,0.3906,0.9475,0.4080,0.9481,0.3934,0.9526,0.3939,0.9436,0.6827,0.9696,0.5185,0.9565
1,XGB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5701,0.9409,0.5342,0.9328,0.5304,0.9375,0.5585,0.9406,0.5347,0.9436,0.5342,0.9328,0.4982,0.9248,0.5158,0.9288
2,XGB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7116,0.9512,0.5323,0.9037,0.5268,0.9299,0.6761,0.9494,0.5562,0.9663,0.5323,0.9037,0.3531,0.8562,0.4334,0.8796
3,XGB_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5495,0.9399,0.5248,0.9353,0.5225,0.9369,0.5456,0.9398,0.5234,0.9389,0.5248,0.9353,0.5002,0.9307,0.5123,0.9330
4,XGB_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.7006,0.9488,0.5512,0.9087,0.5525,0.9279,0.6824,0.9475,0.5681,0.9523,0.5512,0.9087,0.4018,0.8685,0.4703,0.8883
5,XGB_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6456,0.9574,0.5165,0.9456,0.5161,0.9431,0.6424,0.9575,0.5175,0.9408,0.5165,0.9456,0.3874,0.9338,0.4472,0.9397
6,XGB_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5564,0.9296,0.5371,0.9272,0.5352,0.9265,0.5554,0.9297,0.5355,0.9260,0.5371,0.9272,0.5178,0.9248,0.5273,0.9260


#### FE_DATA

##### NO SCALER

In [201]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_XGB_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,FE / No Scaler / 3 Class,0.4025,0.9907,0.3846,0.9903,0.3841,0.9907,0.3978,0.9907,0.3905,0.9910,0.3846,0.9903,0.6749,0.9948,0.5091,0.9926
1,XGB_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5645,0.9883,0.5319,0.9873,0.5291,0.9878,0.5554,0.9883,0.5337,0.9882,0.5319,0.9873,0.4994,0.9864,0.5154,0.9869
2,XGB_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7116,0.9890,0.5417,0.9778,0.5412,0.9850,0.6824,0.9889,0.5698,0.9928,0.5417,0.9778,0.3718,0.9665,0.4486,0.9721
3,XGB_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5576,0.9880,0.5376,0.9880,0.5367,0.9874,0.5569,0.9880,0.5373,0.9870,0.5376,0.9880,0.5175,0.9879,0.5274,0.9879
4,XGB_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6745,0.9852,0.5488,0.9725,0.5504,0.9798,0.6694,0.9851,0.5575,0.9879,0.5488,0.9725,0.4230,0.9597,0.4818,0.9661
5,XGB_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6855,0.9904,0.5523,0.9871,0.5532,0.9871,0.6760,0.9904,0.5582,0.9871,0.5523,0.9871,0.4192,0.9838,0.4810,0.9854
6,XGB_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5769,0.9839,0.5548,0.9836,0.5532,0.9831,0.5742,0.9839,0.5543,0.9827,0.5548,0.9836,0.5327,0.9834,0.5436,0.9835


##### StandardScaler

In [202]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_XGB_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,FE / StandardScaler / 3 Class,0.4025,0.9907,0.3846,0.9903,0.3841,0.9907,0.3978,0.9907,0.3905,0.9910,0.3846,0.9903,0.6749,0.9948,0.5091,0.9926
1,XGB_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5645,0.9883,0.5319,0.9873,0.5291,0.9878,0.5554,0.9883,0.5337,0.9882,0.5319,0.9873,0.4994,0.9864,0.5154,0.9869
2,XGB_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7116,0.9890,0.5417,0.9778,0.5412,0.9850,0.6824,0.9889,0.5698,0.9928,0.5417,0.9778,0.3718,0.9665,0.4486,0.9721
3,XGB_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5604,0.9870,0.5363,0.9870,0.5343,0.9864,0.5567,0.9870,0.5350,0.9859,0.5363,0.9870,0.5122,0.9870,0.5241,0.9870
4,XGB_classifier,FE / StandardScaler / 2 Class S_IW,0.6649,0.9880,0.5330,0.9771,0.5326,0.9836,0.6580,0.9879,0.5356,0.9907,0.5330,0.9771,0.4012,0.9662,0.4622,0.9716
5,XGB_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6855,0.9904,0.5523,0.9871,0.5532,0.9871,0.6760,0.9904,0.5582,0.9871,0.5523,0.9871,0.4192,0.9838,0.4810,0.9854
6,XGB_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5769,0.9839,0.5548,0.9836,0.5532,0.9831,0.5742,0.9839,0.5543,0.9827,0.5548,0.9836,0.5327,0.9834,0.5436,0.9835


##### MinMax

In [203]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(XGB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="XGB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_XGB_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_XGB_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,FE / MinMaxScaler / 3 Class,0.4025,0.9907,0.3846,0.9903,0.3841,0.9907,0.3978,0.9907,0.3905,0.9910,0.3846,0.9903,0.6749,0.9948,0.5091,0.9926
1,XGB_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5645,0.9883,0.5319,0.9873,0.5291,0.9878,0.5554,0.9883,0.5337,0.9882,0.5319,0.9873,0.4994,0.9864,0.5154,0.9869
2,XGB_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7116,0.9890,0.5417,0.9778,0.5412,0.9850,0.6824,0.9889,0.5698,0.9928,0.5417,0.9778,0.3718,0.9665,0.4486,0.9721
3,XGB_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5604,0.9870,0.5345,0.9867,0.5329,0.9864,0.5562,0.9870,0.5343,0.9861,0.5345,0.9867,0.5086,0.9864,0.5214,0.9865
4,XGB_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.6704,0.9876,0.5329,0.9764,0.5323,0.9831,0.6602,0.9875,0.5388,0.9905,0.5329,0.9764,0.3955,0.9652,0.4588,0.9708
5,XGB_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6855,0.9904,0.5523,0.9871,0.5532,0.9871,0.6760,0.9904,0.5582,0.9871,0.5523,0.9871,0.4192,0.9838,0.4810,0.9854
6,XGB_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5769,0.9839,0.5548,0.9836,0.5532,0.9831,0.5742,0.9839,0.5543,0.9827,0.5548,0.9836,0.5327,0.9834,0.5436,0.9835


In [204]:
df_XGB_classifier_final=pd.concat([df_XGB_classifier_results, df_XGB_classifier_StandardScaler, df_XGB_classifier_MinMaxScaler,
                            df_XGB_classifier_results_fe, df_XGB_classifier_StandardScaler_fe, df_XGB_classifier_MinMaxScaler_fe], ignore_index=True)

df_XGB_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,XGB_classifier,Normal / No Scaler / 3 Class,0.4149,0.9481,0.3939,0.9436,0.3906,0.9475,0.4080,0.9481,0.3934,0.9526,0.3939,0.9436,0.6827,0.9696,0.5185,0.9565
1,XGB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5701,0.9409,0.5342,0.9328,0.5304,0.9375,0.5585,0.9406,0.5347,0.9436,0.5342,0.9328,0.4982,0.9248,0.5158,0.9288
2,XGB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7116,0.9512,0.5323,0.9037,0.5268,0.9299,0.6761,0.9494,0.5562,0.9663,0.5323,0.9037,0.3531,0.8562,0.4334,0.8796
3,XGB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5481,0.9306,0.5267,0.9254,0.5244,0.9272,0.5459,0.9305,0.5248,0.9293,0.5267,0.9254,0.5052,0.9201,0.5158,0.9227
4,XGB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6745,0.9502,0.5189,0.9189,0.5174,0.9311,0.6565,0.9494,0.5239,0.9455,0.5189,0.9189,0.3633,0.8876,0.4340,0.9031
5,XGB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.6456,0.9574,0.5165,0.9456,0.5161,0.9431,0.6424,0.9575,0.5175,0.9408,0.5165,0.9456,0.3874,0.9338,0.4472,0.9397
6,XGB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5564,0.9296,0.5371,0.9272,0.5352,0.9265,0.5554,0.9297,0.5355,0.9260,0.5371,0.9272,0.5178,0.9248,0.5273,0.9260
7,XGB_classifier,Normal / StandardScaler / 3 Class,0.4149,0.9481,0.3939,0.9436,0.3906,0.9475,0.4080,0.9481,0.3934,0.9526,0.3939,0.9436,0.6827,0.9696,0.5185,0.9565
8,XGB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5701,0.9409,0.5342,0.9328,0.5304,0.9375,0.5585,0.9406,0.5347,0.9436,0.5342,0.9328,0.4982,0.9248,0.5158,0.9288
9,XGB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7116,0.9512,0.5323,0.9037,0.5268,0.9299,0.6761,0.9494,0.5562,0.9663,0.5323,0.9037,0.3531,0.8562,0.4334,0.8796


----

## Distance Model 

### KNN

#### Normal Data

##### NO SCALER

In [205]:
# Modelo base
KNN_classifier = KNeighborsClassifier(n_neighbors=31)

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_KNN_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,Normal / No Scaler / 3 Class,0.4437,0.4993,0.3976,0.4548,0.3638,0.4325,0.4004,0.4644,0.3887,0.4942,0.3976,0.4548,0.6718,0.7032,0.5168,0.5655
1,KNN_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6209,0.6425,0.5676,0.5873,0.5588,0.5792,0.5933,0.6133,0.5919,0.6222,0.5676,0.5873,0.5143,0.5321,0.5402,0.5590
2,KNN_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7514,0.7545,0.5010,0.5049,0.4342,0.4406,0.6483,0.6524,0.4266,0.6190,0.5010,0.5049,0.2505,0.2553,0.3543,0.3590
3,KNN_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5564,0.6095,0.5622,0.6190,0.5526,0.6066,0.5609,0.6135,0.5597,0.6142,0.5622,0.6190,0.5680,0.6284,0.5651,0.6236
4,KNN_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5289,0.5824,0.5248,0.6102,0.4895,0.5536,0.5597,0.6107,0.5188,0.5822,0.5248,0.6102,0.5207,0.6381,0.5225,0.6240
5,KNN_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5206,0.6157,0.4894,0.6048,0.4691,0.5693,0.5517,0.6406,0.4921,0.5800,0.4894,0.6048,0.4583,0.5940,0.4734,0.5993
6,KNN_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5646,0.6040,0.5720,0.6102,0.5614,0.5997,0.5690,0.6079,0.5689,0.6060,0.5720,0.6102,0.5794,0.6164,0.5757,0.6133


##### StandardScaler

In [206]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_KNN_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,Normal / StandardScaler / 3 Class,0.4162,0.4938,0.3699,0.4424,0.3295,0.4040,0.3666,0.4440,0.3671,0.4658,0.3699,0.4424,0.6484,0.6959,0.4897,0.5548
1,KNN_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6347,0.6532,0.5737,0.5936,0.5585,0.5830,0.5966,0.6187,0.6085,0.6423,0.5737,0.5936,0.5128,0.5340,0.5424,0.5630
2,KNN_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,KNN_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5220,0.6068,0.5320,0.6192,0.5189,0.6047,0.5256,0.6104,0.5314,0.6148,0.5320,0.6192,0.5419,0.6317,0.5369,0.6254
4,KNN_classifier,Normal / StandardScaler / 2 Class S_IW,0.5275,0.5907,0.5351,0.6250,0.4938,0.5638,0.5588,0.6184,0.5262,0.5932,0.5351,0.6250,0.5427,0.6594,0.5387,0.6420
5,KNN_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5439,0.6350,0.4994,0.6139,0.4839,0.5831,0.5720,0.6572,0.5002,0.5886,0.4994,0.6139,0.4548,0.5928,0.4764,0.6032
6,KNN_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5440,0.6140,0.5508,0.6228,0.5400,0.6109,0.5480,0.6180,0.5493,0.6177,0.5508,0.6228,0.5576,0.6316,0.5542,0.6272


##### MinMax

In [207]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_KNN_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,Normal / MinMaxScaler / 3 Class,0.4244,0.4856,0.3785,0.4348,0.3424,0.4019,0.3772,0.4394,0.4058,0.4635,0.3785,0.4348,0.6530,0.6897,0.4970,0.5476
1,KNN_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6251,0.6501,0.5682,0.5901,0.5559,0.5788,0.5923,0.6150,0.5933,0.6379,0.5682,0.5901,0.5112,0.5302,0.5389,0.5593
2,KNN_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,KNN_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5207,0.6044,0.5345,0.6174,0.5176,0.6023,0.5224,0.6079,0.5346,0.6133,0.5345,0.6174,0.5482,0.6304,0.5413,0.6239
4,KNN_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5330,0.5951,0.5406,0.6252,0.4994,0.5664,0.5644,0.6226,0.5303,0.5934,0.5406,0.6252,0.5482,0.6553,0.5443,0.6400
5,KNN_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5577,0.6350,0.5103,0.6148,0.4952,0.5834,0.5848,0.6572,0.5081,0.5893,0.5103,0.6148,0.4630,0.5947,0.4860,0.6046
6,KNN_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5303,0.6085,0.5401,0.6171,0.5275,0.6054,0.5341,0.6126,0.5392,0.6122,0.5401,0.6171,0.5498,0.6256,0.5449,0.6213


#### FE_DATA

##### NO SCALER

In [208]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_KNN_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,FE / No Scaler / 3 Class,0.4409,0.5134,0.3977,0.4683,0.3708,0.4507,0.4031,0.4813,0.4077,0.5186,0.3977,0.4683,0.6676,0.7097,0.5151,0.5765
1,KNN_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6072,0.6408,0.5509,0.5850,0.5383,0.5763,0.5755,0.6108,0.5708,0.6201,0.5509,0.5850,0.4947,0.5292,0.5220,0.5564
2,KNN_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7500,0.7545,0.5000,0.5072,0.4339,0.4466,0.6477,0.6553,0.4764,0.5611,0.5000,0.5072,0.2501,0.2599,0.3536,0.3630
3,KNN_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5646,0.5999,0.5697,0.6092,0.5606,0.5968,0.5691,0.6038,0.5667,0.6051,0.5697,0.6092,0.5747,0.6185,0.5722,0.6138
4,KNN_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5261,0.5896,0.5210,0.6206,0.4878,0.5617,0.5580,0.6174,0.5158,0.5900,0.5210,0.6206,0.5160,0.6516,0.5185,0.6359
5,KNN_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5646,0.6202,0.5373,0.6092,0.5121,0.5738,0.5913,0.6447,0.5294,0.5835,0.5373,0.6092,0.5101,0.5982,0.5233,0.6037
6,KNN_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5647,0.6003,0.5738,0.6087,0.5620,0.5967,0.5687,0.6039,0.5708,0.6048,0.5738,0.6087,0.5830,0.6172,0.5784,0.6130


##### StandardScaler

In [209]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_KNN_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,FE / StandardScaler / 3 Class,0.4162,0.4966,0.3700,0.4478,0.3349,0.4216,0.3711,0.4559,0.3602,0.4942,0.3700,0.4478,0.6519,0.6959,0.4909,0.5582
1,KNN_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6168,0.6463,0.5643,0.5957,0.5545,0.5911,0.5891,0.6225,0.5818,0.6263,0.5643,0.5957,0.5119,0.5451,0.5374,0.5698
2,KNN_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7541,0.7527,0.5028,0.5000,0.4352,0.4295,0.6497,0.6466,0.4769,0.3764,0.5028,0.5000,0.2514,0.2473,0.3555,0.3516
3,KNN_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5289,0.6034,0.5389,0.6168,0.5258,0.6017,0.5323,0.6070,0.5377,0.6125,0.5389,0.6168,0.5490,0.6303,0.5439,0.6235
4,KNN_classifier,FE / StandardScaler / 2 Class S_IW,0.5178,0.5992,0.5305,0.6368,0.4868,0.5730,0.5498,0.6262,0.5229,0.6021,0.5305,0.6368,0.5432,0.6744,0.5367,0.6553
5,KNN_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5742,0.6308,0.5437,0.6158,0.5190,0.5819,0.6003,0.6539,0.5337,0.5893,0.5437,0.6158,0.5132,0.6008,0.5280,0.6082
6,KNN_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5453,0.6133,0.5524,0.6236,0.5420,0.6106,0.5499,0.6172,0.5504,0.6187,0.5524,0.6236,0.5595,0.6339,0.5559,0.6287


##### MinMax

In [210]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(KNN_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="KNN_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_KNN_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_KNN_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,FE / MinMaxScaler / 3 Class,0.4190,0.5010,0.3748,0.4504,0.3444,0.4200,0.3783,0.4570,0.3801,0.4848,0.3748,0.4504,0.6542,0.6997,0.4951,0.5613
1,KNN_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6264,0.6504,0.5764,0.6002,0.5696,0.5961,0.6021,0.6270,0.5958,0.6323,0.5764,0.6002,0.5264,0.5499,0.5508,0.5745
2,KNN_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7527,0.7524,0.5000,0.4998,0.4295,0.4294,0.6466,0.6464,0.3764,0.3763,0.5000,0.4998,0.2473,0.2471,0.3516,0.3514
3,KNN_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5344,0.6089,0.5428,0.6231,0.5299,0.6073,0.5374,0.6124,0.5422,0.6186,0.5428,0.6231,0.5511,0.6372,0.5469,0.6301
4,KNN_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5412,0.5965,0.5498,0.6261,0.5075,0.5677,0.5721,0.6240,0.5373,0.5940,0.5498,0.6261,0.5583,0.6557,0.5540,0.6407
5,KNN_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5700,0.6432,0.5446,0.6282,0.5183,0.5940,0.5971,0.6650,0.5344,0.5997,0.5446,0.6282,0.5193,0.6133,0.5317,0.6206
6,KNN_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5536,0.6185,0.5585,0.6268,0.5486,0.6152,0.5576,0.6225,0.5567,0.6215,0.5585,0.6268,0.5635,0.6352,0.5610,0.6310


In [211]:
df_KNN_classifier_final = pd.concat([df_KNN_classifier_results, df_KNN_classifier_StandardScaler, df_KNN_classifier_MinMaxScaler,
                            df_KNN_classifier_results_fe, df_KNN_classifier_StandardScaler_fe, df_KNN_classifier_MinMaxScaler_fe], ignore_index=True)

df_KNN_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,KNN_classifier,Normal / No Scaler / 3 Class,0.4437,0.4993,0.3976,0.4548,0.3638,0.4325,0.4004,0.4644,0.3887,0.4942,0.3976,0.4548,0.6718,0.7032,0.5168,0.5655
1,KNN_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6209,0.6425,0.5676,0.5873,0.5588,0.5792,0.5933,0.6133,0.5919,0.6222,0.5676,0.5873,0.5143,0.5321,0.5402,0.5590
2,KNN_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7514,0.7545,0.5010,0.5049,0.4342,0.4406,0.6483,0.6524,0.4266,0.6190,0.5010,0.5049,0.2505,0.2553,0.3543,0.3590
3,KNN_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5564,0.6095,0.5622,0.6190,0.5526,0.6066,0.5609,0.6135,0.5597,0.6142,0.5622,0.6190,0.5680,0.6284,0.5651,0.6236
4,KNN_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5289,0.5824,0.5248,0.6102,0.4895,0.5536,0.5597,0.6107,0.5188,0.5822,0.5248,0.6102,0.5207,0.6381,0.5225,0.6240
5,KNN_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5206,0.6157,0.4894,0.6048,0.4691,0.5693,0.5517,0.6406,0.4921,0.5800,0.4894,0.6048,0.4583,0.5940,0.4734,0.5993
6,KNN_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5646,0.6040,0.5720,0.6102,0.5614,0.5997,0.5690,0.6079,0.5689,0.6060,0.5720,0.6102,0.5794,0.6164,0.5757,0.6133
7,KNN_classifier,Normal / StandardScaler / 3 Class,0.4162,0.4938,0.3699,0.4424,0.3295,0.4040,0.3666,0.4440,0.3671,0.4658,0.3699,0.4424,0.6484,0.6959,0.4897,0.5548
8,KNN_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6347,0.6532,0.5737,0.5936,0.5585,0.5830,0.5966,0.6187,0.6085,0.6423,0.5737,0.5936,0.5128,0.5340,0.5424,0.5630
9,KNN_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516


----

## Margin-Based Models  

### SVM (RBF kernel)


#### Normal Data

##### NO SCALER

In [212]:
# Modelo base
SVM_RBF_classifier = SVC(kernel='rbf')

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_RBF_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,Normal / No Scaler / 3 Class,0.4725,0.4794,0.4154,0.4216,0.3567,0.3629,0.4036,0.4104,0.3196,0.3235,0.4154,0.4216,0.6737,0.6779,0.5289,0.5346
1,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5962,0.6150,0.4979,0.5194,0.3913,0.4185,0.4648,0.4886,0.3380,0.3721,0.4979,0.5194,0.3996,0.4237,0.4460,0.4689
2,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5866,0.5907,0.5942,0.5986,0.5823,0.5874,0.5898,0.5949,0.5913,0.5945,0.5942,0.5986,0.6018,0.6065,0.5980,0.6025
4,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5782,0.5882,0.5557,0.5843,0.5275,0.5460,0.6037,0.6155,0.5436,0.5636,0.5557,0.5843,0.5332,0.5803,0.5441,0.5822
5,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5810,0.5889,0.5688,0.5866,0.5350,0.5477,0.6075,0.6162,0.5527,0.5655,0.5688,0.5866,0.5565,0.5842,0.5624,0.5853
6,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5810,0.5862,0.5903,0.5954,0.5778,0.5834,0.5848,0.5903,0.5872,0.5916,0.5903,0.5954,0.5995,0.6047,0.5949,0.6000


##### StandardScaler

In [213]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_RBF_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,Normal / StandardScaler / 3 Class,0.4395,0.5546,0.3900,0.5038,0.3477,0.4827,0.3864,0.5150,0.3874,0.6517,0.3900,0.5038,0.6606,0.7249,0.5074,0.6043
1,SVM_RBF_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6388,0.6597,0.5741,0.5978,0.5567,0.5855,0.5965,0.6221,0.6220,0.6573,0.5741,0.5978,0.5094,0.5358,0.5408,0.5659
2,SVM_RBF_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7514,0.7648,0.4991,0.5243,0.4290,0.4786,0.6459,0.6739,0.3762,0.8810,0.4991,0.5243,0.2468,0.2838,0.3510,0.3857
3,SVM_RBF_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5564,0.6504,0.5569,0.6603,0.5489,0.6470,0.5601,0.6535,0.5546,0.6545,0.5569,0.6603,0.5574,0.6701,0.5571,0.6652
4,SVM_RBF_classifier,Normal / StandardScaler / 2 Class S_IW,0.5880,0.6823,0.5491,0.6799,0.5271,0.6383,0.6113,0.7016,0.5387,0.6406,0.5491,0.6799,0.5103,0.6774,0.5291,0.6785
5,SVM_RBF_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6057,0.7057,0.5423,0.6767,0.5272,0.6501,0.6236,0.7203,0.5322,0.6457,0.5423,0.6767,0.4788,0.6478,0.5089,0.6620
6,SVM_RBF_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5618,0.6435,0.5709,0.6560,0.5575,0.6412,0.5649,0.6470,0.5692,0.6499,0.5709,0.6560,0.5799,0.6684,0.5753,0.6621


##### MinMax

In [214]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_RBF_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,Normal / MinMaxScaler / 3 Class,0.4382,0.5175,0.3855,0.4627,0.3359,0.4290,0.3754,0.4658,0.3648,0.6074,0.3855,0.4627,0.6557,0.6989,0.5027,0.5687
1,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6292,0.6425,0.5578,0.5720,0.5294,0.5453,0.5745,0.5890,0.6217,0.6451,0.5578,0.5720,0.4865,0.5015,0.5209,0.5356
2,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7527,0.7552,0.5000,0.5049,0.4295,0.4397,0.6466,0.6522,0.3764,0.7773,0.5000,0.5049,0.2473,0.2546,0.3516,0.3585
3,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5330,0.6071,0.5454,0.6261,0.5294,0.6059,0.5352,0.6090,0.5447,0.6235,0.5454,0.6261,0.5577,0.6451,0.5515,0.6355
4,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5687,0.6566,0.5438,0.6446,0.5177,0.6086,0.5967,0.6776,0.5332,0.6123,0.5438,0.6446,0.5188,0.6326,0.5310,0.6385
5,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6085,0.6734,0.5516,0.6474,0.5377,0.6185,0.6285,0.6914,0.5434,0.6179,0.5516,0.6474,0.4946,0.6213,0.5223,0.6341
6,SVM_RBF_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5372,0.6044,0.5523,0.6281,0.5356,0.6040,0.5399,0.6054,0.5515,0.6267,0.5523,0.6281,0.5674,0.6518,0.5598,0.6398


#### FE_DATA

##### NO SCALER

In [215]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_RBF_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,FE / No Scaler / 3 Class,0.4725,0.4794,0.4156,0.4215,0.3578,0.3631,0.4046,0.4105,0.3202,0.3248,0.4156,0.4215,0.6736,0.6771,0.5290,0.5343
1,SVM_RBF_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5976,0.6140,0.4990,0.5182,0.3922,0.4171,0.4657,0.4874,0.3400,0.3705,0.4990,0.5182,0.4005,0.4224,0.4470,0.4677
2,SVM_RBF_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_RBF_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5838,0.5917,0.5932,0.6026,0.5803,0.5893,0.5872,0.5956,0.5903,0.5986,0.5932,0.6026,0.6025,0.6134,0.5978,0.6080
4,SVM_RBF_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5796,0.5958,0.5510,0.5902,0.5250,0.5527,0.6046,0.6226,0.5398,0.5683,0.5510,0.5902,0.5225,0.5846,0.5363,0.5874
5,SVM_RBF_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5823,0.6003,0.5584,0.5899,0.5307,0.5545,0.6078,0.6261,0.5458,0.5686,0.5584,0.5899,0.5346,0.5796,0.5462,0.5846
6,SVM_RBF_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5879,0.5917,0.5996,0.6027,0.5854,0.5892,0.5914,0.5955,0.5963,0.5989,0.5996,0.6027,0.6112,0.6137,0.6053,0.6082


##### StandardScaler

In [216]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_RBF_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,FE / StandardScaler / 3 Class,0.4381,0.5704,0.3903,0.5235,0.3513,0.5096,0.3900,0.5378,0.4200,0.6610,0.3903,0.5235,0.6655,0.7370,0.5096,0.6211
1,SVM_RBF_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6388,0.6775,0.5765,0.6178,0.5611,0.6096,0.5996,0.6435,0.6189,0.6857,0.5765,0.6178,0.5142,0.5580,0.5445,0.5871
2,SVM_RBF_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7527,0.7651,0.5000,0.5250,0.4295,0.4799,0.6466,0.6746,0.3764,0.8811,0.5000,0.5250,0.2473,0.2849,0.3516,0.3867
3,SVM_RBF_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5715,0.6731,0.5657,0.6765,0.5599,0.6675,0.5737,0.6761,0.5628,0.6701,0.5657,0.6765,0.5599,0.6798,0.5628,0.6781
4,SVM_RBF_classifier,FE / StandardScaler / 2 Class S_IW,0.5976,0.6964,0.5574,0.7046,0.5352,0.6568,0.6198,0.7152,0.5459,0.6587,0.5574,0.7046,0.5173,0.7128,0.5366,0.7086
5,SVM_RBF_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6250,0.7239,0.5532,0.6939,0.5423,0.6682,0.6397,0.7364,0.5451,0.6630,0.5532,0.6939,0.4814,0.6640,0.5157,0.6787
6,SVM_RBF_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5701,0.6765,0.5670,0.6808,0.5596,0.6714,0.5723,0.6797,0.5640,0.6737,0.5670,0.6808,0.5639,0.6851,0.5654,0.6830


##### MinMax

In [217]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_RBF_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_RBF_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_RBF_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_RBF_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,FE / MinMaxScaler / 3 Class,0.4574,0.5247,0.4070,0.4749,0.3648,0.4477,0.4044,0.4826,0.3864,0.5746,0.4070,0.4749,0.6737,0.7113,0.5236,0.5812
1,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6319,0.6518,0.5731,0.5939,0.5608,0.5846,0.5977,0.6194,0.6060,0.6388,0.5731,0.5939,0.5144,0.5361,0.5429,0.5643
2,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7527,0.7531,0.5000,0.5007,0.4295,0.4309,0.6466,0.6474,0.3764,0.4765,0.5000,0.5007,0.2473,0.2483,0.3516,0.3526
3,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5742,0.6422,0.5669,0.6415,0.5616,0.6351,0.5762,0.6455,0.5636,0.6364,0.5669,0.6415,0.5595,0.6408,0.5631,0.6411
4,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5632,0.6428,0.5383,0.6471,0.5124,0.6025,0.5915,0.6664,0.5292,0.6117,0.5383,0.6471,0.5134,0.6514,0.5256,0.6492
5,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5824,0.6665,0.5417,0.6517,0.5207,0.6171,0.6061,0.6864,0.5330,0.6190,0.5417,0.6517,0.5010,0.6368,0.5205,0.6441
6,SVM_RBF_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5729,0.6319,0.5764,0.6407,0.5650,0.6279,0.5750,0.6348,0.5740,0.6362,0.5764,0.6407,0.5800,0.6496,0.5781,0.6452


In [218]:
df_SVM_RBF_classifier_final = pd.concat([df_SVM_RBF_classifier_results, df_SVM_RBF_classifier_StandardScaler, df_SVM_RBF_classifier_MinMaxScaler,
                            df_SVM_RBF_classifier_results_fe, df_SVM_RBF_classifier_StandardScaler_fe, df_SVM_RBF_classifier_MinMaxScaler_fe], ignore_index=True)

df_SVM_RBF_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_RBF_classifier,Normal / No Scaler / 3 Class,0.4725,0.4794,0.4154,0.4216,0.3567,0.3629,0.4036,0.4104,0.3196,0.3235,0.4154,0.4216,0.6737,0.6779,0.5289,0.5346
1,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5962,0.6150,0.4979,0.5194,0.3913,0.4185,0.4648,0.4886,0.3380,0.3721,0.4979,0.5194,0.3996,0.4237,0.4460,0.4689
2,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5866,0.5907,0.5942,0.5986,0.5823,0.5874,0.5898,0.5949,0.5913,0.5945,0.5942,0.5986,0.6018,0.6065,0.5980,0.6025
4,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5782,0.5882,0.5557,0.5843,0.5275,0.5460,0.6037,0.6155,0.5436,0.5636,0.5557,0.5843,0.5332,0.5803,0.5441,0.5822
5,SVM_RBF_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5810,0.5889,0.5688,0.5866,0.5350,0.5477,0.6075,0.6162,0.5527,0.5655,0.5688,0.5866,0.5565,0.5842,0.5624,0.5853
6,SVM_RBF_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5810,0.5862,0.5903,0.5954,0.5778,0.5834,0.5848,0.5903,0.5872,0.5916,0.5903,0.5954,0.5995,0.6047,0.5949,0.6000
7,SVM_RBF_classifier,Normal / StandardScaler / 3 Class,0.4395,0.5546,0.3900,0.5038,0.3477,0.4827,0.3864,0.5150,0.3874,0.6517,0.3900,0.5038,0.6606,0.7249,0.5074,0.6043
8,SVM_RBF_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6388,0.6597,0.5741,0.5978,0.5567,0.5855,0.5965,0.6221,0.6220,0.6573,0.5741,0.5978,0.5094,0.5358,0.5408,0.5659
9,SVM_RBF_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7514,0.7648,0.4991,0.5243,0.4290,0.4786,0.6459,0.6739,0.3762,0.8810,0.4991,0.5243,0.2468,0.2838,0.3510,0.3857


---

### SVM (Polynomial kernel) 

#### Normal Data

##### NO SCALER

In [219]:
# Modelo base
SVM_Poly_classifier = SVC(kernel='poly')

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Poly_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,Normal / No Scaler / 3 Class,0.4711,0.4763,0.4135,0.4177,0.3554,0.3588,0.4020,0.4061,0.3231,0.3249,0.4135,0.4177,0.6708,0.6736,0.5265,0.5304
1,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5976,0.6154,0.4978,0.5184,0.3885,0.4167,0.4630,0.4874,0.3369,0.4744,0.4978,0.5184,0.3981,0.4215,0.4452,0.4673
2,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5907,0.5903,0.6000,0.5999,0.5864,0.5870,0.5933,0.5936,0.5977,0.5964,0.6000,0.5999,0.6093,0.6096,0.6046,0.6047
4,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5631,0.5687,0.5605,0.5820,0.5222,0.5343,0.5895,0.5972,0.5468,0.5616,0.5605,0.5820,0.5580,0.5953,0.5590,0.5885
5,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5700,0.5573,0.5894,0.5814,0.5388,0.5284,0.5983,0.5868,0.5669,0.5610,0.5894,0.5814,0.6088,0.6056,0.5990,0.5933
6,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.6003,0.5975,0.6074,0.6035,0.5960,0.5933,0.6038,0.6016,0.6038,0.5993,0.6074,0.6035,0.6145,0.6094,0.6109,0.6064


##### StandardScaler

In [220]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_Poly_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,Normal / StandardScaler / 3 Class,0.4395,0.5457,0.3851,0.4927,0.3303,0.4696,0.3661,0.4959,0.4093,0.7170,0.3851,0.4927,0.6534,0.7079,0.5015,0.5906
1,SVM_Poly_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6044,0.6659,0.5186,0.5824,0.4557,0.5406,0.5150,0.5907,0.5371,0.7605,0.5186,0.5824,0.4328,0.4988,0.4737,0.5390
2,SVM_Poly_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7307,0.7864,0.5059,0.5695,0.4713,0.5602,0.6585,0.7196,0.5266,0.8764,0.5059,0.5695,0.2811,0.3525,0.3769,0.4480
3,SVM_Poly_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.4959,0.5762,0.5560,0.6382,0.4805,0.5645,0.4618,0.5496,0.5768,0.6856,0.5560,0.6382,0.6161,0.7001,0.5852,0.6684
4,SVM_Poly_classifier,Normal / StandardScaler / 2 Class S_IW,0.5648,0.6891,0.5320,0.6797,0.4864,0.6346,0.5661,0.6938,0.5312,0.6772,0.5320,0.6797,0.4993,0.6702,0.5117,0.6729
5,SVM_Poly_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6045,0.7245,0.5192,0.6748,0.4884,0.6514,0.5960,0.7231,0.5201,0.6942,0.5192,0.6748,0.4339,0.6250,0.4706,0.6474
6,SVM_Poly_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.4986,0.5742,0.5588,0.6359,0.4833,0.5624,0.4648,0.5475,0.5810,0.6823,0.5588,0.6359,0.6190,0.6976,0.5882,0.6660


##### MinMax

In [226]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Poly_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,Normal / MinMaxScaler / 3 Class,0.4299,0.5343,0.3826,0.4846,0.3444,0.4668,0.3776,0.4954,0.3987,0.6290,0.3826,0.4846,0.6539,0.7087,0.5000,0.5860
1,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5879,0.6563,0.5147,0.5859,0.4746,0.5578,0.5250,0.6010,0.5316,0.6789,0.5147,0.5859,0.4415,0.5156,0.4767,0.5495
2,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7445,0.7764,0.5113,0.5488,0.4691,0.5247,0.6623,0.6996,0.5452,0.8739,0.5113,0.5488,0.2781,0.3213,0.3770,0.4199
3,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5426,0.6096,0.5643,0.6374,0.5395,0.6086,0.5414,0.6084,0.5672,0.6387,0.5643,0.6374,0.5859,0.6652,0.5749,0.6511
4,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5632,0.6463,0.5513,0.6578,0.5189,0.6086,0.5924,0.6699,0.5385,0.6192,0.5513,0.6578,0.5394,0.6693,0.5453,0.6635
5,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5810,0.6803,0.5408,0.6701,0.5214,0.6329,0.6056,0.6994,0.5339,0.6330,0.5408,0.6701,0.5005,0.6599,0.5201,0.6650
6,SVM_Poly_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5632,0.6391,0.5745,0.6547,0.5590,0.6370,0.5656,0.6415,0.5733,0.6500,0.5745,0.6547,0.5857,0.6702,0.5800,0.6624


#### FE_DATA

##### NO SCALER

In [222]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Poly_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,FE / No Scaler / 3 Class,0.4670,0.4773,0.4098,0.4192,0.3525,0.3621,0.3988,0.4089,0.3217,0.3538,0.4098,0.4192,0.6679,0.6743,0.5230,0.5316
1,SVM_Poly_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5948,0.6150,0.4950,0.5181,0.3849,0.4164,0.4597,0.4871,0.3303,0.4736,0.4950,0.5181,0.3952,0.4212,0.4422,0.4671
2,SVM_Poly_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_Poly_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5866,0.5941,0.5966,0.6044,0.5832,0.5915,0.5898,0.5980,0.5938,0.6003,0.5966,0.6044,0.6067,0.6147,0.6016,0.6095
4,SVM_Poly_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5700,0.5821,0.5595,0.5927,0.5255,0.5465,0.5969,0.6103,0.5459,0.5695,0.5595,0.5927,0.5491,0.6034,0.5541,0.5980
5,SVM_Poly_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5768,0.5800,0.5791,0.5909,0.5377,0.5446,0.6040,0.6079,0.5600,0.5684,0.5791,0.5909,0.5813,0.6018,0.5800,0.5962
6,SVM_Poly_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5948,0.5934,0.6053,0.6040,0.5917,0.5907,0.5982,0.5972,0.6019,0.6002,0.6053,0.6040,0.6158,0.6145,0.6105,0.6092


##### StandardScaler

In [223]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_Poly_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,FE / StandardScaler / 3 Class,0.4258,0.5549,0.3719,0.5036,0.3129,0.4858,0.3480,0.5097,0.3916,0.7352,0.3719,0.5036,0.6426,0.7133,0.4887,0.5994
1,SVM_Poly_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6195,0.6861,0.5419,0.6126,0.4989,0.5916,0.5501,0.6326,0.5750,0.7478,0.5419,0.6126,0.4643,0.5391,0.5015,0.5746
2,SVM_Poly_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7335,0.7912,0.5059,0.5778,0.4679,0.5733,0.6579,0.7274,0.5134,0.8914,0.5059,0.5778,0.2783,0.3644,0.3751,0.4588
3,SVM_Poly_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5000,0.6109,0.5461,0.6607,0.4937,0.6064,0.4824,0.5980,0.5563,0.6851,0.5461,0.6607,0.5922,0.7105,0.5686,0.6852
4,SVM_Poly_classifier,FE / StandardScaler / 2 Class S_IW,0.5716,0.6977,0.5514,0.7069,0.5105,0.6556,0.5885,0.7106,0.5396,0.6805,0.5514,0.7069,0.5313,0.7160,0.5393,0.7104
5,SVM_Poly_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6196,0.7517,0.5217,0.6965,0.5055,0.6817,0.6182,0.7532,0.5224,0.7023,0.5217,0.6965,0.4238,0.6414,0.4680,0.6674
6,SVM_Poly_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5000,0.6030,0.5503,0.6564,0.4916,0.5971,0.4784,0.5870,0.5644,0.6869,0.5503,0.6564,0.6006,0.7099,0.5749,0.6826


##### MinMax

In [224]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Poly_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Poly_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Poly_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Poly_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,FE / MinMaxScaler / 3 Class,0.4546,0.5539,0.4078,0.5051,0.3751,0.4866,0.4102,0.5174,0.4329,0.6421,0.4078,0.5051,0.6731,0.7261,0.5238,0.6056
1,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6250,0.6720,0.5705,0.6152,0.5608,0.6084,0.5957,0.6412,0.5967,0.6726,0.5705,0.6152,0.5160,0.5583,0.5425,0.5861
2,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7390,0.7788,0.5021,0.5542,0.4534,0.5343,0.6531,0.7049,0.5001,0.8674,0.5021,0.5542,0.2651,0.3295,0.3648,0.4273
3,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5536,0.6480,0.5593,0.6590,0.5459,0.6445,0.5551,0.6506,0.5576,0.6542,0.5593,0.6590,0.5651,0.6701,0.5621,0.6645
4,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5453,0.6507,0.5264,0.6701,0.4992,0.6161,0.5757,0.6743,0.5200,0.6280,0.5264,0.6701,0.5074,0.6894,0.5168,0.6797
5,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5756,0.6761,0.5334,0.6688,0.5136,0.6303,0.5997,0.6958,0.5269,0.6323,0.5334,0.6688,0.4913,0.6614,0.5116,0.6650
6,SVM_Poly_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5632,0.6391,0.5745,0.6547,0.5590,0.6370,0.5656,0.6415,0.5733,0.6500,0.5745,0.6547,0.5857,0.6702,0.5800,0.6624


In [227]:
df_SVM_Poly_classifier_final = pd.concat([df_SVM_Poly_classifier_results, df_SVM_Poly_classifier_StandardScaler, df_SVM_Poly_classifier_MinMaxScaler,
                            df_SVM_Poly_classifier_results_fe, df_SVM_Poly_classifier_StandardScaler_fe, df_SVM_Poly_classifier_MinMaxScaler_fe], ignore_index=True)

df_SVM_Poly_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Poly_classifier,Normal / No Scaler / 3 Class,0.4711,0.4763,0.4135,0.4177,0.3554,0.3588,0.4020,0.4061,0.3231,0.3249,0.4135,0.4177,0.6708,0.6736,0.5265,0.5304
1,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5976,0.6154,0.4978,0.5184,0.3885,0.4167,0.4630,0.4874,0.3369,0.4744,0.4978,0.5184,0.3981,0.4215,0.4452,0.4673
2,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7527,0.7527,0.5000,0.5000,0.4295,0.4295,0.6466,0.6466,0.3764,0.3764,0.5000,0.5000,0.2473,0.2473,0.3516,0.3516
3,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5907,0.5903,0.6000,0.5999,0.5864,0.5870,0.5933,0.5936,0.5977,0.5964,0.6000,0.5999,0.6093,0.6096,0.6046,0.6047
4,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5631,0.5687,0.5605,0.5820,0.5222,0.5343,0.5895,0.5972,0.5468,0.5616,0.5605,0.5820,0.5580,0.5953,0.5590,0.5885
5,SVM_Poly_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5700,0.5573,0.5894,0.5814,0.5388,0.5284,0.5983,0.5868,0.5669,0.5610,0.5894,0.5814,0.6088,0.6056,0.5990,0.5933
6,SVM_Poly_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.6003,0.5975,0.6074,0.6035,0.5960,0.5933,0.6038,0.6016,0.6038,0.5993,0.6074,0.6035,0.6145,0.6094,0.6109,0.6064
7,SVM_Poly_classifier,Normal / StandardScaler / 3 Class,0.4395,0.5457,0.3851,0.4927,0.3303,0.4696,0.3661,0.4959,0.4093,0.7170,0.3851,0.4927,0.6534,0.7079,0.5015,0.5906
8,SVM_Poly_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6044,0.6659,0.5186,0.5824,0.4557,0.5406,0.5150,0.5907,0.5371,0.7605,0.5186,0.5824,0.4328,0.4988,0.4737,0.5390
9,SVM_Poly_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7307,0.7864,0.5059,0.5695,0.4713,0.5602,0.6585,0.7196,0.5266,0.8764,0.5059,0.5695,0.2811,0.3525,0.3769,0.4480


----

### SVM (Sigmoid kernel)  

#### Normal Data

##### NO SCALER

In [228]:
# Modelo base
SVM_Sigmoid_classifier = SVC(kernel='sigmoid')

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Sigmoid_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,Normal / No Scaler / 3 Class,0.4698,0.4602,0.4091,0.4004,0.3456,0.3367,0.3928,0.3830,0.3220,0.3121,0.4091,0.4004,0.6669,0.6611,0.5222,0.5144
1,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6044,0.6044,0.5000,0.5000,0.3767,0.3767,0.4554,0.4554,0.3022,0.3022,0.5000,0.5000,0.3956,0.3956,0.4447,0.4447
2,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7514,0.7510,0.4991,0.4989,0.4290,0.4289,0.6459,0.6457,0.3762,0.3762,0.4991,0.4989,0.2468,0.2467,0.3510,0.3508
3,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5700,0.5649,0.6012,0.5972,0.5685,0.5643,0.5665,0.5616,0.6063,0.6003,0.6012,0.5972,0.6324,0.6294,0.6166,0.6131
4,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.4448,0.4403,0.5229,0.5229,0.4178,0.4189,0.4438,0.4451,0.5215,0.5241,0.5229,0.5229,0.6010,0.6054,0.5582,0.5608
5,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.3160,0.3180,0.5102,0.5167,0.3015,0.3035,0.2558,0.2562,0.5270,0.5330,0.5102,0.5167,0.7044,0.7154,0.5994,0.6078
6,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5700,0.5694,0.5982,0.5983,0.5691,0.5692,0.5683,0.5680,0.6009,0.5996,0.5982,0.5983,0.6263,0.6273,0.6120,0.6126


##### StandardScaler

In [229]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_Sigmoid_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,Normal / StandardScaler / 3 Class,0.4258,0.4251,0.3910,0.3892,0.3712,0.3710,0.3999,0.4003,0.3784,0.3769,0.3910,0.3892,0.6764,0.6758,0.5139,0.5128
1,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5563,0.5618,0.5350,0.5408,0.5340,0.5388,0.5549,0.5593,0.5352,0.5425,0.5350,0.5408,0.5137,0.5199,0.5242,0.5302
2,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7294,0.7078,0.5181,0.5000,0.4949,0.4776,0.6687,0.6528,0.5471,0.5027,0.5181,0.5000,0.3068,0.2922,0.3984,0.3822
3,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5412,0.5385,0.5358,0.5290,0.5299,0.5249,0.5437,0.5403,0.5347,0.5283,0.5358,0.5290,0.5304,0.5195,0.5330,0.5242
4,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class S_IW,0.5453,0.5316,0.5450,0.5242,0.5066,0.4913,0.5755,0.5633,0.5336,0.5181,0.5450,0.5242,0.5447,0.5169,0.5447,0.5205
5,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5577,0.5388,0.5364,0.5202,0.5086,0.4931,0.5863,0.5699,0.5275,0.5151,0.5364,0.5202,0.5151,0.5016,0.5255,0.5107
6,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5522,0.5419,0.5471,0.5399,0.5408,0.5327,0.5545,0.5446,0.5453,0.5388,0.5471,0.5399,0.5419,0.5380,0.5444,0.5389


##### MinMax

In [230]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Sigmoid_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 3 Class,0.4450,0.4464,0.3862,0.3870,0.3224,0.3227,0.3660,0.3661,0.3457,0.3529,0.3862,0.3870,0.6511,0.6509,0.5014,0.5019
1,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5934,0.5931,0.4969,0.4954,0.4009,0.3958,0.4718,0.4679,0.4499,0.4627,0.4969,0.4954,0.4003,0.3978,0.4460,0.4439
2,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7418,0.7373,0.5077,0.5009,0.4610,0.4512,0.6574,0.6514,0.4885,0.4721,0.5077,0.5009,0.2735,0.2645,0.3724,0.3639
3,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5439,0.5429,0.5719,0.5725,0.5413,0.5394,0.5392,0.5370,0.5744,0.5764,0.5719,0.5725,0.6000,0.6020,0.5857,0.5870
4,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5934,0.6058,0.5117,0.5269,0.5041,0.5175,0.6096,0.6207,0.5092,0.5222,0.5117,0.5269,0.4300,0.4480,0.4687,0.4854
5,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5824,0.5690,0.5641,0.5538,0.5338,0.5222,0.6090,0.5965,0.5492,0.5412,0.5641,0.5538,0.5458,0.5385,0.5547,0.5459
6,SVM_Sigmoid_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5371,0.5484,0.5661,0.5772,0.5327,0.5440,0.5301,0.5420,0.5691,0.5820,0.5661,0.5772,0.5952,0.6060,0.5804,0.5913


#### FE_DATA

##### NO SCALER

In [231]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Sigmoid_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,FE / No Scaler / 3 Class,0.4807,0.4842,0.4217,0.4247,0.3618,0.3646,0.4097,0.4128,0.3266,0.3283,0.4217,0.4247,0.6776,0.6791,0.5344,0.5370
1,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6044,0.6044,0.5000,0.5000,0.3767,0.3767,0.4554,0.4554,0.3022,0.3022,0.5000,0.5000,0.3956,0.3956,0.4447,0.4447
2,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7514,0.7500,0.5028,0.5014,0.4395,0.4378,0.6508,0.6496,0.5269,0.5919,0.5028,0.5014,0.2543,0.2529,0.3575,0.3561
3,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5961,0.5920,0.6167,0.6121,0.5945,0.5914,0.5972,0.5940,0.6167,0.6098,0.6167,0.6121,0.6372,0.6322,0.6269,0.6220
4,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5891,0.6140,0.5312,0.5818,0.5127,0.5543,0.6063,0.6336,0.5270,0.5675,0.5312,0.5818,0.4733,0.5496,0.5004,0.5649
5,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5879,0.5748,0.5622,0.5698,0.5330,0.5317,0.6123,0.6011,0.5478,0.5540,0.5622,0.5698,0.5365,0.5647,0.5487,0.5669
6,SVM_Sigmoid_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5962,0.5913,0.6143,0.6107,0.5944,0.5907,0.5981,0.5937,0.6132,0.6080,0.6143,0.6107,0.6325,0.6300,0.6233,0.6202


##### StandardScaler

In [232]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_SVM_Sigmoid_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,FE / StandardScaler / 3 Class,0.4643,0.4554,0.4192,0.4108,0.3935,0.3850,0.4272,0.4189,0.4317,0.4132,0.4192,0.4108,0.6816,0.6783,0.5343,0.5279
1,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5893,0.5886,0.5558,0.5522,0.5537,0.5506,0.5798,0.5777,0.5585,0.5586,0.5558,0.5522,0.5224,0.5157,0.5388,0.5336
2,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.7376,0.7167,0.5161,0.4952,0.4871,0.4610,0.6682,0.6486,0.5742,0.4847,0.5161,0.4952,0.2945,0.2737,0.3898,0.3681
3,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5673,0.5381,0.5671,0.5345,0.5587,0.5290,0.5698,0.5415,0.5652,0.5338,0.5671,0.5345,0.5668,0.5310,0.5669,0.5327
4,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class S_IW,0.5370,0.5206,0.5506,0.5174,0.5056,0.4834,0.5677,0.5533,0.5377,0.5130,0.5506,0.5174,0.5643,0.5142,0.5574,0.5158
5,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5576,0.5415,0.5644,0.5304,0.5225,0.4992,0.5877,0.5725,0.5483,0.5228,0.5644,0.5304,0.5712,0.5192,0.5678,0.5247
6,SVM_Sigmoid_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5550,0.5384,0.5610,0.5458,0.5509,0.5349,0.5590,0.5425,0.5587,0.5441,0.5610,0.5458,0.5670,0.5531,0.5640,0.5494


##### MinMax

In [233]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(SVM_Sigmoid_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="SVM_Sigmoid_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_SVM_Sigmoid_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_SVM_Sigmoid_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,FE / MinMaxScaler / 3 Class,0.4424,0.4519,0.3967,0.4037,0.3482,0.3543,0.3861,0.3948,0.4056,0.4447,0.3967,0.4037,0.6674,0.6719,0.5143,0.5208
1,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6085,0.5989,0.5389,0.5291,0.5110,0.4995,0.5565,0.5461,0.5724,0.5507,0.5389,0.5291,0.4692,0.4592,0.5028,0.4929
2,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7390,0.7208,0.5058,0.4881,0.4618,0.4415,0.6570,0.6411,0.5892,0.4487,0.5058,0.4881,0.2726,0.2554,0.3713,0.3531
3,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5700,0.5797,0.5342,0.5446,0.5298,0.5417,0.5577,0.5688,0.5380,0.5492,0.5342,0.5446,0.4985,0.5095,0.5160,0.5267
4,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.4533,0.4801,0.5083,0.5418,0.4378,0.4641,0.4798,0.5046,0.5082,0.5346,0.5083,0.5418,0.5632,0.6035,0.5346,0.5713
5,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5014,0.5127,0.5345,0.5350,0.4764,0.4827,0.5305,0.5401,0.5277,0.5271,0.5345,0.5350,0.5677,0.5573,0.5505,0.5456
6,SVM_Sigmoid_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5742,0.5721,0.5470,0.5435,0.5417,0.5390,0.5655,0.5631,0.5492,0.5470,0.5470,0.5435,0.5197,0.5149,0.5330,0.5289


In [234]:
df_SVM_Sigmoid_classifier_final = pd.concat([df_SVM_Sigmoid_classifier_results, df_SVM_Sigmoid_classifier_StandardScaler, df_SVM_Sigmoid_classifier_MinMaxScaler,
                            df_SVM_Sigmoid_classifier_results_fe, df_SVM_Sigmoid_classifier_StandardScaler_fe, df_SVM_Sigmoid_classifier_MinMaxScaler_fe], ignore_index=True)

df_SVM_Sigmoid_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,SVM_Sigmoid_classifier,Normal / No Scaler / 3 Class,0.4698,0.4602,0.4091,0.4004,0.3456,0.3367,0.3928,0.3830,0.3220,0.3121,0.4091,0.4004,0.6669,0.6611,0.5222,0.5144
1,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6044,0.6044,0.5000,0.5000,0.3767,0.3767,0.4554,0.4554,0.3022,0.3022,0.5000,0.5000,0.3956,0.3956,0.4447,0.4447
2,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7514,0.7510,0.4991,0.4989,0.4290,0.4289,0.6459,0.6457,0.3762,0.3762,0.4991,0.4989,0.2468,0.2467,0.3510,0.3508
3,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5700,0.5649,0.6012,0.5972,0.5685,0.5643,0.5665,0.5616,0.6063,0.6003,0.6012,0.5972,0.6324,0.6294,0.6166,0.6131
4,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.4448,0.4403,0.5229,0.5229,0.4178,0.4189,0.4438,0.4451,0.5215,0.5241,0.5229,0.5229,0.6010,0.6054,0.5582,0.5608
5,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.3160,0.3180,0.5102,0.5167,0.3015,0.3035,0.2558,0.2562,0.5270,0.5330,0.5102,0.5167,0.7044,0.7154,0.5994,0.6078
6,SVM_Sigmoid_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5700,0.5694,0.5982,0.5983,0.5691,0.5692,0.5683,0.5680,0.6009,0.5996,0.5982,0.5983,0.6263,0.6273,0.6120,0.6126
7,SVM_Sigmoid_classifier,Normal / StandardScaler / 3 Class,0.4258,0.4251,0.3910,0.3892,0.3712,0.3710,0.3999,0.4003,0.3784,0.3769,0.3910,0.3892,0.6764,0.6758,0.5139,0.5128
8,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5563,0.5618,0.5350,0.5408,0.5340,0.5388,0.5549,0.5593,0.5352,0.5425,0.5350,0.5408,0.5137,0.5199,0.5242,0.5302
9,SVM_Sigmoid_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7294,0.7078,0.5181,0.5000,0.4949,0.4776,0.6687,0.6528,0.5471,0.5027,0.5181,0.5000,0.3068,0.2922,0.3984,0.3822


----

## Probabilistic Models  

### Multinomial Naive Bayes  


#### Normal Data

##### NO SCALER

In [235]:
# Modelo base
MNB_classifier = MultinomialNB()

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MNB_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MNB_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB_classifier,Normal / No Scaler / 3 Class,0.4533,0.4938,0.4149,0.4586,0.4022,0.4521,0.4317,0.4764,0.4148,0.4734,0.4149,0.4586,0.6902,0.7098,0.5350,0.5705
1,MNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5920,0.5893,0.5994,0.5974,0.5881,0.5862,0.5959,0.5936,0.5958,0.5933,0.5994,0.5974,0.6068,0.6055,0.6031,0.6014
2,MNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6785,0.7006,0.5309,0.5581,0.5289,0.5604,0.6625,0.6858,0.5375,0.5704,0.5309,0.5581,0.3833,0.4157,0.4507,0.4816
3,MNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5907,0.5924,0.6049,0.6096,0.5887,0.5915,0.5938,0.5954,0.6019,0.6063,0.6049,0.6096,0.6192,0.6267,0.6120,0.6181
4,MNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5604,0.5907,0.5401,0.5845,0.5121,0.5473,0.5884,0.6174,0.5311,0.5641,0.5401,0.5845,0.5198,0.5783,0.5297,0.5813
5,MNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5851,0.5906,0.5640,0.5817,0.5340,0.5460,0.6110,0.6170,0.5486,0.5625,0.5640,0.5817,0.5430,0.5727,0.5531,0.5770
6,MNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5907,0.5917,0.6049,0.6076,0.5886,0.5906,0.5937,0.5950,0.6022,0.6042,0.6049,0.6076,0.6191,0.6236,0.6119,0.6156


##### StandardScaler
It is not valid because StandardScaler uses a distribution with a mean of 0 and a standard deviation of 1, which introduces negative values.

##### MinMax

In [236]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MNB_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MNB_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB_classifier,Normal / MinMaxScaler / 3 Class,0.4395,0.4550,0.3795,0.3967,0.3041,0.3309,0.3443,0.3677,0.3854,0.4884,0.3795,0.3967,0.6460,0.6539,0.4951,0.5093
1,MNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6044,0.6041,0.5000,0.4999,0.3767,0.3774,0.4554,0.4558,0.3022,0.3355,0.5000,0.4999,0.3956,0.3957,0.4447,0.4447
2,MNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7431,0.7534,0.4936,0.5051,0.4262,0.4426,0.6417,0.6530,0.3751,0.5728,0.4936,0.5051,0.2441,0.2568,0.3471,0.3601
3,MNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.4780,0.5089,0.5407,0.5741,0.4575,0.4889,0.4360,0.4683,0.5598,0.6114,0.5407,0.5741,0.6033,0.6393,0.5711,0.6058
4,MNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6084,0.6521,0.5067,0.5553,0.5034,0.5515,0.6184,0.6586,0.5052,0.5511,0.5067,0.5553,0.4050,0.4586,0.4527,0.5045
5,MNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5619,0.6171,0.4853,0.5535,0.4627,0.5248,0.5617,0.6171,0.4867,0.5538,0.4853,0.5535,0.4086,0.4899,0.4427,0.5181
6,MNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.4725,0.5107,0.5313,0.5710,0.4553,0.4952,0.4355,0.4772,0.5396,0.5994,0.5313,0.5710,0.5902,0.6314,0.5599,0.6004


#### FE_DATA

##### NO SCALER

In [237]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MNB_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MNB_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB_classifier,FE / No Scaler / 3 Class,0.4643,0.4835,0.4397,0.4579,0.4381,0.4569,0.4579,0.4762,0.4458,0.4635,0.4397,0.4579,0.7087,0.7161,0.5579,0.5726
1,MNB_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5961,0.6013,0.6016,0.6078,0.5917,0.5976,0.6003,0.6057,0.5975,0.6031,0.6016,0.6078,0.6071,0.6143,0.6044,0.6110
2,MNB_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.6469,0.6693,0.5565,0.5905,0.5514,0.5831,0.6557,0.6789,0.5513,0.5805,0.5565,0.5905,0.4662,0.5118,0.5092,0.5497
3,MNB_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5934,0.5968,0.6078,0.6116,0.5915,0.5955,0.5965,0.6003,0.6047,0.6077,0.6078,0.6116,0.6222,0.6264,0.6149,0.6189
4,MNB_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5824,0.6027,0.5566,0.5929,0.5299,0.5574,0.6085,0.6288,0.5436,0.5705,0.5566,0.5929,0.5308,0.5832,0.5434,0.5880
5,MNB_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5796,0.6030,0.5566,0.5927,0.5293,0.5575,0.6069,0.6291,0.5433,0.5704,0.5566,0.5927,0.5337,0.5823,0.5450,0.5874
6,MNB_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5961,0.5944,0.6089,0.6081,0.5938,0.5929,0.5996,0.5981,0.6057,0.6042,0.6089,0.6081,0.6216,0.6218,0.6152,0.6149


##### StandardScaler
It is not valid because StandardScaler uses a distribution with a mean of 0 and a standard deviation of 1, which introduces negative values.

##### MinMax

In [238]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MNB_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MNB_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB_classifier,FE / MinMaxScaler / 3 Class,0.4300,0.4749,0.3825,0.4267,0.3467,0.3988,0.3800,0.4298,0.4003,0.4802,0.3825,0.4267,0.6577,0.6803,0.5014,0.5388
1,MNB_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6333,0.6322,0.5804,0.5801,0.5725,0.5736,0.6061,0.6066,0.6067,0.6067,0.5804,0.5801,0.5274,0.5281,0.5532,0.5535
2,MNB_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7390,0.7527,0.4927,0.5093,0.4292,0.4543,0.6417,0.6583,0.3861,0.6816,0.4927,0.5093,0.2464,0.2659,0.3484,0.3679
3,MNB_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5591,0.5848,0.5694,0.5985,0.5559,0.5829,0.5627,0.5879,0.5672,0.5955,0.5694,0.5985,0.5796,0.6122,0.5744,0.6053
4,MNB_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.5413,0.5834,0.5238,0.5745,0.4945,0.5369,0.5697,0.6077,0.5179,0.5587,0.5238,0.5745,0.5062,0.5656,0.5146,0.5695
5,MNB_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.5345,0.5724,0.5380,0.5742,0.4951,0.5291,0.5607,0.5956,0.5289,0.5589,0.5380,0.5742,0.5414,0.5760,0.5390,0.5743
6,MNB_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5509,0.5656,0.5757,0.5931,0.5492,0.5642,0.5496,0.5631,0.5783,0.5945,0.5757,0.5931,0.6004,0.6206,0.5879,0.6067


In [239]:
df_MNB_classifier_final = pd.concat([df_MNB_classifier_results, df_MNB_classifier_MinMaxScaler,
                            df_MNB_classifier_results_fe, df_MNB_classifier_MinMaxScaler_fe], ignore_index=True)

df_MNB_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MNB_classifier,Normal / No Scaler / 3 Class,0.4533,0.4938,0.4149,0.4586,0.4022,0.4521,0.4317,0.4764,0.4148,0.4734,0.4149,0.4586,0.6902,0.7098,0.5350,0.5705
1,MNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5920,0.5893,0.5994,0.5974,0.5881,0.5862,0.5959,0.5936,0.5958,0.5933,0.5994,0.5974,0.6068,0.6055,0.6031,0.6014
2,MNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6785,0.7006,0.5309,0.5581,0.5289,0.5604,0.6625,0.6858,0.5375,0.5704,0.5309,0.5581,0.3833,0.4157,0.4507,0.4816
3,MNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5907,0.5924,0.6049,0.6096,0.5887,0.5915,0.5938,0.5954,0.6019,0.6063,0.6049,0.6096,0.6192,0.6267,0.6120,0.6181
4,MNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.5604,0.5907,0.5401,0.5845,0.5121,0.5473,0.5884,0.6174,0.5311,0.5641,0.5401,0.5845,0.5198,0.5783,0.5297,0.5813
5,MNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5851,0.5906,0.5640,0.5817,0.5340,0.5460,0.6110,0.6170,0.5486,0.5625,0.5640,0.5817,0.5430,0.5727,0.5531,0.5770
6,MNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5907,0.5917,0.6049,0.6076,0.5886,0.5906,0.5937,0.5950,0.6022,0.6042,0.6049,0.6076,0.6191,0.6236,0.6119,0.6156
7,MNB_classifier,Normal / MinMaxScaler / 3 Class,0.4395,0.4550,0.3795,0.3967,0.3041,0.3309,0.3443,0.3677,0.3854,0.4884,0.3795,0.3967,0.6460,0.6539,0.4951,0.5093
8,MNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6044,0.6041,0.5000,0.4999,0.3767,0.3774,0.4554,0.4558,0.3022,0.3355,0.5000,0.4999,0.3956,0.3957,0.4447,0.4447
9,MNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7431,0.7534,0.4936,0.5051,0.4262,0.4426,0.6417,0.6530,0.3751,0.5728,0.4936,0.5051,0.2441,0.2568,0.3471,0.3601


### Bernoulli Naive Bayes  


#### Normal Data

##### NO SCALER

In [240]:
# Modelo base
BNB_classifier = BernoulliNB()

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_BNB_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,Normal / No Scaler / 3 Class,0.4135,0.4536,0.3637,0.4048,0.3153,0.3620,0.3478,0.3916,0.3966,0.4686,0.3637,0.4048,0.6445,0.6660,0.4840,0.5192
1,BNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5989,0.6082,0.4979,0.5097,0.3865,0.4107,0.4618,0.4819,0.3850,0.6038,0.4979,0.5097,0.3969,0.4113,0.4445,0.4579
2,BNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7390,0.7500,0.4983,0.5108,0.4432,0.4606,0.6482,0.6603,0.5260,0.7036,0.4983,0.5108,0.2576,0.2715,0.3582,0.3723
3,BNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5179,0.5216,0.5605,0.5678,0.5135,0.5162,0.5044,0.5058,0.5665,0.5796,0.5605,0.5678,0.6031,0.6139,0.5814,0.5904
4,BNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6072,0.6274,0.5021,0.5301,0.4974,0.5261,0.6154,0.6364,0.4989,0.5274,0.5021,0.5301,0.3970,0.4327,0.4457,0.4788
5,BNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5345,0.5855,0.4783,0.5418,0.4421,0.5017,0.5310,0.5848,0.4706,0.5425,0.4783,0.5418,0.4220,0.4982,0.4456,0.5164
6,BNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.4875,0.5134,0.5377,0.5652,0.4766,0.5043,0.4618,0.4909,0.5405,0.5830,0.5377,0.5652,0.5879,0.6170,0.5622,0.5905


##### StandardScaler

In [241]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_BNB_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,Normal / StandardScaler / 3 Class,0.4698,0.4907,0.4252,0.4489,0.4032,0.4333,0.4366,0.4621,0.4366,0.4760,0.4252,0.4489,0.6899,0.6998,0.5415,0.5605
1,BNB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6031,0.6051,0.5953,0.5955,0.5919,0.5931,0.6057,0.6076,0.5929,0.5930,0.5953,0.5955,0.5875,0.5859,0.5914,0.5907
2,BNB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7376,0.7479,0.5086,0.5220,0.4708,0.4880,0.6605,0.6723,0.5772,0.6175,0.5086,0.5220,0.2796,0.2961,0.3770,0.3929
3,BNB_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5742,0.5845,0.5890,0.5990,0.5729,0.5831,0.5776,0.5880,0.5862,0.5956,0.5890,0.5990,0.6037,0.6135,0.5963,0.6062
4,BNB_classifier,Normal / StandardScaler / 2 Class S_IW,0.5866,0.6044,0.5389,0.5721,0.5216,0.5466,0.6100,0.6275,0.5311,0.5562,0.5389,0.5721,0.4912,0.5399,0.5144,0.5555
5,BNB_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5687,0.6027,0.5438,0.5822,0.5173,0.5515,0.5961,0.6274,0.5337,0.5631,0.5438,0.5822,0.5189,0.5617,0.5311,0.5717
6,BNB_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5756,0.5807,0.5889,0.5956,0.5739,0.5794,0.5792,0.5841,0.5861,0.5924,0.5889,0.5956,0.6023,0.6104,0.5956,0.6029


##### MinMax

In [242]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_BNB_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,Normal / MinMaxScaler / 3 Class,0.4121,0.4550,0.3635,0.4068,0.3179,0.3673,0.3497,0.3969,0.3975,0.4672,0.3635,0.4068,0.6448,0.6677,0.4840,0.5212
1,BNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5989,0.6095,0.4985,0.5115,0.3895,0.4141,0.4641,0.4847,0.4780,0.6163,0.4985,0.5115,0.3981,0.4134,0.4454,0.4598
2,BNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7390,0.7493,0.4983,0.5103,0.4432,0.4603,0.6482,0.6599,0.5260,0.6514,0.4983,0.5103,0.2576,0.2713,0.3582,0.3720
3,BNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.4931,0.5151,0.5424,0.5658,0.4849,0.5066,0.4716,0.4936,0.5508,0.5808,0.5424,0.5658,0.5916,0.6166,0.5665,0.5906
4,BNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.5632,0.5917,0.5065,0.5366,0.4931,0.5215,0.5876,0.6132,0.5045,0.5295,0.5065,0.5366,0.4499,0.4816,0.4771,0.5082
5,BNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5331,0.5865,0.4773,0.5425,0.4420,0.5026,0.5312,0.5859,0.4708,0.5431,0.4773,0.5425,0.4215,0.4985,0.4451,0.5170
6,BNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.4889,0.5134,0.5395,0.5652,0.4778,0.5043,0.4629,0.4909,0.5426,0.5828,0.5395,0.5652,0.5900,0.6170,0.5642,0.5905


#### FE_DATA

##### NO SCALER

In [243]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_BNB_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,FE / No Scaler / 3 Class,0.4121,0.4595,0.3702,0.4179,0.3359,0.3934,0.3656,0.4210,0.3738,0.4518,0.3702,0.4179,0.6600,0.6822,0.4942,0.5339
1,BNB_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.6209,0.6326,0.5689,0.5821,0.5610,0.5765,0.5948,0.6087,0.5895,0.6069,0.5689,0.5821,0.5169,0.5316,0.5422,0.5563
2,BNB_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7376,0.7521,0.4974,0.5140,0.4426,0.4662,0.6474,0.6636,0.4758,0.6602,0.4974,0.5140,0.2572,0.2760,0.3576,0.3765
3,BNB_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5769,0.5900,0.5671,0.5865,0.5585,0.5784,0.5740,0.5901,0.5684,0.5861,0.5671,0.5865,0.5573,0.5829,0.5620,0.5846
4,BNB_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.5138,0.5247,0.5222,0.5328,0.4814,0.4918,0.5464,0.5569,0.5167,0.5245,0.5222,0.5328,0.5306,0.5408,0.5262,0.5367
5,BNB_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.5030,0.5219,0.5115,0.5430,0.4611,0.4847,0.5240,0.5424,0.5092,0.5352,0.5115,0.5430,0.5200,0.5640,0.5141,0.5520
6,BNB_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5933,0.6109,0.5738,0.5946,0.5710,0.5935,0.5900,0.6105,0.5758,0.5953,0.5738,0.5946,0.5542,0.5782,0.5638,0.5863


##### StandardScaler

In [244]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_BNB_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,FE / StandardScaler / 3 Class,0.4657,0.4842,0.4398,0.4598,0.4361,0.4587,0.4573,0.4781,0.4408,0.4625,0.4398,0.4598,0.7107,0.7199,0.5588,0.5754
1,BNB_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.6045,0.6054,0.6002,0.6010,0.5957,0.5968,0.6081,0.6091,0.5965,0.5974,0.6002,0.6010,0.5960,0.5966,0.5981,0.5988
2,BNB_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.6661,0.6885,0.5338,0.5581,0.5353,0.5593,0.6597,0.6798,0.5393,0.5643,0.5338,0.5581,0.4016,0.4276,0.4629,0.4884
3,BNB_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5935,0.5924,0.6001,0.5983,0.5898,0.5885,0.5977,0.5968,0.5960,0.5941,0.6001,0.5983,0.6067,0.6042,0.6034,0.6013
4,BNB_classifier,FE / StandardScaler / 2 Class S_IW,0.5810,0.5948,0.5781,0.5975,0.5403,0.5554,0.6089,0.6220,0.5592,0.5733,0.5781,0.5975,0.5751,0.6001,0.5766,0.5988
5,BNB_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.5741,0.5937,0.5716,0.5972,0.5341,0.5550,0.6028,0.6213,0.5540,0.5730,0.5716,0.5972,0.5691,0.6007,0.5704,0.5990
6,BNB_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5880,0.5924,0.5956,0.5988,0.5846,0.5887,0.5922,0.5968,0.5917,0.5945,0.5956,0.5988,0.6032,0.6051,0.5993,0.6019


##### MinMax

In [245]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(BNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="BNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_BNB_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_BNB_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,FE / MinMaxScaler / 3 Class,0.4080,0.4588,0.3673,0.4170,0.3348,0.3923,0.3636,0.4198,0.3703,0.4527,0.3673,0.4170,0.6584,0.6815,0.4916,0.5330
1,BNB_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.6223,0.6329,0.5712,0.5831,0.5637,0.5780,0.5970,0.6097,0.5908,0.6073,0.5712,0.5831,0.5201,0.5333,0.5450,0.5577
2,BNB_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7376,0.7510,0.4974,0.5138,0.4426,0.4670,0.6474,0.6636,0.4758,0.6539,0.4974,0.5138,0.2572,0.2765,0.3576,0.3768
3,BNB_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.6057,0.6168,0.5659,0.5834,0.5626,0.5831,0.5909,0.6076,0.5756,0.5924,0.5659,0.5834,0.5262,0.5499,0.5456,0.5664
4,BNB_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.4905,0.5103,0.5142,0.5367,0.4628,0.4815,0.5195,0.5384,0.5111,0.5282,0.5142,0.5367,0.5380,0.5630,0.5255,0.5491
5,BNB_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.4837,0.4972,0.5191,0.5448,0.4572,0.4755,0.5091,0.5220,0.5136,0.5357,0.5191,0.5448,0.5545,0.5923,0.5355,0.5674
6,BNB_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5975,0.6126,0.5784,0.5965,0.5755,0.5953,0.5942,0.6123,0.5804,0.5971,0.5784,0.5965,0.5593,0.5803,0.5687,0.5883


In [246]:
df_BNB_classifier_final = pd.concat([df_BNB_classifier_results,df_BNB_classifier_StandardScaler, df_BNB_classifier_MinMaxScaler,
                                    df_BNB_classifier_results_fe, df_BNB_classifier_StandardScaler_fe, df_BNB_classifier_MinMaxScaler_fe], ignore_index=True)

df_BNB_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,BNB_classifier,Normal / No Scaler / 3 Class,0.4135,0.4536,0.3637,0.4048,0.3153,0.3620,0.3478,0.3916,0.3966,0.4686,0.3637,0.4048,0.6445,0.6660,0.4840,0.5192
1,BNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5989,0.6082,0.4979,0.5097,0.3865,0.4107,0.4618,0.4819,0.3850,0.6038,0.4979,0.5097,0.3969,0.4113,0.4445,0.4579
2,BNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7390,0.7500,0.4983,0.5108,0.4432,0.4606,0.6482,0.6603,0.5260,0.7036,0.4983,0.5108,0.2576,0.2715,0.3582,0.3723
3,BNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5179,0.5216,0.5605,0.5678,0.5135,0.5162,0.5044,0.5058,0.5665,0.5796,0.5605,0.5678,0.6031,0.6139,0.5814,0.5904
4,BNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6072,0.6274,0.5021,0.5301,0.4974,0.5261,0.6154,0.6364,0.4989,0.5274,0.5021,0.5301,0.3970,0.4327,0.4457,0.4788
5,BNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5345,0.5855,0.4783,0.5418,0.4421,0.5017,0.5310,0.5848,0.4706,0.5425,0.4783,0.5418,0.4220,0.4982,0.4456,0.5164
6,BNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.4875,0.5134,0.5377,0.5652,0.4766,0.5043,0.4618,0.4909,0.5405,0.5830,0.5377,0.5652,0.5879,0.6170,0.5622,0.5905
7,BNB_classifier,Normal / StandardScaler / 3 Class,0.4698,0.4907,0.4252,0.4489,0.4032,0.4333,0.4366,0.4621,0.4366,0.4760,0.4252,0.4489,0.6899,0.6998,0.5415,0.5605
8,BNB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.6031,0.6051,0.5953,0.5955,0.5919,0.5931,0.6057,0.6076,0.5929,0.5930,0.5953,0.5955,0.5875,0.5859,0.5914,0.5907
9,BNB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.7376,0.7479,0.5086,0.5220,0.4708,0.4880,0.6605,0.6723,0.5772,0.6175,0.5086,0.5220,0.2796,0.2961,0.3770,0.3929


----

### Gaussian Naive Bayes

#### Normal Data

##### NO SCALER

In [247]:
# Modelo base
GNB_classifier = GaussianNB()

# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GNB_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,Normal / No Scaler / 3 Class,0.4368,0.4677,0.3909,0.4207,0.3543,0.3906,0.3840,0.4198,0.4237,0.4760,0.3909,0.4207,0.6630,0.6784,0.5091,0.5342
1,GNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5207,0.5278,0.5685,0.5781,0.5110,0.5186,0.4990,0.5062,0.5819,0.5970,0.5685,0.5781,0.6164,0.6283,0.5919,0.6026
2,GNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6964,0.7174,0.5111,0.5320,0.4973,0.5230,0.6567,0.6766,0.5207,0.5638,0.5111,0.5320,0.3258,0.3466,0.4073,0.4291
3,GNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5096,0.5192,0.5600,0.5738,0.5005,0.5082,0.4872,0.4934,0.5724,0.5947,0.5600,0.5738,0.6104,0.6284,0.5847,0.6005
4,GNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.3862,0.4285,0.4768,0.5290,0.3706,0.4156,0.3861,0.4299,0.4726,0.5257,0.4768,0.5290,0.5673,0.6294,0.5186,0.5759
5,GNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5566,0.6050,0.5024,0.5590,0.4626,0.5143,0.5535,0.6006,0.5011,0.5637,0.5024,0.5590,0.4481,0.5129,0.4701,0.5313
6,GNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5042,0.5182,0.5567,0.5743,0.4917,0.5053,0.4767,0.4896,0.5696,0.5993,0.5567,0.5743,0.6092,0.6304,0.5823,0.6017


##### StandardScaler

In [248]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_GNB_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_StandardScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,Normal / StandardScaler / 3 Class,0.4272,0.4626,0.3820,0.4155,0.3392,0.3796,0.3676,0.4081,0.4040,0.4793,0.3820,0.4155,0.6564,0.6742,0.5006,0.5292
1,GNB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5097,0.5144,0.5600,0.5686,0.4941,0.4960,0.4794,0.4796,0.5688,0.5912,0.5600,0.5686,0.6104,0.6229,0.5846,0.5950
2,GNB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6964,0.7174,0.5111,0.5320,0.4973,0.5230,0.6567,0.6766,0.5207,0.5638,0.5111,0.5320,0.3258,0.3466,0.4073,0.4291
3,GNB_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.4946,0.5096,0.5506,0.5690,0.4762,0.4910,0.4583,0.4725,0.5617,0.5968,0.5506,0.5690,0.6065,0.6284,0.5778,0.5979
4,GNB_classifier,Normal / StandardScaler / 2 Class S_IW,0.4785,0.5232,0.4990,0.5522,0.4196,0.4702,0.4731,0.5202,0.4876,0.5570,0.4990,0.5522,0.5196,0.5812,0.5044,0.5626
5,GNB_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.5566,0.6050,0.5024,0.5590,0.4626,0.5143,0.5535,0.6006,0.5011,0.5637,0.5024,0.5590,0.4481,0.5129,0.4701,0.5313
6,GNB_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.4946,0.5103,0.5505,0.5690,0.4742,0.4917,0.4560,0.4736,0.5617,0.5997,0.5505,0.5690,0.6064,0.6277,0.5777,0.5975


##### MinMax

In [249]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GNB_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_MinMaxScaler


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,Normal / MinMaxScaler / 3 Class,0.4327,0.4657,0.3871,0.4186,0.3483,0.3865,0.3775,0.4156,0.4192,0.4771,0.3871,0.4186,0.6601,0.6766,0.5054,0.5322
1,GNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5111,0.5185,0.5612,0.5716,0.4962,0.5035,0.4819,0.4884,0.5717,0.5942,0.5612,0.5716,0.6113,0.6247,0.5856,0.5975
2,GNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.6964,0.7174,0.5111,0.5320,0.4973,0.5230,0.6567,0.6766,0.5207,0.5638,0.5111,0.5320,0.3258,0.3466,0.4073,0.4291
3,GNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.4959,0.5127,0.5505,0.5720,0.4806,0.4955,0.4641,0.4775,0.5662,0.6028,0.5505,0.5720,0.6050,0.6313,0.5770,0.6009
4,GNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.4551,0.4995,0.4947,0.5439,0.4016,0.4495,0.4440,0.4902,0.4840,0.5483,0.4947,0.5439,0.5343,0.5883,0.5092,0.5616
5,GNB_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.5566,0.6050,0.5024,0.5590,0.4626,0.5143,0.5535,0.6006,0.5011,0.5637,0.5024,0.5590,0.4481,0.5129,0.4701,0.5313
6,GNB_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5042,0.5144,0.5585,0.5722,0.4890,0.4983,0.4728,0.4811,0.5785,0.6029,0.5585,0.5722,0.6127,0.6301,0.5849,0.6004


#### FE_DATA

##### NO SCALER

In [250]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GNB_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,FE / No Scaler / 3 Class,0.4231,0.4578,0.4002,0.4347,0.3671,0.4043,0.3848,0.4219,0.4253,0.4788,0.4002,0.4347,0.6876,0.7055,0.5245,0.5538
1,GNB_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5358,0.5443,0.5823,0.5907,0.5263,0.5357,0.5153,0.5254,0.5970,0.6091,0.5823,0.5907,0.6288,0.6370,0.6050,0.6133
2,GNB_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.5205,0.5515,0.5117,0.5566,0.4384,0.4746,0.5012,0.5285,0.5059,0.5707,0.5117,0.5566,0.5030,0.5617,0.5014,0.5537
3,GNB_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5069,0.5247,0.5632,0.5813,0.4933,0.5124,0.4769,0.4969,0.5830,0.6067,0.5632,0.5813,0.6195,0.6380,0.5907,0.6090
4,GNB_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.3820,0.4104,0.5000,0.5430,0.3767,0.4044,0.3801,0.4048,0.4953,0.5432,0.5000,0.5430,0.6180,0.6757,0.5554,0.6053
5,GNB_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.4835,0.4976,0.5301,0.5553,0.4392,0.4533,0.4760,0.4863,0.5230,0.5587,0.5301,0.5553,0.5767,0.6130,0.5494,0.5799
6,GNB_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5165,0.5237,0.5699,0.5790,0.5027,0.5111,0.4874,0.4960,0.5871,0.6055,0.5699,0.5790,0.6234,0.6343,0.5960,0.6060


##### StandardScaler

In [251]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_GNB_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_StandardScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,FE / StandardScaler / 3 Class,0.4052,0.4275,0.3863,0.4093,0.3213,0.3507,0.3329,0.3635,0.3750,0.4978,0.3863,0.4093,0.6797,0.6943,0.5121,0.5330
1,GNB_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5207,0.5333,0.5698,0.5828,0.5040,0.5171,0.4898,0.5035,0.5726,0.6056,0.5698,0.5828,0.6188,0.6323,0.5937,0.6069
2,GNB_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.5040,0.5289,0.5119,0.5481,0.4150,0.4442,0.4641,0.4843,0.4987,0.5738,0.5119,0.5481,0.5199,0.5673,0.5087,0.5511
3,GNB_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.4973,0.5172,0.5540,0.5751,0.4800,0.5005,0.4621,0.4833,0.5702,0.6045,0.5540,0.5751,0.6108,0.6331,0.5817,0.6033
4,GNB_classifier,FE / StandardScaler / 2 Class S_IW,0.3778,0.4055,0.5103,0.5496,0.3551,0.3822,0.3463,0.3671,0.5069,0.5650,0.5103,0.5496,0.6429,0.6937,0.5708,0.6156
5,GNB_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.4656,0.4763,0.5294,0.5472,0.4141,0.4251,0.4365,0.4452,0.5138,0.5560,0.5294,0.5472,0.5932,0.6181,0.5560,0.5773
6,GNB_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5069,0.5161,0.5620,0.5734,0.4883,0.4991,0.4710,0.4820,0.5708,0.6024,0.5620,0.5734,0.6171,0.6306,0.5888,0.6012


##### MinMax

In [252]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(GNB_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="GNB_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_GNB_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_GNB_classifier_MinMaxScaler_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,FE / MinMaxScaler / 3 Class,0.4011,0.4344,0.3838,0.4156,0.3263,0.3636,0.3374,0.3776,0.3793,0.4996,0.3838,0.4156,0.6797,0.6982,0.5104,0.5386
1,GNB_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5276,0.5405,0.5754,0.5883,0.5146,0.5291,0.5019,0.5176,0.5874,0.6106,0.5754,0.5883,0.6233,0.6361,0.5988,0.6116
2,GNB_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.5012,0.5344,0.5045,0.5503,0.4135,0.4518,0.4646,0.4953,0.4835,0.5724,0.5045,0.5503,0.5078,0.5663,0.4992,0.5520
3,GNB_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5028,0.5203,0.5586,0.5777,0.4876,0.5055,0.4709,0.4890,0.5771,0.6070,0.5586,0.5777,0.6144,0.6351,0.5858,0.6057
4,GNB_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.3600,0.3986,0.5003,0.5460,0.3389,0.3773,0.3238,0.3604,0.4939,0.5614,0.5003,0.5460,0.6407,0.6933,0.5645,0.6136
5,GNB_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.4697,0.4835,0.5284,0.5501,0.4210,0.4352,0.4476,0.4599,0.5137,0.5586,0.5284,0.5501,0.5871,0.6168,0.5529,0.5785
6,GNB_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5165,0.5213,0.5699,0.5770,0.5027,0.5077,0.4874,0.4922,0.5871,0.6033,0.5699,0.5770,0.6234,0.6327,0.5960,0.6042


In [253]:
df_GNB_classifier_final = pd.concat([df_GNB_classifier_results,df_GNB_classifier_StandardScaler, df_GNB_classifier_MinMaxScaler,
                                    df_GNB_classifier_results_fe, df_GNB_classifier_StandardScaler_fe, df_GNB_classifier_MinMaxScaler_fe], ignore_index=True)

df_GNB_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,GNB_classifier,Normal / No Scaler / 3 Class,0.4368,0.4677,0.3909,0.4207,0.3543,0.3906,0.3840,0.4198,0.4237,0.4760,0.3909,0.4207,0.6630,0.6784,0.5091,0.5342
1,GNB_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5207,0.5278,0.5685,0.5781,0.5110,0.5186,0.4990,0.5062,0.5819,0.5970,0.5685,0.5781,0.6164,0.6283,0.5919,0.6026
2,GNB_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.6964,0.7174,0.5111,0.5320,0.4973,0.5230,0.6567,0.6766,0.5207,0.5638,0.5111,0.5320,0.3258,0.3466,0.4073,0.4291
3,GNB_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5096,0.5192,0.5600,0.5738,0.5005,0.5082,0.4872,0.4934,0.5724,0.5947,0.5600,0.5738,0.6104,0.6284,0.5847,0.6005
4,GNB_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.3862,0.4285,0.4768,0.5290,0.3706,0.4156,0.3861,0.4299,0.4726,0.5257,0.4768,0.5290,0.5673,0.6294,0.5186,0.5759
5,GNB_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5566,0.6050,0.5024,0.5590,0.4626,0.5143,0.5535,0.6006,0.5011,0.5637,0.5024,0.5590,0.4481,0.5129,0.4701,0.5313
6,GNB_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5042,0.5182,0.5567,0.5743,0.4917,0.5053,0.4767,0.4896,0.5696,0.5993,0.5567,0.5743,0.6092,0.6304,0.5823,0.6017
7,GNB_classifier,Normal / StandardScaler / 3 Class,0.4272,0.4626,0.3820,0.4155,0.3392,0.3796,0.3676,0.4081,0.4040,0.4793,0.3820,0.4155,0.6564,0.6742,0.5006,0.5292
8,GNB_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5097,0.5144,0.5600,0.5686,0.4941,0.4960,0.4794,0.4796,0.5688,0.5912,0.5600,0.5686,0.6104,0.6229,0.5846,0.5950
9,GNB_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6964,0.7174,0.5111,0.5320,0.4973,0.5230,0.6567,0.6766,0.5207,0.5638,0.5111,0.5320,0.3258,0.3466,0.4073,0.4291


----

# Neural Networks

### MLP

#### Normal Data

##### NO SCALER

In [254]:
# Modelo base
MLP_classifier = MLPClassifier(
    hidden_layer_sizes=(64, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    max_iter=500,
    random_state=42
)


# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MLP_classifier_results = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_results



,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,Normal / No Scaler / 3 Class,0.4574,0.5361,0.4165,0.4947,0.4002,0.4839,0.4284,0.5092,0.4405,0.5574,0.4165,0.4947,0.6786,0.7227,0.5315,0.5979
1,MLP_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6305,0.6628,0.5614,0.5931,0.5333,0.5703,0.5774,0.6117,0.6073,0.6854,0.5614,0.5931,0.4922,0.5234,0.5256,0.5571
2,MLP_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7486,0.7589,0.5066,0.5204,0.4527,0.4758,0.6561,0.6705,0.5176,0.7535,0.5066,0.5204,0.2646,0.2819,0.3659,0.3830
3,MLP_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5288,0.5900,0.5585,0.6192,0.5175,0.5799,0.5137,0.5785,0.5686,0.6355,0.5585,0.6192,0.5881,0.6483,0.5729,0.6334
4,MLP_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6225,0.6837,0.5349,0.6233,0.4928,0.5939,0.5992,0.6724,0.5504,0.6626,0.5349,0.6233,0.4473,0.5630,0.4838,0.5892
5,MLP_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5552,0.6287,0.5163,0.6092,0.4588,0.5363,0.5383,0.6077,0.5447,0.6549,0.5163,0.6092,0.4774,0.5898,0.4909,0.5933
6,MLP_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5372,0.5900,0.5594,0.6086,0.5175,0.5714,0.5159,0.5736,0.5720,0.6284,0.5594,0.6086,0.5817,0.6273,0.5700,0.6175


##### StandardScaler

In [255]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_MLP_classifier_StandardScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_StandardScaler


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) 

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,Normal / StandardScaler / 3 Class,0.3874,0.8901,0.3704,0.8828,0.3644,0.8887,0.3780,0.8899,0.3703,0.8979,0.3704,0.8828,0.6676,0.9369,0.4971,0.9094
1,MLP_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5660,0.9196,0.5426,0.9137,0.5398,0.9156,0.5619,0.9194,0.5410,0.9178,0.5426,0.9137,0.5192,0.9078,0.5307,0.9108
2,MLP_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6799,0.9392,0.5244,0.8850,0.5229,0.9120,0.6611,0.9367,0.5302,0.9515,0.5244,0.8850,0.3688,0.8308,0.4395,0.8575
3,MLP_classifier,Normal / StandardScaler / 2 Class IS_W / SMOTE,0.5741,0.9293,0.5560,0.9280,0.5539,0.9263,0.5730,0.9294,0.5552,0.9253,0.5560,0.9280,0.5380,0.9267,0.5469,0.9273
4,MLP_classifier,Normal / StandardScaler / 2 Class S_IW,0.6400,0.9571,0.5258,0.9510,0.5240,0.9433,0.6425,0.9574,0.5240,0.9364,0.5258,0.9510,0.4117,0.9449,0.4649,0.9479
5,MLP_classifier,Normal / StandardScaler / 2 Class S_IW / RANDO...,0.6648,0.9646,0.5292,0.9574,0.5293,0.9529,0.6568,0.9648,0.5313,0.9489,0.5292,0.9574,0.3937,0.9501,0.4563,0.9538
6,MLP_classifier,Normal / StandardScaler / 2 Class IS_W / RANDO...,0.5714,0.9248,0.5555,0.9261,0.5512,0.9221,0.5698,0.9251,0.5529,0.9197,0.5555,0.9261,0.5397,0.9274,0.5475,0.9267


##### MinMax

In [256]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="Normal / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MLP_classifier_MinMaxScaler = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_MinMaxScaler


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) 

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,Normal / MinMaxScaler / 3 Class,0.4176,0.6538,0.3918,0.6317,0.3858,0.6390,0.4045,0.6490,0.3971,0.6659,0.3918,0.6317,0.6748,0.8016,0.5137,0.7116
1,MLP_classifier,Normal / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5962,0.7600,0.5622,0.7290,0.5617,0.7356,0.5873,0.7523,0.5682,0.7599,0.5622,0.7290,0.5283,0.6981,0.5450,0.7134
2,MLP_classifier,Normal / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.7129,0.8324,0.5276,0.6854,0.5192,0.7177,0.6733,0.8086,0.5490,0.8382,0.5276,0.6854,0.3423,0.5383,0.4246,0.6074
3,MLP_classifier,Normal / MinMaxScaler / 2 Class IS_W / SMOTE,0.5385,0.7455,0.5349,0.7465,0.5297,0.7395,0.5427,0.7477,0.5333,0.7382,0.5349,0.7465,0.5313,0.7474,0.5331,0.7469
4,MLP_classifier,Normal / MinMaxScaler / 2 Class S_IW / SMOTE,0.6030,0.7905,0.5367,0.7788,0.5261,0.7466,0.6219,0.7998,0.5314,0.7341,0.5367,0.7788,0.4704,0.7670,0.5024,0.7729
5,MLP_classifier,Normal / MinMaxScaler / 2 Class S_IW / RANDOM ...,0.6044,0.8108,0.5302,0.7927,0.5205,0.7664,0.6216,0.8178,0.5237,0.7528,0.5302,0.7927,0.4560,0.7746,0.4913,0.7836
6,MLP_classifier,Normal / MinMaxScaler / 2 Class IS_W / RANDOM ...,0.5454,0.7466,0.5461,0.7494,0.5390,0.7410,0.5498,0.7487,0.5442,0.7414,0.5461,0.7494,0.5468,0.7522,0.5464,0.7508


#### FE_DATA

##### NO SCALER

In [257]:
# 1) NO SCALER 3 CLASS
pipe = Pipeline([
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 3 Class",
    sep=" ± "
)

# 2) NO SCALER 2 CLASS IS_W

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) NO SCALER 2 CLASS S_IW

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)

# 4) NO SCALER 2 CLASS IS_W / SMOTE
pipe_smote = Pipeline(steps=[
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) NO SCALER 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) NO SCALER 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) NO SCALER 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / No Scaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MLP_classifier_results_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_results_fe


,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,FE / No Scaler / 3 Class,0.4231,0.5663,0.3879,0.5321,0.3663,0.5263,0.3899,0.5440,0.4173,0.6121,0.3879,0.5321,0.6631,0.7427,0.5070,0.6286
1,MLP_classifier,FE / No Scaler / 2 Class IS_W / UNBALANCED,0.5797,0.7321,0.5282,0.6840,0.5164,0.6791,0.5520,0.7055,0.5459,0.7665,0.5282,0.6840,0.4767,0.6358,0.5017,0.6593
2,MLP_classifier,FE / No Scaler / 2 Class S_IW / UNBALANCED,0.7252,0.8369,0.5451,0.6990,0.5422,0.7284,0.6884,0.8150,0.5818,0.8376,0.5451,0.6990,0.3651,0.5612,0.4459,0.6260
3,MLP_classifier,FE / No Scaler / 2 Class IS_W / SMOTE,0.5413,0.7133,0.5201,0.6950,0.5164,0.6935,0.5372,0.7086,0.5219,0.7074,0.5201,0.6950,0.4990,0.6767,0.5094,0.6857
4,MLP_classifier,FE / No Scaler / 2 Class S_IW / SMOTE,0.6069,0.7198,0.5746,0.7197,0.5441,0.6754,0.6247,0.7317,0.5601,0.6887,0.5746,0.7197,0.5423,0.7196,0.5570,0.7189
5,MLP_classifier,FE / No Scaler / 2 Class S_IW / RANDOM UNDERSA...,0.6512,0.7795,0.5651,0.7407,0.5572,0.7206,0.6568,0.7840,0.5663,0.7249,0.5651,0.7407,0.4789,0.7018,0.5195,0.7204
6,MLP_classifier,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...,0.5564,0.6919,0.5387,0.6817,0.5201,0.6711,0.5405,0.6837,0.5416,0.7050,0.5387,0.6817,0.5211,0.6714,0.5295,0.6763


##### StandardScaler

In [258]:
# 1) StandardScaler 3 CLASS
pipe = Pipeline([
    ("scaler",StandardScaler()),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 3 Class",
    sep=" ± "
)

# 2) StandardScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / UNBALANCED ",
    sep=" ± "
)

# 3) StandardScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / UNBALANCED ",
    sep=" ± "
)

# 4) StandardScaler 2 CLASS IS_W /SMOTE
pipe_smote = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5)StandardScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW   ",
    sep=" ± "
)

# 6) StandardScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",StandardScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) StandardScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / StandardScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# Concatenar resultados
df_MLP_classifier_StandardScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_StandardScaler_fe


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) 

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,FE / StandardScaler / 3 Class,0.3490,0.9622,0.3311,0.9606,0.3283,0.9615,0.3431,0.9622,0.3309,0.9625,0.3311,0.9606,0.6491,0.9794,0.4632,0.9700
1,MLP_classifier,FE / StandardScaler / 2 Class IS_W / UNBALANCED,0.5508,0.9698,0.5329,0.9675,0.5294,0.9683,0.5494,0.9698,0.5307,0.9693,0.5329,0.9675,0.5149,0.9652,0.5237,0.9664
2,MLP_classifier,FE / StandardScaler / 2 Class S_IW / UNBALANCED,0.6812,0.9856,0.5420,0.9727,0.5438,0.9803,0.6703,0.9855,0.5496,0.9886,0.5420,0.9727,0.4029,0.9598,0.4671,0.9662
3,MLP_classifier,FE / StandardScaler / 2 Class IS_W / SMOTE,0.5618,0.9770,0.5432,0.9760,0.5400,0.9759,0.5601,0.9770,0.5412,0.9759,0.5432,0.9760,0.5246,0.9750,0.5337,0.9755
4,MLP_classifier,FE / StandardScaler / 2 Class S_IW,0.6523,0.9859,0.5359,0.9860,0.5362,0.9813,0.6532,0.9860,0.5386,0.9770,0.5359,0.9860,0.4195,0.9861,0.4740,0.9860
5,MLP_classifier,FE / StandardScaler / 2 Class S_IW / RANDOM UN...,0.6441,0.9907,0.5267,0.9906,0.5273,0.9876,0.6457,0.9908,0.5290,0.9848,0.5267,0.9906,0.4093,0.9904,0.4642,0.9905
6,MLP_classifier,FE / StandardScaler / 2 Class IS_W / RANDOM UN...,0.5619,0.9777,0.5433,0.9764,0.5399,0.9767,0.5597,0.9777,0.5419,0.9769,0.5433,0.9764,0.5248,0.9752,0.5339,0.9758


##### MinMax

In [259]:
# 1) MinMaxScaler 3 CLASS
pipe = Pipeline([
    ("scaler",MinMaxScaler()),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_xgb,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df1 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 3 Class",
    sep=" ± "
)

# 2) MinMaxScaler 2 CLASS IS_W/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df2 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / UNBALANCED",
    sep=" ± "
)

# 3) MinMaxScaler 2 CLASS S_IW/ UNBALANCED
results_none = cross_validate(
    pipe,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df3 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / UNBALANCED",
    sep=" ± "
)


# 4) MinMaxScaler 2 CLASS IS_W / Smote
pipe_smote = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('smote', SMOTE(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df4 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / SMOTE",
    sep=" ± "
)

# 5) MinMaxScaler 2 CLASS S_IW / SMOTE

results_none = cross_validate(
    pipe_smote,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)

df5 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / SMOTE",
    sep=" ± "
)

# 6) MinMaxScaler 2 CLASS S_IW / RANDOM UNDERSAMPLING
pipe_rus = Pipeline(steps=[
    ("scaler",MinMaxScaler()),
    ('rus', RandomOverSampler(random_state=42)),
    ("clf", clone(MLP_classifier))
])

results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_S_IW,
    cv=skf,
    scoring=scoring,
    return_train_score=True
)
df6 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDERSAMPLING",
    sep=" ± "
)

# 7) MinMaxScaler 2 CLASS IS_W / RANDOM UNDERSAMPLING
results_none = cross_validate(
    pipe_rus,
    X_train_fe,
    y_train_2_IS_W,
    cv=skf,
    scoring=scoring,
    return_train_score=True)

df7 = cv_results_to_row(
    results_none,
    modelo_name="MLP_classifier",
    parameters="FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDERSAMPLING",
    sep=" ± "
)


# Concatenar resultados
df_MLP_classifier_MinMaxScaler_fe = pd.concat([df1, df2, df3, df4, df5, df6, df7], ignore_index=True)
df_MLP_classifier_MinMaxScaler_fe


/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/fserracrespi/Desktop/PD_Project_UofL/.env/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) 

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,FE / MinMaxScaler / 3 Class,0.3943,0.7325,0.3782,0.7219,0.3767,0.7272,0.3889,0.7312,0.3859,0.7388,0.3782,0.7219,0.6705,0.8510,0.5031,0.7838
1,MLP_classifier,FE / MinMaxScaler / 2 Class IS_W / UNBALANCED,0.5783,0.8194,0.5464,0.7979,0.5432,0.8050,0.5692,0.8160,0.5453,0.8204,0.5464,0.7979,0.5144,0.7765,0.5301,0.7871
2,MLP_classifier,FE / MinMaxScaler / 2 Class S_IW / UNBALANCED,0.6963,0.8733,0.5259,0.7685,0.5226,0.8021,0.6678,0.8620,0.5466,0.8800,0.5259,0.7685,0.3555,0.6637,0.4321,0.7140
3,MLP_classifier,FE / MinMaxScaler / 2 Class IS_W / SMOTE,0.5660,0.8252,0.5493,0.8182,0.5480,0.8173,0.5666,0.8252,0.5479,0.8173,0.5493,0.8182,0.5325,0.8112,0.5408,0.8147
4,MLP_classifier,FE / MinMaxScaler / 2 Class S_IW / SMOTE,0.6277,0.8297,0.5475,0.8323,0.5411,0.7938,0.6415,0.8372,0.5432,0.7773,0.5475,0.8323,0.4674,0.8349,0.5058,0.8336
5,MLP_classifier,FE / MinMaxScaler / 2 Class S_IW / RANDOM UNDE...,0.6332,0.8664,0.5362,0.8553,0.5330,0.8317,0.6421,0.8703,0.5337,0.8173,0.5362,0.8553,0.4393,0.8442,0.4852,0.8497
6,MLP_classifier,FE / MinMaxScaler / 2 Class IS_W / RANDOM UNDE...,0.5728,0.8273,0.5591,0.8208,0.5557,0.8198,0.5731,0.8274,0.5566,0.8196,0.5591,0.8208,0.5454,0.8143,0.5522,0.8176


In [261]:
df_MLP_classifier_final = pd.concat([df_MLP_classifier_results,df_MLP_classifier_StandardScaler, df_MLP_classifier_MinMaxScaler,
                                    df_MLP_classifier_results_fe, df_MLP_classifier_StandardScaler_fe, df_MLP_classifier_MinMaxScaler_fe], ignore_index=True)

df_MLP_classifier_final

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,MLP_classifier,Normal / No Scaler / 3 Class,0.4574,0.5361,0.4165,0.4947,0.4002,0.4839,0.4284,0.5092,0.4405,0.5574,0.4165,0.4947,0.6786,0.7227,0.5315,0.5979
1,MLP_classifier,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.6305,0.6628,0.5614,0.5931,0.5333,0.5703,0.5774,0.6117,0.6073,0.6854,0.5614,0.5931,0.4922,0.5234,0.5256,0.5571
2,MLP_classifier,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.7486,0.7589,0.5066,0.5204,0.4527,0.4758,0.6561,0.6705,0.5176,0.7535,0.5066,0.5204,0.2646,0.2819,0.3659,0.3830
3,MLP_classifier,Normal / No Scaler / 2 Class IS_W / SMOTE,0.5288,0.5900,0.5585,0.6192,0.5175,0.5799,0.5137,0.5785,0.5686,0.6355,0.5585,0.6192,0.5881,0.6483,0.5729,0.6334
4,MLP_classifier,Normal / No Scaler / 2 Class S_IW / SMOTE,0.6225,0.6837,0.5349,0.6233,0.4928,0.5939,0.5992,0.6724,0.5504,0.6626,0.5349,0.6233,0.4473,0.5630,0.4838,0.5892
5,MLP_classifier,Normal / No Scaler / 2 Class S_IW / RANDOM UND...,0.5552,0.6287,0.5163,0.6092,0.4588,0.5363,0.5383,0.6077,0.5447,0.6549,0.5163,0.6092,0.4774,0.5898,0.4909,0.5933
6,MLP_classifier,Normal / No Scaler / 2 Class IS_W / RANDOM UND...,0.5372,0.5900,0.5594,0.6086,0.5175,0.5714,0.5159,0.5736,0.5720,0.6284,0.5594,0.6086,0.5817,0.6273,0.5700,0.6175
7,MLP_classifier,Normal / StandardScaler / 3 Class,0.3874,0.8901,0.3704,0.8828,0.3644,0.8887,0.3780,0.8899,0.3703,0.8979,0.3704,0.8828,0.6676,0.9369,0.4971,0.9094
8,MLP_classifier,Normal / StandardScaler / 2 Class IS_W / UNBAL...,0.5660,0.9196,0.5426,0.9137,0.5398,0.9156,0.5619,0.9194,0.5410,0.9178,0.5426,0.9137,0.5192,0.9078,0.5307,0.9108
9,MLP_classifier,Normal / StandardScaler / 2 Class S_IW / UNBAL...,0.6799,0.9392,0.5244,0.8850,0.5229,0.9120,0.6611,0.9367,0.5302,0.9515,0.5244,0.8850,0.3688,0.8308,0.4395,0.8575


# Default Models Performance  Development

In [265]:
df_final_results = pd.concat([df_logreg_final,
                                df_linear_svc_final,
                                df_SGD_classifier_final,
                                df_Tree_classifier_final,
                                df_RF_classifier_final,
                                df_extratree_classifier_final,
                                df_GB_classifier_final,
                                df_XGB_classifier_final,
                                df_KNN_classifier_final,
                                df_SVM_RBF_classifier_final,
                                df_SVM_Poly_classifier_final,
                                df_SVM_Sigmoid_classifier_final,
                                df_MNB_classifier_final,
                                df_BNB_classifier_final,
                                df_GNB_classifier_final,
                                df_MLP_classifier_final], ignore_index=True)

df_final_results.shape
df_final_results.to_csv(ROOT/'DATA_MODEL_Dev/Motor_default_models.csv', index=False)

# Motor Models Performance Analysis

In [268]:
models_info=pd.read_csv(ROOT/'DATA_MODEL_Dev/Motor_default_models.csv')
models_info.head()

,Model,Parameters,test_acc,train_acc,test_bal_acc,train_bal_acc,test_f1_macro,train_f1_macro,test_f1_weighted,train_f1_weighted,test_prec_macro,train_prec_macro,test_rec_macro,train_rec_macro,test_specificity_weighted,train_specificity_weighted,test_g_mean,train_g_mean
0,LogisticRegression,Normal / No Scaler / 3 Class,0.4341,0.4821,0.4246,0.4782,0.4226,0.4748,0.4351,0.4835,0.4276,0.4751,0.4246,0.4782,0.7125,0.7376,0.5494,0.5939
1,LogisticRegression,Normal / No Scaler / 2 Class IS_W / UNBALANCED,0.5647,0.5962,0.5698,0.6049,0.5600,0.5932,0.5687,0.6004,0.5671,0.6005,0.5698,0.6049,0.5749,0.6136,0.5723,0.6092
2,LogisticRegression,Normal / No Scaler / 2 Class S_IW / UNBALANCED,0.5838,0.6119,0.5650,0.6093,0.5357,0.5698,0.6110,0.6378,0.5498,0.5826,0.5650,0.6093,0.5462,0.6067,0.5555,0.6080
3,LogisticRegression,Normal / No Scaler / 2 Class IS_W / SMOTE,0.6196,0.6360,0.5822,0.5992,0.5803,0.5995,0.6070,0.6247,0.5911,0.6121,0.5822,0.5992,0.5448,0.5623,0.5631,0.5805
4,LogisticRegression,Normal / No Scaler / 2 Class S_IW / SMOTE,0.7418,0.7503,0.5170,0.5329,0.4827,0.5091,0.6677,0.6830,0.5640,0.6165,0.5170,0.5329,0.2921,0.3155,0.3884,0.4098


## Best Models per Metric

In [288]:
results = []

for col in models_info.filter(like='test').columns:
    top3 = models_info.nlargest(3, col)
    for _, row in top3.iterrows():
        results.append({
            "Metric": col,
            "Model": row["Model"],
            "Value_test": row[col],
            "Value_train": row[col.replace("test", "train")],
            "Parameters": row["Parameters"]

        })

top3_df = pd.DataFrame(results)
top3_df

,Metric,Model,Value_test,Value_train,Parameters
0,test_acc,KNN_classifier,0.7541,0.7527,FE / StandardScaler / 2 Class S_IW / UNBALANCED
1,test_acc,linear_svc,0.7527,0.7534,Normal / No Scaler / 2 Class S_IW / UNBALANCED
2,test_acc,GB_classifier,0.7527,0.8458,Normal / No Scaler / 2 Class S_IW / UNBALANCED
3,test_bal_acc,SVM_Sigmoid_classifier,0.6167,0.6121,FE / No Scaler / 2 Class IS_W / SMOTE
4,test_bal_acc,SVM_Sigmoid_classifier,0.6143,0.6107,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...
5,test_bal_acc,MNB_classifier,0.6089,0.6081,FE / No Scaler / 2 Class IS_W / RANDOM UNDERSA...
6,test_f1_macro,SVM_Poly_classifier,0.5960,0.5933,Normal / No Scaler / 2 Class IS_W / RANDOM UND...
7,test_f1_macro,BNB_classifier,0.5957,0.5968,FE / StandardScaler / 2 Class IS_W / UNBALANCED
8,test_f1_macro,GB_classifier,0.5955,0.8466,FE / No Scaler / 2 Class S_IW / SMOTE
9,test_f1_weighted,GB_classifier,0.7027,0.8871,FE / No Scaler / 2 Class S_IW / SMOTE


### Findings:
The table summarizes that the 3 class classification is not suitable for the model performance. The use of joining the clases impovement and stability or improvement or worsening induces a beter overall performance acorss all the models.

### Priority Hyperparameter Search 
1. GradientBoostingClassifier
2. LinearSVC
3. KNNClassifier
4. LogisticRegression
5. Models such DecisionTree based
